# 01. 알약 데이터 EDA (통합본)

팀원별 EDA 노트북 6개를 주제별로 합친 노트북입니다. 같은 주제는 한 셀을 기준본으로 두고
나머지 노트북의 장점을 기준본에 녹였습니다. 섹션마다 첫 셀에 원작자와 원본 노트북을 적었습니다.

## 실행 방법

- `uv sync` 후 VS Code나 `uv run jupyter lab`에서 프로젝트 `.venv` 커널로 엽니다.
- 위에서 아래로 한 번 실행(Restart & Run All)합니다. 작업 폴더는 프로젝트 루트와 `notebooks/` 모두 됩니다.
- 원본은 `data/raw/`에서 읽기만 합니다. `ann_df`·`img_df`·`cat_df`는 원본 상태로 유지하고,
  보정 비교는 별도 `ann_corrected_df`에서 합니다.
- 커밋 전에 출력을 지웁니다(Clear All Outputs).

## 실행 옵션 (1장 설정 셀)

| 옵션 | 기본값 | 설명 |
|---|---|---|
| `RUN_HEAVY` | `False` | 전체 이미지 SHA256·디코딩, 전체 박스 GrabCut 같은 무거운 연산을 실행합니다. `False`이면 입력·버전·파라미터가 일치하는 캐시만 재사용하고, 없으면 `미실행`으로 표시합니다. `True`이면 캐시와 관계없이 다시 계산합니다. |
| `SAVE_OUTPUTS` | `False` | 그림·보고서·캐시 파일을 저장합니다. `False`이면 파일을 쓰지 않습니다. 켜도 원본은 수정하지 않습니다. |
| `SEED` | `42` | 무작위 연산 전체에 쓰는 seed입니다. |

`미실행` 결과는 0건이나 검사 통과로 해석하지 않습니다. 후속 셀은 결과 상태를 확인한 뒤 집계합니다.

## 데이터 상태 표시

표와 그림 제목 앞에 데이터 상태를 붙입니다. `[원본]`은 `data/raw/`를 그대로 읽은 결과,
`[보정]`은 보정 기록을 메모리에서 적용한 결과입니다.

## 섹션별 실행 시간

| 섹션 | `RUN_HEAVY=False` | `RUN_HEAVY=True` |
|---|---|---|
| 1–2. 설정·데이터 로드 | 약 3초 (Apple Silicon Mac) | 같음 |
| 3–4. 규모·구조·결측 | 약 2초 | 약 15초 (전체 디코딩·SHA256 13초) |
| 5. 라벨 오류와 이상치 | 약 12초 (박스 픽셀 측정 10초) | 약 2분 10초 (GrabCut 여백 추정 2분) |
| 6–7. 분포·촬영 조건 | 약 2초 | 약 1분 50초 (GrabCut 장단축 측정 1분 45초) |
| 8–9. 누수·분할·결론 | 약 10초 (사진 해시 8초) | 같음 |

## 목차

0. 개요
1. 환경 설정
2. 데이터 로드
3. 데이터 규모와 구조
4. 결측
5. 라벨 오류와 이상치
6. 분포
7. 촬영 조건
8. 데이터 누수와 분할
9. 결론과 다음 작업

## 원본 노트북

| 노트북 | 작성자 |
|---|---|
| `EDA_jang.ipynb` | younghoon1215 |
| `JJAM99_eda.ipynb` | potato |
| `dongwook_eda.ipynb` | def-uk |
| `eda_kwon.ipynb` | gwon753-dev |
| `eda_sohee.ipynb`, `sohee_eda.ipynb` | bravedolphinhere |

## 1. 환경 설정

> 출처: dongwook_eda.ipynb 셀 1 (def-uk), eda_kwon.ipynb 셀 2, 35 (gwon753-dev),
> sohee_eda.ipynb 셀 1, 3 (bravedolphinhere)

- 경로는 모두 `src/config.py`의 `PROJECT_ROOT` 기준입니다.
- 한글 폰트는 Malgun Gothic → AppleGothic → NanumGothic → Noto Sans CJK KR 순서로 찾습니다.
  WSL에서는 Windows의 맑은 고딕 파일을 등록합니다.
- 무작위 연산은 모두 `SEED`를 씁니다(`random_state=SEED`, `np.random.default_rng(SEED)`).

In [ ]:
import sys
from pathlib import Path

# 노트북은 notebooks/에서 실행되므로 프로젝트 루트를 import 경로에 추가한다.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
import hashlib
import json
import math
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold

from data_process.data_processing import (
    AXIS_VERSION,
    BBOX_COLS,
    PADDING_VERSION,
    bbox_flags,
    bbox_iou,
    build_cat_df,
    build_img_df,
    compare_name_codes,
    find_box_pairs,
    fingerprint_files,
    group_setups,
    hash_similarity,
    image_hashes,
    inspect_image_integrity,
    iqr_bounds,
    list_images,
    load_annotations,
    load_cache,
    load_schema,
    measure_bbox_padding,
    measure_box_pixels,
    measure_pill_axes,
    propose_bbox_at_center,
    propose_bbox_by_shape,
    read_image_headers,
    rotate_bboxes_180,
    save_cache,
    scan_missing_values,
    setup_pair_distances,
)
from data_process.make_labels import (
    CORRECTION_COLS,
    STAGES,
    apply_corrections,
    assign_new_ids,
    correction_for_label,
    corrections_from_kwon,
)
from src.config import PROJECT_ROOT
from visualization import (
    draw_bbox,
    draw_bboxes,
    draw_image_bounds,
    image_grid,
    plot_corr_heatmap,
    set_korean_font,
    show_image,
    zoom_to,
)

SEED = 42
RUN_HEAVY = False  # True면 무거운 연산을 캐시와 관계없이 다시 계산한다.
SAVE_OUTPUTS = False  # True일 때만 그림·보고서·캐시 파일을 쓴다.
NOT_RUN = "미실행"  # 생략한 분석의 결과 상태. 0건이나 통과로 해석하지 않는다.

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"  # 받은 원본 그대로 (읽기만 함)
# 압축을 푼 위치에 따라 sprint_ai_project1_data 폴더가 한 단계 더 있을 수 있다.
if (RAW_DIR / "sprint_ai_project1_data").is_dir():
    RAW_DIR = RAW_DIR / "sprint_ai_project1_data"
PROCESSED_DIR = DATA_DIR / "processed"  # 보정본·보정 기록
CACHE_DIR = PROCESSED_DIR / "eda_cache"  # 무거운 연산 캐시. SAVE_OUTPUTS=True일 때만 쓴다.

TRAIN_IMG_DIR = RAW_DIR / "train_images"
TRAIN_ANN_DIR = RAW_DIR / "train_annotations"
TEST_IMG_DIR = RAW_DIR / "test_images"
for folder in [TRAIN_IMG_DIR, TRAIN_ANN_DIR, TEST_IMG_DIR]:
    if not folder.is_dir():
        raise FileNotFoundError(f"데이터 폴더가 없습니다: {folder.relative_to(PROJECT_ROOT)}")

np.random.seed(SEED)
pd.set_option("display.max_columns", 60)
font = set_korean_font()
if font is None:
    print("한글 폰트를 찾지 못했습니다. 약 이름이 네모로 보일 수 있습니다.")

print(f"SEED={SEED}, RUN_HEAVY={RUN_HEAVY}, SAVE_OUTPUTS={SAVE_OUTPUTS}, 폰트={font}")
print(f"원본 데이터: {RAW_DIR.relative_to(PROJECT_ROOT).as_posix()}/")

## 2. 데이터 로드

> 출처: dongwook_eda.ipynb 셀 0–3 (def-uk) 기준. eda_kwon.ipynb 셀 1–2 (gwon753-dev)의
> 디스크 기준 `img_df`·`cat_df`·`test_files`, sohee_eda.ipynb 셀 2, 7 (bravedolphinhere)의
> JSON 내부 참조 연결과 읽기 오류 기록을 합쳤습니다. 로더는 `data_process/data_processing.py`에 있습니다.

- annotation JSON 하나 = 이미지 1장 × 알약 1개입니다. 같은 이미지의 알약이 약 코드별 폴더에 흩어져 있습니다.
  경로: `train_annotations/<조합>_json/<약코드>/<이미지이름>.json`
- bbox는 COCO 형식 `[x, y, w, h]`(왼쪽 위 기준, 픽셀)입니다.
- 이미지 메타데이터는 JSON 안의 `annotation.image_id`로 연결합니다. 전역 `image_id`로 합치지 않습니다.
- 라벨 식별자는 `(source_json, annotation_id)`입니다. `source_json`은 `train_annotations/` 기준 상대 경로입니다.
- bbox 값이 비어 있는 라벨도 빼지 않고 `x, y, w, h`를 NaN으로 둡니다(4.1에서 확인).
- 읽기·연결 실패는 `load_error_df`에 남기고, 분석에서 제외된 입력 수를 보여 줍니다.

| 표 | 한 행 | 설명 |
|---|---|---|
| `ann_df` | 원본 bbox 1개 | 이미지 메타데이터 + 약 정보 + bbox. 원본 상태를 유지합니다. |
| `img_df` | 디스크의 train 이미지 1장 | 라벨 없는 이미지도 포함합니다(`n_boxes=0`). `combo`는 파일 이름의 알약 조합입니다. |
| `cat_df` | 클래스 1개 | `category_id` ↔ 약 이름, bbox 수, 이미지 수 |
| `test_files` | test 이미지 1장 | 파일 이름 목록 |
| `load_error_df` | 오류 1건 | `stage`: `parse`(JSON 읽기), `image_link`, `category_link`, `empty`(annotation 없음) |

In [ ]:
ann_df, load_error_df = load_annotations(TRAIN_ANN_DIR)
img_df = build_img_df(ann_df, list_images(TRAIN_IMG_DIR))
cat_df = build_cat_df(ann_df)
test_files = [p.name for p in list_images(TEST_IMG_DIR)]

# 제외된 입력: parse 오류는 JSON 파일 전체, image_link 오류는 annotation 하나가 빠진다.
n_json = sum(1 for _ in TRAIN_ANN_DIR.rglob("*.json"))
excluded = load_error_df[load_error_df["excluded"]]
n_json_excluded = (excluded["stage"] == "parse").sum()
n_ann_excluded = (excluded["stage"] == "image_link").sum()
n_no_image = (~ann_df["file_name"].isin(img_df["file_name"])).sum()

print(f"[원본] annotation JSON: {n_json}개 (읽기 실패로 제외: {n_json_excluded}개)")
print(
    f"[원본] bbox: {len(ann_df)}개 (연결 실패로 제외: {n_ann_excluded}개), 클래스: {len(cat_df)}개"
)
print(
    f"[원본] train 이미지: {len(img_df)}장 (bbox 없음: {(img_df['n_boxes'] == 0).sum()}장), "
    f"test 이미지: {len(test_files)}장"
)
print(f"[원본] 디스크에 이미지 파일이 없는 bbox: {n_no_image}개")
print(f"로드 오류: {len(load_error_df)}건")
if not load_error_df.empty:
    display(load_error_df)

In [ ]:
display(ann_df.head(3))
display(img_df.head(3))
display(cat_df.head(3))

### 2.1 데이터 정의서

> 출처: eda_kwon.ipynb 셀 3 (gwon753-dev) 기준. dongwook_eda.ipynb 셀 9 (def-uk)의
> 스키마 필수여부 읽기를 합쳤습니다.

먼저 `data/README.md`의 스키마 표와 실제 JSON 키를 대조하고, `ann_df` 컬럼마다 필수여부(M 필수,
O 선택, C 조건부)와 스키마 설명을 붙입니다. 결측 비율은 4.3에서 봅니다.

In [ ]:
schema_df = load_schema(DATA_DIR / "README.md")
# 최상위 키(images, annotations 등)는 field가 비어 있어 section만 키로 쓴다.
schema_df["key"] = (schema_df["section"] + "." + schema_df["field"]).str.rstrip(".")

# ann_df 컬럼 → 스키마 키. 로더가 바꾸거나 나눈 컬럼만 적고, 나머지는 images[]의 같은 이름.
LOADER_COLS = {
    "image_id": "images.id",
    "source_json": None,  # 로더가 붙인 원본 JSON 상대 경로
    "annotation_id": "annotations.id",
    "category_id": "annotations.category_id",
    "category_name": "categories.name",
    "x": "annotations.bbox",
    "y": "annotations.bbox",
    "w": "annotations.bbox",
    "h": "annotations.bbox",
    "area": "annotations.area",
    "iscrowd": "annotations.iscrowd",
    "ignore": "annotations.ignore",
    "has_segmentation": "annotations.segmentation",
}
col_to_key = {c: LOADER_COLS.get(c, f"images.{c}") for c in ann_df.columns}

json_image_keys = {k for c, k in col_to_key.items() if k and k.startswith("images.")}
is_image_field = (schema_df["section"] == "images") & (schema_df["field"] != "")
schema_image_keys = set(schema_df.loc[is_image_field, "key"])
print("스키마에 없는 JSON 키:", sorted(json_image_keys - schema_image_keys) or "없음")
print("JSON에 없는 스키마 키:", sorted(schema_image_keys - json_image_keys) or "없음")

definition_df = pd.DataFrame(
    {
        "컬럼": ann_df.columns,
        "스키마 키": [col_to_key[c] for c in ann_df.columns],
        "dtype": ann_df.dtypes.astype(str).to_numpy(),
        "고유값 수": ann_df.nunique().to_numpy(),
        "예시": [ann_df[c].dropna().iloc[0] if ann_df[c].notna().any() else None for c in ann_df],
    }
)
definition_df = definition_df.merge(
    schema_df[["key", "type", "required", "description"]],
    left_on="스키마 키",
    right_on="key",
    how="left",
).drop(columns="key")
definition_df = definition_df.rename(
    columns={"type": "스키마 타입", "required": "필수여부", "description": "스키마 설명"}
)
with pd.option_context("display.max_rows", 100, "display.max_colwidth", 40):
    display(definition_df)

아래는 kwon이 값을 보고 정리한 컬럼 해석입니다. 수치는 원본 상태 관측값이며 3–4장에서 다시 확인합니다.
필수여부는 위 표를 따릅니다.

### 이미지 정보

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| file_name | str | 이미지 파일명. 이미지 단위의 키로 사용됩니다. 앞부분(`K-003351-018357-036637`)에 사진에 찍힌 약 코드가 들어 있습니다. |
| imgfile | str | 이미지 파일명. `file_name`과 같은 값입니다. 스키마 표에는 없는 키입니다. |
| width | int | 이미지 너비(px). 전부 976입니다. |
| height | int | 이미지 높이(px). 전부 1280입니다. |
| image_id | int | JSON 안의 이미지 아이디(`images[].id`)입니다. |
| source_json | str | 이 행을 읽어 온 annotation JSON의 상대 경로입니다(`train_annotations/` 기준). |

### 촬영 조건

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| back_color | str | 배경색. 전부 `연회색 배경`입니다. |
| light_color | str | 조명 색. 전부 `주백색`입니다. |
| camera_la | int | 카메라 촬영 각도(위도). 70, 75, 90 세 가지입니다. |
| camera_lo | int | 카메라 회전 각도(경도). 전부 0입니다. |
| size | int | 촬영 크기 설정값. 전부 200입니다. |
| drug_dir | str | 촬영된 알약 면. `앞면` 또는 `뒷면`입니다. |
| drug_S | str | 알약 상태. 전부 `정상알약`입니다. |

### 정답 (타깃)

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| annotation_id | int | 어노테이션 아이디. `source_json`과 함께 라벨 하나를 가리킵니다. |
| category_id | int | **타깃 변수(클래스)**. 약 종류 코드이며 56종입니다. `drug_N`의 숫자 부분과 같습니다. |
| category_name | str | JSON `categories[].name`의 약 이름입니다. |
| x | int | **타깃 변수(bbox)**. 박스 왼쪽 위 꼭짓점의 x 좌표(px)입니다. |
| y | int | **타깃 변수(bbox)**. 박스 왼쪽 위 꼭짓점의 y 좌표(px)입니다. |
| w | int | **타깃 변수(bbox)**. 박스 너비(px)입니다. |
| h | int | **타깃 변수(bbox)**. 박스 높이(px)입니다. |
| area | int | 박스 넓이(px²). 대부분 `w × h`와 같습니다. 다른 경우는 5.4에서 봅니다. |
| iscrowd, ignore, has_segmentation | int, int, bool | 평가 분류, 무시 여부, segmentation 유무입니다. |

### 약 식별 코드

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| drug_N | str | 약 코드(예: `K-003351`). `category_id`에 `K-`를 붙인 형태입니다. |
| dl_mapping_code | str | 약 매핑 코드. `drug_N`과 같은 값입니다. |
| dl_idx | str | 약 코드의 숫자 부분(예: `3351`)입니다. (이 부분은 일단 눈여겨 보자) |
| item_seq | int | 식약처 품목기준코드입니다. |
| di_edi_code | str | 건강보험 EDI 코드. 여러 개면 쉼표로 구분합니다. |

### 약 기본 정보

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| dl_name | str | 약 이름(한글)입니다. |
| dl_name_en | str | 약 이름(영문). 14행이 빈 문자열입니다. |
| dl_material | str | 주성분(한글). 성분이 여러 개면 `\|`로 구분합니다. 9행이 빈 문자열입니다. |
| dl_material_en | str | 주성분(영문)입니다. |
| dl_company | str | 제약회사(한글)입니다. |
| dl_company_en | str | 제약회사(영문)입니다. |
| di_company_mf | str | 위탁 제조사(한글). 대부분(637행) 빈 문자열입니다. |
| di_company_mf_en | str | 위탁 제조사(영문). 대부분(724행) 빈 문자열입니다. |
| di_class_no | str | 약효 분류(예: `[02180]동맥경화용제`)입니다. |
| di_etc_otc_code | str | 의약품 구분. `전문의약품` 또는 `일반의약품`입니다. |
| di_item_permit_date | str | 품목 허가일(YYYYMMDD)입니다. |

### 약 외형

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| dl_custom_shape | str | 제형 분류(예: `정제, 저작정`, `경질캡슐제`)입니다. |
| form_code_name | str | 제형 이름(예: `필름코팅정`, `나정`)입니다. |
| drug_shape | str | 알약 모양(예: `원형`, `타원형`, `장방형`)입니다. |
| chart | str | 성상. 알약 생김새를 글로 설명한 것입니다. |
| color_class1 | str | 주 색상입니다. |
| color_class2 | str | 보조 색상. 대부분 빈 문자열입니다. |
| thick | float | 두께(mm). 0.0은 값이 없는 경우입니다. |
| leng_long | float | 장축 길이(mm). 0.0은 값이 없는 경우입니다. |
| leng_short | float | 단축 길이(mm). 0.0은 값이 없는 경우입니다. |
| print_front | str | 앞면 각인 문자(예: `R5`)입니다. |
| print_back | str | 뒷면 각인 문자. `분할선`처럼 적힌 경우도 있습니다. |
| line_front | str | 앞면 분할선. `-`(일자), `+`(십자), 빈 문자열(없음)입니다. |
| line_back | str | 뒷면 분할선. 값은 `line_front`와 같은 방식입니다. |

### 마크 정보

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| mark_code_front_anal | str | 앞면 마크 설명(예: `pfizer`). 대부분 빈 문자열입니다. |
| mark_code_front_img | str | 앞면 마크 이미지 URL. 대부분 빈 문자열입니다. |
| mark_code_back_anal | str | 뒷면 마크 설명. 전부 비어 있습니다. |
| mark_code_back_img | str | 뒷면 마크 이미지 URL. 전부 비어 있습니다. |
| mark_code_front | str | 앞면 마크 코드. 전부 비어 있습니다. |
| mark_code_back | str | 뒷면 마크 코드. 전부 비어 있습니다. |

### 기타

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| img_key | str | 공공 낱알 이미지 URL입니다. |
| img_regist_ts | str | 낱알 이미지 등록일(YYYYMMDD)입니다. |
| change_date | str | 약 정보 변경일(YYYYMMDD)입니다. |

> 참고: `img_key`, `chart`, `drug_shape`, `print_*`, `color_*`, `line_*`, `mark_*`, `img_regist_ts`,
> `change_date`, `form_code_name`은 18행이 결측(NaN)입니다. 키는 있고 값이 `null`입니다(4.3에서 확인).
> 원본 kwon 정의서와 dongwook 셀 9는 이를 "키 없음"으로 적었습니다.

## 3. 데이터 규모와 구조

> 출처: JJAM99_eda.ipynb 셀 3–17, 19 (potato) 기준. sohee_eda.ipynb 셀 7, 9, 11 (bravedolphinhere)의
> image_id 대응 확인, 조합별 예시 이미지, 디코딩 실패·SHA256·train/test 공통 해시를 합쳤습니다.

### 3.1 파일 수, 해상도, 색상 모드, 디코딩 오류

> 출처: JJAM99_eda.ipynb 셀 3, 5, 19 (potato) 기준. sohee_eda.ipynb 셀 11 (bravedolphinhere)의
> 디코딩 실패·SHA256을 합쳤습니다.

먼저 이미지 헤더만 읽어 해상도와 색상 모드를 봅니다(빠름). 헤더가 정상이어도 픽셀 손상까지 확인된
것은 아니므로, 전체 디코딩과 SHA256은 다음 셀에서 `RUN_HEAVY`일 때만 계산합니다.
JSON에 적힌 `width`·`height`와 실제 이미지 크기도 라벨마다 대조합니다.

In [ ]:
img_paths = {"train": list_images(TRAIN_IMG_DIR), "test": list_images(TEST_IMG_DIR)}
header_df = pd.concat(
    [read_image_headers(paths, split) for split, paths in img_paths.items()], ignore_index=True
)
header_ok = header_df["error"].isna()
header_df["resolution"] = [
    f"{int(w)}×{int(h)}" if ok else None
    for w, h, ok in zip(header_df["width"], header_df["height"], header_ok, strict=True)
]
header_summary = header_df.groupby("split", sort=False).agg(
    n_images=("file_name", "size"),
    resolution=("resolution", lambda s: sorted(s.dropna().unique())),
    mode=("mode", lambda s: sorted(s.dropna().unique())),
    n_errors=("error", lambda s: s.notna().sum()),
)
print("[원본] 이미지 헤더")
display(
    header_summary.rename(
        columns={
            "n_images": "이미지 수",
            "resolution": "해상도",
            "mode": "색상 모드",
            "n_errors": "헤더 읽기 오류",
        }
    )
)
if not header_ok.all():
    display(header_df[~header_ok])

# 라벨마다 JSON의 width·height와 실제 이미지 크기를 비교한다. 헤더를 못 읽은 이미지는 위에서 셌다.
train_header = header_df[(header_df["split"] == "train") & header_ok]
size_check = ann_df[["source_json", "file_name", "width", "height"]].merge(
    train_header[["file_name", "width", "height"]], on="file_name", suffixes=("_json", "_actual")
)
size_mismatch = size_check[
    (size_check["width_json"] != size_check["width_actual"])
    | (size_check["height_json"] != size_check["height_actual"])
]
print(f"[원본] JSON 크기와 실제 이미지 크기가 다른 라벨: {len(size_mismatch)}개")
if not size_mismatch.empty:
    display(size_mismatch)

In [ ]:
# [RUN_HEAVY] 전체 디코딩·SHA256. 입력 지문·계산 버전이 저장 당시와 같은 캐시만 재사용한다.
INTEGRITY_VERSION = 1  # inspect_image_integrity의 계산 방식을 바꾸면 올린다.
integrity_cache = CACHE_DIR / "image_integrity.json"
integrity_key = {
    "version": INTEGRITY_VERSION,
    "data_state": "원본",
    "inputs": fingerprint_files([*img_paths["train"], *img_paths["test"]], RAW_DIR),
}
if RUN_HEAVY:
    start = time.perf_counter()
    integrity_df = pd.concat(
        [inspect_image_integrity(paths, split) for split, paths in img_paths.items()],
        ignore_index=True,
    )
    integrity_status = f"실행 ({time.perf_counter() - start:.0f}초)"
    if SAVE_OUTPUTS:
        save_cache(integrity_cache, integrity_key, integrity_df)
else:
    integrity_df, integrity_status = load_cache(integrity_cache, integrity_key)
    if integrity_df is None:
        integrity_status = f"{NOT_RUN} ({integrity_status})"

# integrity_df가 None이면 미실행이다. 뒤 셀은 None을 확인하고, 0건으로 바꿔 세지 않는다.
print(f"[원본] 전체 디코딩·SHA256: {integrity_status}")
if integrity_df is not None:
    decoded = integrity_df["decoded"].astype(bool)
    display(
        integrity_df.assign(failed=~decoded)
        .groupby("split", sort=False)
        .agg(n_images=("file_name", "size"), n_failed=("failed", "sum"))
        .rename(columns={"n_images": "이미지 수", "n_failed": "디코딩 실패"})
    )
    if not decoded.all():
        display(integrity_df[~decoded])

### 3.2 1 사진 : N JSON 구조와 bbox 시각 확인

> 출처: JJAM99_eda.ipynb 셀 7–14 (potato) 기준. sohee_eda.ipynb 셀 7, 9 (bravedolphinhere)의
> image_id 대응 확인과 여러 장 예시를 합쳤습니다.

- 사진과 JSON은 JSON 안의 `images[].file_name`으로 연결합니다. JSON 파일명이나 전역 `image_id`로
  합치지 않습니다. 그래도 둘이 사진과 어떻게 대응하는지는 확인해 둡니다.
- 학습할 때는 같은 사진을 가리키는 JSON의 라벨을 모두 모아야 합니다.

In [ ]:
# 사진당 JSON 수와 bbox 수. JSON 하나에 bbox가 하나씩이면 대각선에만 값이 있다.
print("[원본] 사진당 JSON 수 × bbox 수 (값: 사진 수)")
display(
    pd.crosstab(img_df["n_json"], img_df["n_boxes"]).rename_axis(index="JSON 수", columns="bbox 수")
)
boxes_per_json = ann_df.groupby("source_json").size()
n_missing_image = ann_df.loc[~ann_df["file_name"].isin(img_df["file_name"]), "file_name"].nunique()
print(f"[원본] bbox가 2개 이상인 JSON: {(boxes_per_json > 1).sum()}개")
print(f"[원본] JSON이 없는 train 이미지: {(img_df['n_json'] == 0).sum()}장")
print(f"[원본] JSON이 가리키지만 디스크에 없는 이미지: {n_missing_image}장")

# 연결 기준이 아닌 두 값도 사진과 1:1인지 본다. 1:1이어도 연결에는 file_name을 쓴다.
ids_per_file = ann_df.groupby("file_name")["image_id"].nunique()
files_per_id = ann_df.groupby("image_id")["file_name"].nunique()
print(
    f"[원본] image_id가 2개 이상인 사진: {(ids_per_file > 1).sum()}장 / "
    f"사진이 2장 이상인 image_id: {(files_per_id > 1).sum()}개"
)
json_stem = ann_df["source_json"].str.rsplit("/", n=1).str[-1].str.removesuffix(".json")
img_stem = ann_df["file_name"].str.rsplit(".", n=1).str[0]
print(f"[원본] JSON 파일명이 이미지 파일명과 다른 라벨: {(json_stem != img_stem).sum()}개")

In [ ]:
# bbox 수가 다른 사진을 한 장씩 골라 그 사진의 JSON 라벨을 모두 그린다.
sample_names = (
    img_df[img_df["n_boxes"] > 0]
    .drop_duplicates("n_boxes")
    .sort_values("n_boxes")["file_name"]
    .tolist()
)
fig, axes = image_grid(len(sample_names), cols=4)
for ax, name in zip(axes, sample_names, strict=True):
    boxes = ann_df[ann_df["file_name"] == name]
    boxes = boxes.assign(label=boxes["category_id"].astype(str) + " " + boxes["dl_name"])
    title = f"[원본] {name}\nJSON {boxes['source_json'].nunique()}개 / bbox {len(boxes)}개"
    show_image(ax, TRAIN_IMG_DIR / name, title, fontsize=7)
    draw_bboxes(ax, boxes, label_col="label")
plt.tight_layout()
plt.show()

- JSON 하나에는 사진 1장과 그 사진 속 알약 1개의 라벨이 들어 있습니다. 같은 사진의 라벨이
  약 코드별 폴더에 흩어져 있으므로 사진 단위로 모아야 합니다(JJAM 셀 8).
- JSON 파일명은 이미지 파일명과 같아서, 파일명이 같은 JSON이 사진 수만큼 묶입니다. 이것은 구조이지
  중복이 아닙니다(JJAM 셀 17). 내용 중복은 3.3에서 따로 봅니다.
- bbox가 2개인 사진 7장은 모두 파일명에 약이 3개인 사진입니다. 하나씩 라벨이 빠진 것으로,
  4.2에서 봅니다. bbox 3개 사진 중 1장도 파일명에는 약이 4개입니다(원본 상태 관측값).
- 그림의 박스는 JSON에 적힌 좌표입니다. 클래스 ID가 표시된다고 종류까지 맞다는 뜻은 아니며,
  그림만으로 라벨이 빠짐없이 있는지 확정하지 않습니다(라벨 누락은 4.2).

### 3.3 JSON 내용·라벨·픽셀 중복, train/test 중복

> 출처: JJAM99_eda.ipynb 셀 16, 19 (potato) 기준. sohee_eda.ipynb 셀 11 (bravedolphinhere)의
> train/test 공통 SHA256을 합쳤습니다.

파일명이 같다는 이유만으로 중복으로 보지 않습니다. 아래를 따로 셉니다.

| 검사 | 같다고 보는 기준 | 같으면 |
|---|---|---|
| JSON 내용 중복 | 키 순서·공백을 무시한 JSON 전체 | 같은 라벨 파일이 두 번 들어 있음 |
| 라벨 중복 | 사진·`category_id`·bbox | 같은 알약을 두 번 셈 |
| train/test 같은 파일명 | 파일명 | 같은 사진일 수 있음 |
| 파일·픽셀 중복 [RUN_HEAVY] | 파일 SHA256, RGB 픽셀 SHA256 | 같은 사진이 두 번 들어 있음. train/test 사이면 누수 |

회전·재촬영처럼 비슷하기만 한 사진은 8.2(유사도)에서, 다른 약이 같은 좌표에 붙은 경우는 5.2에서 봅니다.

In [ ]:
# JSON 내용 중복: 키 순서·공백을 무시하고 전체 내용이 같은 JSON을 묶는다.
json_hashes = {}
for json_path in sorted(TRAIN_ANN_DIR.rglob("*.json")):
    try:
        data = json.loads(json_path.read_text(encoding="utf-8-sig"))
    except (OSError, UnicodeDecodeError, ValueError):
        continue  # 읽기 실패는 2장의 load_error_df에 있다.
    normalized = json.dumps(data, sort_keys=True, ensure_ascii=False)
    source_json = json_path.relative_to(TRAIN_ANN_DIR).as_posix()
    json_hashes[source_json] = hashlib.sha256(normalized.encode()).hexdigest()
json_hash = pd.Series(json_hashes, name="content_sha256")
dup_json = json_hash[json_hash.duplicated(keep=False)].sort_values()

# 라벨 중복: 같은 사진에 같은 클래스·같은 bbox가 두 번 이상 있는 경우
label_key = ["file_name", "category_id", "x", "y", "w", "h"]
dup_labels = ann_df[ann_df.duplicated(label_key, keep=False)].sort_values(label_key)
same_name = sorted(set(img_df["file_name"]) & set(test_files))


def count_hash_groups(df: pd.DataFrame, col: str) -> dict[str, int]:
    """해시가 같은 이미지 묶음 수를 train 안, test 안, train/test 사이로 나눠 센다."""
    splits = df.dropna(subset=[col]).groupby(col)["split"].agg(list)
    return {
        "train 안": int(splits.map(lambda s: s.count("train") > 1).sum()),
        "test 안": int(splits.map(lambda s: s.count("test") > 1).sum()),
        "train/test 사이": int(splits.map(lambda s: {"train", "test"} <= set(s)).sum()),
    }


dup_rows = [
    ("JSON 내용 중복 (묶음)", dup_json.nunique(), "완료"),
    ("라벨 중복: 사진·클래스·bbox (묶음)", dup_labels.groupby(label_key).ngroups, "완료"),
    ("train/test 같은 파일명 (장)", len(same_name), "완료"),
]
for col, label in [("file_sha256", "파일 SHA256"), ("pixel_sha256", "픽셀 SHA256")]:
    if integrity_df is None:
        groups = dict.fromkeys(["train 안", "test 안", "train/test 사이"], NOT_RUN)
    else:
        groups = count_hash_groups(integrity_df, col)
    for scope, value in groups.items():
        dup_rows.append((f"{label} 중복: {scope} (묶음)", value, integrity_status))
dup_check_df = pd.DataFrame(dup_rows, columns=["검사", "결과", "상태"])

print("[원본] 중복 검사")
display(dup_check_df)
for title, df in [("JSON 내용 중복", dup_json), ("라벨 중복", dup_labels)]:
    if not df.empty:
        print(title)
        display(df)

- JSON 내용 중복과 같은 사진·클래스·bbox의 라벨 중복은 없습니다(JJAM 셀 16, 19 관측과 같음).
- train/test 사이에 같은 파일명은 없습니다. 파일·픽셀이 완전히 같은 사진도 기존 관측에서는 없었습니다
  (JJAM 셀 19: 동일 픽셀 0묶음, sohee 결과: 학습/테스트 공통 SHA256 0). `RUN_HEAVY=False`이고 캐시가
  없으면 위 표에 `미실행`으로 나오며, 이 경우 중복이 없다고 확인된 것이 아닙니다.
- 완전 중복이 없다고 누수가 없는 것은 아닙니다. 같은 알약 조합을 각도만 바꿔 찍은 사진은 8장에서 봅니다.

## 4. 결측

> 출처: dongwook_eda.ipynb 셀 5–9 (def-uk), eda_kwon.ipynb 셀 4–8, 11–12 (gwon753-dev),
> JJAM99_eda.ipynb 셀 32–37 (potato), eda_sohee.ipynb 셀 6–14 (bravedolphinhere)

결측을 세 층으로 나눕니다.

| 절 | 무엇이 비었나 | 학습 영향 |
|---|---|---|
| 4.1 bbox 값 결측 | 라벨은 있는데 좌표·클래스 값이 없음 | 그 라벨을 쓸 수 없음 |
| 4.2 라벨 누락 | 사진에 있는 알약의 라벨 자체가 없음 | 빠진 알약이 배경으로 학습됨 |
| 4.3 메타데이터 결측 | 약 설명 키(각인, 색상, 마크 등)가 비어 있음 | bbox·클래스 학습에는 직접 영향 없음 |

### 4.1 bbox 값 결측

> 출처: dongwook_eda.ipynb 셀 6 (def-uk)의 값 결측 검사 기준. JJAM99_eda.ipynb 셀 33 (potato)의
> bbox 내부 결측 검사를 합쳤습니다.

로더는 bbox가 비었거나 숫자 4개가 아닌 라벨을 빼지 않고 `x, y, w, h`를 NaN으로 둡니다.
크기 0 이하, 이미지 밖 좌표, `area` 불일치처럼 값은 있지만 틀린 경우는 5장에서 봅니다.

In [ ]:
bbox_cols = ["x", "y", "w", "h", "area", "category_id", "category_name"]
print("[원본] bbox·클래스 컬럼별 결측 수")
display(ann_df[bbox_cols].isna().sum().rename("결측 수").to_frame().T)
n_category_link = (load_error_df["stage"] == "category_link").sum()
print(f"[원본] categories에서 category_id를 찾지 못한 라벨: {n_category_link}개")

bbox_missing = ann_df[ann_df[bbox_cols].isna().any(axis=1)]
if not bbox_missing.empty:
    display(bbox_missing[["source_json", "annotation_id", *bbox_cols]])

### 4.2 라벨 누락 (파일명에 있는 약의 bbox 없음)

> 출처: eda_kwon.ipynb 셀 4–8, 11–12 (gwon753-dev) 기준. dongwook_eda.ipynb 셀 6 (def-uk)의
> 같은 검사로 결과를 대조했습니다. kwon 셀 11의 약별 누락 비율 표는 손으로 쓴 표 대신 계산합니다.

파일명 앞부분(`K-003351-018357-036637`)에는 사진에 찍힌 약 코드가 들어 있습니다. 파일명에는 있는데
bbox가 없는 약을 라벨 누락으로 봅니다. 반대로 파일명에 없는 약의 bbox가 있는지도 확인합니다.

- 초록 박스: 라벨된 bbox (숫자는 `category_id`)
- 제목의 `누락`: bbox가 빠진 약 코드와 약 이름

In [ ]:
# 파일명의 약 코드(K-003351 → 3351)와 실제로 라벨된 category_id를 사진마다 비교한다.
label_check = compare_name_codes(img_df, ann_df)
missing_df = label_check[label_check["missing_codes"].str.len() > 0].reset_index(drop=True)
extra_df = label_check[label_check["extra_codes"].str.len() > 0].reset_index(drop=True)

print(
    f"[원본] 라벨이 빠진 이미지: {len(missing_df)}장 / {len(img_df)}장, "
    f"빠진 bbox: {missing_df['missing_codes'].str.len().sum()}개 "
    f"(그 사진들에 라벨된 bbox: {missing_df['n_boxes'].sum()}개)"
)
print(f"[원본] 파일명에 없는 약의 bbox가 있는 이미지: {len(extra_df)}장")
display(missing_df.drop(columns="extra_codes"))
if not extra_df.empty:
    display(extra_df.drop(columns="missing_codes"))

# 약별 누락 비율 = 빠진 bbox ÷ (라벨된 bbox + 빠진 bbox)
missing_by_drug = (
    missing_df["missing_codes"]
    .explode()
    .astype(int)
    .value_counts()
    .rename_axis("category_id")
    .rename("n_missing")
    .reset_index()
    .merge(cat_df[["category_id", "dl_name", "n_boxes"]], on="category_id", how="left")
)
missing_by_drug["누락 비율(%)"] = (
    100 * missing_by_drug["n_missing"] / (missing_by_drug["n_boxes"] + missing_by_drug["n_missing"])
).round(1)
print("[원본] 약별 누락")
display(missing_by_drug.rename(columns={"n_missing": "빠진 bbox", "n_boxes": "라벨된 bbox"}))

In [ ]:
# 라벨이 빠진 사진에 라벨된 bbox를 그리고, 제목에 빠진 약을 적는다.
drug_names = dict(zip(cat_df["category_id"], cat_df["dl_name"], strict=True))
if missing_df.empty:
    print("[원본] 라벨이 빠진 이미지가 없습니다.")
else:
    fig, axes = image_grid(len(missing_df), cols=4)
    for ax, row in zip(axes, missing_df.itertuples(), strict=True):
        missing = "\n".join(f"{c} {drug_names.get(c, '?')}" for c in row.missing_codes)
        show_image(ax, TRAIN_IMG_DIR / row.file_name, f"[원본] {row.combo}\n누락: {missing}")
        draw_bboxes(ax, ann_df[ann_df["file_name"] == row.file_name], color="green")
    plt.tight_layout()
    plt.show()

사진 전체의 라벨이 없는 것이 아니라, 사진 속 알약 하나의 bbox만 빠져 있습니다(kwon 셀 8).
그대로 학습하면 다음 문제가 생길 수 있습니다.

1. 빠진 알약을 배경으로 학습합니다. 모델이 알약을 덜 검출하는(미탐) 쪽으로 기울 수 있습니다.
2. 같은 약의 정답이 헷갈립니다. 예를 들어 3351은 다른 사진에서는 알약으로 라벨되어 있는데,
   누락된 사진에서는 배경으로 취급됩니다. 그 약에 대한 확신이 낮아질 수 있습니다.
3. 이 사진이 validation에 들어가면, 모델이 빠진 알약을 맞게 찾아도 오답(false positive)으로 채점됩니다.
4. 사진을 섞는 증강(mosaic 등)이 bbox 없는 알약 조각을 여러 학습 이미지로 퍼뜨립니다.

**처리 방안 (결정 전, kwon 셀 12)**

1. 그대로 쓰기
2. 빠진 bbox를 직접 그려 넣기 (kwon의 `added_annotations.csv`)
3. 해당 사진을 전부 빼기 (같은 사진의 멀쩡한 bbox도 함께 잃음)
4. 최소한 이 사진들은 validation에서 빼기

이 노트북은 누락을 찾고 보여 주기만 합니다. 보정 기록의 형식과 적용 방식은 5.8에서 정리합니다.

### 4.3 메타데이터 결측

> 출처: dongwook_eda.ipynb 셀 8–9 (def-uk) 기준. JJAM99_eda.ipynb 셀 33, 35, 37 (potato)의 결측 유형
> 구분·null 레코드 집계·null 시각화, eda_sohee.ipynb 셀 9–14 (bravedolphinhere)의 빈 문자열 집계와
> 결측 이유 표를 합쳤습니다.

- 원본 JSON을 다시 읽어 `data/README.md` 스키마의 모든 키를 레코드마다 검사합니다. 로더(`ann_df`)는
  키 없음과 null을 모두 NaN으로 만들기 때문입니다. 함수는 `scan_missing_values`입니다.
- 결측 유형: `키 없음`, `null`, `빈 문자열`(공백만 있는 값 포함), `빈 배열/객체`, `NaN/무한대`,
  `bbox 내부 결측`. 0과 False는 결측으로 세지 않습니다(크기 0은 아래에서 따로 봅니다).
- `필수여부`는 스키마 기준입니다(M 필수, O 선택, C 조건부). O·C 키는 비어 있어도 정상일 수 있습니다.

In [ ]:
missing_scan_df = scan_missing_values(TRAIN_ANN_DIR, schema_df)
kinds = ["키 없음", "null", "빈 문자열", "빈 배열/객체", "NaN/무한대", "bbox 내부 결측"]
missing_table = pd.crosstab(missing_scan_df["key"], missing_scan_df["kind"]).reindex(
    columns=kinds, fill_value=0
)
n_checked = missing_scan_df.groupby("key").size()
missing_table.insert(0, "검사 수", n_checked)
missing_table["결측 비율(%)"] = (100 * missing_table[kinds].sum(axis=1) / n_checked).round(1)
missing_table = missing_table.join(schema_df.set_index("key")[["required", "description"]])
missing_table = missing_table[missing_table["결측 비율(%)"] > 0].sort_values(
    ["결측 비율(%)", "null"], ascending=False
)

# 학습에 바로 쓰는 키. 여기가 비면 라벨을 연결하거나 bbox를 만들 수 없다.
core_keys = [
    "images.id",
    "images.file_name",
    "images.width",
    "images.height",
    "annotations.id",
    "annotations.image_id",
    "annotations.category_id",
    "annotations.bbox",
    "categories.id",
]
core_missing = missing_scan_df[
    missing_scan_df["key"].isin(core_keys) & (missing_scan_df["kind"] != "값 있음")
]
print(
    f"[원본] 스키마 키 {n_checked.size}개 중 결측이 있는 키: {len(missing_table)}개 "
    f"(필수 M: {(missing_table['required'] == 'M').sum()}개)"
)
print(f"[원본] 학습에 쓰는 핵심 키의 결측: {len(core_missing)}건")
with pd.option_context("display.max_rows", 100):
    display(missing_table.rename(columns={"required": "필수여부", "description": "스키마 설명"}))

In [ ]:
# null이 있는 images 레코드가 어떤 약인지 본다.
# 한 약의 레코드가 모두 null이면 약 정보 자체가 빈 것이다.
null_scan = missing_scan_df[
    (missing_scan_df["section"] == "images") & (missing_scan_df["kind"] == "null")
]
null_records = (
    null_scan.groupby(["source_json", "record_index", "file_name", "drug_N"])["key"]
    .agg(n_null="size", null_keys=lambda s: ", ".join(k.removeprefix("images.") for k in s))
    .reset_index()
)
print(
    f"[원본] null 키가 있는 images 레코드: {len(null_records)}개, "
    f"사진 {null_records['file_name'].nunique()}장"
)
null_by_drug = null_records.groupby("drug_N").agg(
    null_records=("source_json", "size"), n_null_keys=("n_null", "max")
)
drug_info = ann_df.groupby("drug_N").agg(dl_name=("dl_name", "first"), all_records=("x", "size"))
null_by_drug = null_by_drug.join(drug_info)
print("[원본] 약별 null 레코드 수 / 그 약의 전체 레코드 수")
display(null_by_drug)
print("[원본] null인 키 목록 (중복 제거)")
display(null_records["null_keys"].value_counts().rename("레코드 수").to_frame())

In [ ]:
# 0은 결측으로 세지 않았지만, 크기(mm)의 0.0은 값이 없다는 뜻으로 쓰였다(kwon 정의서).
size_cols = ["thick", "leng_long", "leng_short"]
size_zero = ann_df[size_cols].eq(0)
print("[원본] 크기(mm)가 0인 bbox 수")
display(size_zero.sum().rename("0인 bbox 수").to_frame().T)
display(
    ann_df.loc[size_zero.any(axis=1)]
    .groupby(["drug_N", "dl_name", *size_cols])
    .size()
    .rename("bbox 수")
    .reset_index()
)

In [ ]:
# null 레코드의 알약(빨강)에도 bbox와 클래스가 연결되어 있는지 본다. 조합이 다른 사진 최대 4장.
null_examples = (
    null_records.assign(combo=null_records["file_name"].str.split("_").str[0])
    .drop_duplicates("combo")
    .head(4)
)
if null_examples.empty:
    print("[원본] null 레코드가 없습니다.")
else:
    fig, axes = image_grid(len(null_examples), cols=4)
    for ax, row in zip(axes, null_examples.itertuples(), strict=True):
        boxes = ann_df[ann_df["file_name"] == row.file_name]
        is_null = boxes["source_json"] == row.source_json
        title = f"[원본] {row.combo}\n빨강: {row.drug_N} (null 키 {row.n_null}개)"
        show_image(ax, TRAIN_IMG_DIR / row.file_name, title)
        draw_bboxes(ax, boxes[~is_null], color="gray")
        draw_bboxes(ax, boxes[is_null], color="red")
    plt.tight_layout()
    plt.show()

**결측을 어떻게 읽을까**

빈 값만 보고 "마크 없음", "분할선 없음"으로 확정하지 않습니다. 정보가 원래 없는 것인지 수집되지 않은
것인지는 데이터 설명과 실제 이미지를 함께 봐야 구분할 수 있습니다(eda_sohee 셀 10). 아래는 비어 있는
이유로 생각해 볼 수 있는 것입니다.

| 키 | 비어 있는 이유로 생각해 볼 수 있는 것 |
|---|---|
| `print_front`, `print_back` | 그 면에 식별문자가 없을 수 있음 |
| `line_front`, `line_back` | 분할선이 없을 수 있음 |
| `color_class2` | 두 번째 색이 없는 단색 알약일 수 있음 |
| `mark_code_*` | 마크가 없거나 마크 정보가 수집되지 않았을 수 있음 |
| `di_company_mf`, `di_company_mf_en` | 위탁제조사가 없거나 영문명이 제공되지 않았을 수 있음 |
| `annotations.segmentation` | 이 데이터는 bbox만 라벨링함 (스키마상 선택 O) |

- 빈 값 대부분은 사진이 아니라 JSON의 약 설명이 비어 있는 것입니다. `dl_company_en`처럼 사진으로 확인할
  수 없는 키도 있고, `print_back`처럼 사진과 비교해 볼 수 있는 키도 있습니다(eda_sohee 셀 13).
- `null` 18건은 모두 `K-016548`(가바토파정 100mg)의 레코드이고, 이 약의 레코드 18개 전부입니다.
  키는 있고 값이 `null`이며(키 없음 0건), 크기(mm)도 0입니다. 사진이 아니라 약 정보 하나가 통째로
  비어 있는 것입니다(원본 상태 관측값).
- 위 그림처럼 이 레코드에도 bbox와 클래스는 연결되어 있으므로 메타데이터 결측 자체는 bbox 학습에
  영향이 없습니다(JJAM 셀 37). 다만 `K-001900-016548-019607-033009` 70° 사진에서는 16548 박스가
  다른 약(33009)과 같은 좌표에 붙어 있고, 실제 16548 알약에는 박스가 없습니다. 이것은 결측이 아니라
  좌표 오류로, 5.2에서 봅니다(JJAM 셀 38).
- 메타데이터를 학습 입력으로 쓰지 않는 한 결측을 채울 필요는 없어 보입니다(dongwook 셀 8). 알약 실제
  크기를 쓰는 7.3에서는 크기 0인 약을 빼고 계산합니다.

## 5. 라벨 오류와 이상치

> 출처: JJAM99_eda.ipynb 셀 19–31, 38–40, 45–48, 53–60 (potato), eda_kwon.ipynb 셀 14–23, 54–86
> (gwon753-dev), EDA_jang.ipynb 셀 8–11 (younghoon1215), eda_sohee.ipynb 셀 15–22,
> sohee_eda.ipynb 셀 2, 11 (bravedolphinhere)

값은 있지만 틀렸거나 이상해 보이는 라벨을 찾습니다. 5.1–5.7은 모두 `[원본]` `ann_df`에서 찾기만 하고
고치지 않습니다. 보정 기록을 모아 메모리에서 적용하고 원본과 비교하는 일은 5.8에서 합니다.

| 절 | 무엇을 찾나 | 판정 |
|---|---|---|
| 5.1 | 이미지 밖 좌표, 크기 0 이하 | 오류 |
| 5.2 | 다른 약과 좌표가 같은 박스 | 오류 (어느 쪽이 틀렸는지는 사람이 판정) |
| 5.3 | 알약 없이 배경만 있는 박스 | 오류 |
| 5.4 | 기록된 `area`와 `w×h`가 다름 | 확인 대상 (bbox 오류 확정 아님) |
| 5.5 | 면적·가로세로 비율이 분포에서 먼 박스 | 후보 (오류 아님) |
| 5.6 | 붙은·부서진 알약 | 확인 대상 |
| 5.7 | 알약보다 넉넉하게 그린 박스 (여백) | 추정 `[RUN_HEAVY]` |
| 5.8 | 위 결과 요약, 보정 기록 통합과 원본·보정 비교 | |

아래 셀은 5장 전체가 쓰는 검사를 한 번에 계산합니다. 박스 픽셀 측정(kwon 방식)은 train 사진을
한 장씩 한 번 읽으며 약 10초 걸립니다.

In [ ]:
AREA_TOL = 1.0  # |area − w×h|가 이보다 크면 area 불일치 (JJAM 셀 46)
TOUCH_GAP = 10  # 같은 사진의 두 박스 간격이 이 px 이하이면 쌍으로 본다 (kwon 셀 58)
DUP_IOU = 0.9  # 쌍의 IoU가 이 이상이면 좌표 중복, 미만이면 맞닿은 알약 (kwon 셀 57)
EMPTY_DIFF_MAX = 15  # 박스 안 diff_p90이 이보다 작으면 빈 배경 (kwon 셀 71)

flag_df = bbox_flags(ann_df, area_tol=AREA_TOL)
pair_df = find_box_pairs(ann_df, max_gap=TOUCH_GAP)
start = time.perf_counter()
pixel_df = measure_box_pixels(ann_df, TRAIN_IMG_DIR)
measured = pixel_df["status"] == "측정"
print(f"[원본] 박스 픽셀 측정: {measured.sum()}개 ({time.perf_counter() - start:.0f}초)")
display(pixel_df["status"].value_counts().rename("bbox 수").to_frame().T)

### 5.1 이미지 밖 좌표 (크기 0 이하 포함)

> 출처: JJAM99_eda.ipynb 셀 19, 21–31 (potato) 기준. eda_kwon.ipynb 셀 56 (gwon753-dev),
> eda_sohee.ipynb 셀 18 (bravedolphinhere), EDA_jang.ipynb 셀 9 (younghoon1215)의 같은 검사로 대조했습니다.

박스 일부라도 이미지(`width × height`) 밖에 있거나 너비·높이가 0 이하인 라벨을 찾습니다.
보통 화면은 이미지 범위만 보여 주므로, 이미지 밖 박스는 좌표를 넓혀 그려야 보입니다(JJAM 셀 22).

- 왼쪽: 원본 사진 범위. 이미지 밖 박스는 보이지 않습니다.
- 가운데: 좌표를 넓힌 화면. 초록 점선이 사진 경계, 빨강이 문제 박스, 회색이 같은 사진의 나머지 라벨입니다.
- 오른쪽: 같은 조합을 다른 각도로 찍은 사진의 같은 약 박스(JJAM 셀 30).

In [ ]:
coord_error_df = ann_df[flag_df["out_of_image"] | flag_df["non_positive"]].join(
    flag_df[["out_of_image", "non_positive"]]
)
print(
    f"[원본] 이미지 밖 좌표: {flag_df['out_of_image'].sum()}개, "
    f"크기 0 이하: {flag_df['non_positive'].sum()}개"
)
cols = ["source_json", "annotation_id", "category_id", "dl_name", *BBOX_COLS, "width", "height"]
display(coord_error_df[[*cols, "out_of_image", "non_positive"]])

for row in coord_error_df.itertuples():
    bad_box = (row.x, row.y, row.w, row.h)
    others = ann_df[(ann_df["file_name"] == row.file_name) & (ann_df.index != row.Index)]
    combo = row.file_name.split("_")[0]
    same_drug = ann_df[
        ann_df["file_name"].str.startswith(combo + "_")
        & (ann_df["file_name"] != row.file_name)
        & (ann_df["category_id"] == row.category_id)
    ]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
    show_image(axes[0], TRAIN_IMG_DIR / row.file_name, f"[원본] 사진 범위 ({row.camera_la}°)")
    draw_bboxes(axes[0], others, color="gray")
    bad_text = [int(v) for v in bad_box]
    title = f"[원본] 넓힌 좌표 (초록 점선: 사진 경계)\n{row.category_id} bbox={bad_text}"
    show_image(axes[1], TRAIN_IMG_DIR / row.file_name, title)
    draw_image_bounds(axes[1], row.width, row.height)
    draw_bboxes(axes[1], others, color="gray")
    draw_bbox(axes[1], bad_box, str(row.category_id))
    zoom_to(axes[1], [(0, 0, row.width, row.height), bad_box], margin=0.03)
    axes[1].axis("on")
    if same_drug.empty:
        axes[2].set_title("같은 조합의 다른 사진 없음")
    else:
        ref = same_drug.iloc[0]
        ref_text = [int(v) for v in ref[BBOX_COLS]]
        title = f"[원본] 비교: {ref['camera_la']}° 사진의 {ref['category_id']}\nbbox={ref_text}"
        show_image(axes[2], TRAIN_IMG_DIR / ref["file_name"], title)
        draw_bboxes(axes[2], same_drug.iloc[[0]], color="lime")
    fig.suptitle(f"{row.file_name} | {row.dl_name}", fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# 새 위치 제안 (JJAM 셀 26): 믿을 수 없는 x는 버리고, y·크기가 비슷한 색상 윤곽을 사진에서 찾는다.
# 메모리에만 두고 5.8에서 미검수 보정 기록으로 다룬다.
shape_proposals = {}
for row in ann_df[flag_df["out_of_image"]].itertuples():
    bbox, note = propose_bbox_by_shape(TRAIN_IMG_DIR / row.file_name, (row.x, row.y, row.w, row.h))
    shape_proposals[(row.source_json, row.annotation_id)] = bbox
    print(f"[원본] {row.category_id} 라벨 {row.annotation_id}: 제안 bbox {bbox} ({note})")

- `K-003351-016262-018357` 75° 사진의 18357(종근당글리아티린연질캡슐) 1개입니다. bbox `[6567, 625, 311, 315]`의
  x가 사진 너비 976을 훨씬 넘습니다. 크기 0 이하는 없습니다. 다섯 노트북이 모두 같은 1개를 찾았습니다(원본 상태 관측값).
- 같은 조합의 다른 각도 사진에서는 18357 박스가 정상 범위에 있습니다. 압축 전 원본 데이터도 같은 값이어서,
  압축 해제 오류가 아니라 annotation 자체가 틀린 것입니다(JJAM 셀 31).
- 사진 속 18357 캡슐에는 박스가 없습니다. 그대로 쓰면 학습 도구에 따라 좌표를 잘라내거나 라벨·이미지를
  건너뛰어, 이 캡슐이 배경으로 학습될 수 있습니다.
- 고치는 값이 둘입니다. kwon은 박스 없는 알약을 찾아 옮겼고(`x=574`, x=567 앞에 6이 붙은 오타로 추정,
  kwon 셀 83), JJAM은 의도된 값을 추측하지 않고 위 셀처럼 색상 윤곽에서 위치를 다시 찾았습니다(JJAM 셀 24–26).
  두 값은 거의 같습니다(kwon `[574, 626, 300, 309]`, JJAM 방식 `[572, 625, 303, 312]`, IoU 약 0.98). 5.8에서 나란히
  그립니다.

### 5.2 다른 약과 좌표 중복

> 출처: eda_kwon.ipynb 셀 57–60, 67–68 (gwon753-dev) 기준. EDA_jang.ipynb 셀 9 (younghoon1215)의
> 정답 약 판정, JJAM99_eda.ipynb 셀 38–40 (potato)의 같은 사진 확인을 합쳤습니다.

같은 사진에서 간격이 `TOUCH_GAP`(10px) 이하인 박스 쌍을 찾고 IoU로 나눕니다.

- IoU ≥ `DUP_IOU`(0.9): 두 약에 같은 좌표를 붙인 라벨 오류입니다. 한 알약에 박스가 두 개 붙고,
  다른 알약 하나에는 박스가 없습니다.
- IoU < 0.9: 실제로 맞닿은 알약입니다(5.6).

어느 쪽 박스가 틀렸는지는 계산으로 정할 수 없습니다. 박스 자리의 알약을 같은 약의 다른 사진과 색·모양·각인으로
비교해 사람이 정했습니다(jang 셀 9, kwon 셀 67). 아래 `DUPLICATE_WRONG`이 그 판정입니다.

In [ ]:
# 사람이 판정한 위치가 틀린 쪽 약 (jang 셀 9, kwon 셀 67). 키: 사진, 값: 틀린 박스의 category_id
DUPLICATE_WRONG = {
    "K-001900-016548-019607-033009_0_2_0_2_70_000_200.png": 16548,
    "K-003351-020238-031863_0_2_0_2_70_000_200.png": 20238,
    "K-003351-029667-031863_0_2_0_2_70_000_200.png": 3351,
}
dup_pair_df = pair_df[pair_df["iou"] >= DUP_IOU].copy()
dup_pair_df["wrong_category"] = dup_pair_df["file_name"].map(DUPLICATE_WRONG).astype("Int64")
dup_pair_df["right_category"] = [
    pd.NA if pd.isna(wrong) else (b if wrong == a else a)
    for a, b, wrong in zip(
        dup_pair_df["category_id_1"],
        dup_pair_df["category_id_2"],
        dup_pair_df["wrong_category"],
        strict=True,
    )
]
stale = sorted(set(DUPLICATE_WRONG) - set(dup_pair_df["file_name"]))
print(
    f"[원본] 간격 {TOUCH_GAP}px 이하 박스 쌍: {len(pair_df)}개 "
    f"(좌표 중복 {len(dup_pair_df)}개, 맞닿음 {len(pair_df) - len(dup_pair_df)}개)"
)
print(
    f"판정 없는 좌표 중복: {dup_pair_df['wrong_category'].isna().sum()}개, "
    f"쌍이 없어진 판정: {stale}"
)
display(
    dup_pair_df.assign(
        name_1=dup_pair_df["category_id_1"].map(drug_names),
        name_2=dup_pair_df["category_id_2"].map(drug_names),
    )[["file_name", "category_id_1", "name_1", "category_id_2", "name_2", "iou", "wrong_category"]]
)

In [ ]:
# 빨간 점선: 위치가 틀린 박스, 파랑: 그 자리에 실제로 있는 알약의 박스, 회색: 나머지 라벨
if dup_pair_df.empty:
    print("[원본] 좌표 중복이 없습니다.")
else:
    fig, axes = image_grid(len(dup_pair_df), cols=3, cell=(5, 6.8))
    for ax, pair in zip(axes, dup_pair_df.itertuples(), strict=True):
        boxes = ann_df[ann_df["file_name"] == pair.file_name]
        is_wrong = boxes["category_id"] == pair.wrong_category
        is_right = boxes["category_id"] == pair.right_category
        wrong = "판정 없음" if pd.isna(pair.wrong_category) else pair.wrong_category
        title = (
            f"[원본] {pair.file_name.split('_')[0]}\n"
            f"빨간 점선: {wrong} {drug_names.get(pair.wrong_category, '')} (틀림)\n"
            f"파랑: {pair.right_category} {drug_names.get(pair.right_category, '')} (이 자리 알약)"
        )
        show_image(ax, TRAIN_IMG_DIR / pair.file_name, title)
        draw_bboxes(ax, boxes[~is_wrong & ~is_right], color="gray")
        draw_bboxes(ax, boxes[is_right], color="tab:blue")
        draw_bboxes(ax, boxes[is_wrong], label_col=None, color="red", linestyle="--")
    plt.tight_layout()
    plt.show()

- 좌표 중복은 3쌍이고 모두 IoU 1.0, 모두 70° 사진입니다. jang이 처음 찾았고 kwon과 JJAM(16548 사진)이 확인했습니다(원본 상태 관측값).
  - `K-001900-016548-019607-033009`: 16548(가바토파정 100mg) 박스가 33009 자리에 있습니다. 16548 알약(노란 원형)에는 박스가
    없습니다.
    이 16548 레코드는 4.3의 null 메타데이터 레코드이기도 합니다(JJAM 셀 38–40).
  - `K-003351-020238-031863`: 20238 박스가 3351 자리에 있습니다.
  - `K-003351-029667-031863`: 3351 박스가 31863 자리에 있습니다.
- 그대로 학습하면 한 알약을 두 클래스로 가르치고, 박스 없는 알약은 배경으로 가르칩니다.
- kwon은 맞는 쪽 박스는 두고 틀린 쪽 박스를 박스 없는 알약 위치로 옮겼습니다(`fixed_annotations.csv`, 5.8).

### 5.3 빈 배경 박스

> 출처: eda_kwon.ipynb 셀 69–73 (gwon753-dev)

박스 안 픽셀이 배경과 얼마나 다른지 잽니다(kwon 셀 69).

- `fill`: 박스 넓이 중 알약 픽셀 비율. 정상이면 0.6 안팎이지만 투명·어두운 캡슐은 알약이 있어도 낮게 나와 판정에 쓰지 않습니다.
- `diff_p90`: 박스 안 픽셀의 배경 대비 색 차이 상위 10% 값. 투명·어두운 알약도 윤곽과 각인 때문에 크게 나오고,
  빈 배경만 0에 가깝습니다. **판정에는 이 값을 씁니다**(`EMPTY_DIFF_MAX` = 15 미만).

이미지 밖 박스(5.1)는 잴 픽셀이 없어 여기서 빼고 셉니다. kwon 셀 71은 이 박스도 빈 배경으로 세서 2개였습니다.

In [ ]:
empty_mask = measured & (pixel_df["diff_p90"] < EMPTY_DIFF_MAX)
empty_df = ann_df[empty_mask].join(pixel_df[["fill", "diff_p90"]])
print(
    f"[원본] 빈 배경 박스: {len(empty_df)}개 / 측정 {measured.sum()}개 "
    f"(diff_p90 < {EMPTY_DIFF_MAX})"
)
print(
    "diff_p90가 가장 낮은 5개:", pixel_df.loc[measured, "diff_p90"].nsmallest(5).round(1).tolist()
)
display(
    empty_df[
        ["source_json", "annotation_id", "category_id", "dl_name", *BBOX_COLS, "fill", "diff_p90"]
    ]
)
display(pixel_df.loc[measured, ["fill", "diff_p90"]].describe().round(3).T)

In [ ]:
# diff_p90가 가장 낮은(배경과 가장 비슷한) 박스를 잘라 본다. 빨강: 빈 배경 판정, 초록: 알약 있음
lowest = pixel_df.loc[measured, "diff_p90"].nsmallest(12).index
fig, axes = image_grid(len(lowest), cols=6, cell=(2.6, 3.2))
for ax, i in zip(axes, lowest, strict=True):
    row = ann_df.loc[i]
    color = "red" if empty_mask[i] else "green"
    status = "빈 배경" if empty_mask[i] else "알약 있음"
    show_image(ax, TRAIN_IMG_DIR / row["file_name"])
    draw_bbox(ax, row[BBOX_COLS], color=color)
    zoom_to(ax, [row[BBOX_COLS]], margin=0.05, limit=(row["width"], row["height"]))
    title = f"{row['category_id']}  diff {pixel_df.loc[i, 'diff_p90']:.1f}\n{status}"
    ax.set_title(title, fontsize=9, color=color)
fig.suptitle(
    f"[원본] 배경과 가장 비슷한 박스 {len(lowest)}개 (빨강: diff_p90 < {EMPTY_DIFF_MAX})",
    x=0.01,
    ha="left",
)
plt.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

- 빈 배경 박스는 `K-003351-020014-020238` 90° 사진의 3351 박스 1개입니다(diff_p90 약 7.5). 다음으로 낮은 박스는
  24 이상이라 기준 15가 두 무리를 잘 가릅니다(원본 상태 관측값).
- 실제 3351 알약은 바로 옆에 박스 없이 있습니다(kwon 셀 83 케이스 3). kwon은 이 박스를 그 알약 위치로 옮겼습니다(5.8).

### 5.4 area ≠ w×h

> 출처: JJAM99_eda.ipynb 셀 45–48 (potato) 기준. sohee_eda.ipynb 셀 2, 11 (bravedolphinhere)의
> 같은 검사로 대조했습니다.

JSON의 `area`와 bbox `w × h`의 차이가 `AREA_TOL`(1px²)보다 큰 라벨을 찾습니다. `area`가 분할 영역 넓이일 수도
있어 불일치만으로 bbox 오류를 확정하지 않습니다(JJAM 셀 45). YOLO 형식 라벨은 bbox만 쓰지만, COCO 방식 평가는
`area`로 작은·중간·큰 물체를 나누므로 값이 틀리면 크기별 점수가 달라질 수 있습니다.

In [ ]:
area_df = ann_df[flag_df["area_mismatch"]].copy()
area_df["bbox_area"] = area_df["w"] * area_df["h"]
area_df["area_diff"] = area_df["area"] - area_df["bbox_area"]
area_df["area_diff_pct"] = (100 * area_df["area_diff"] / area_df["bbox_area"]).round(2)
# sohee는 상대 오차 0.1%까지 같다고 봤다(math.isclose(rel_tol=1e-3, abs_tol=1)).
sohee_same = [
    math.isclose(a, b, rel_tol=1e-3, abs_tol=1.0)
    for a, b in zip(area_df["area"], area_df["bbox_area"], strict=True)
]
print(
    f"[원본] area 불일치: {len(area_df)}개 (|차이| > {AREA_TOL}px²), "
    f"sohee 기준(상대 오차 0.1% 초과): {len(area_df) - sum(sohee_same)}개"
)
cols = ["file_name", "annotation_id", "category_id", "dl_name", *BBOX_COLS, "area", "bbox_area"]
display(
    area_df[[*cols, "area_diff", "area_diff_pct"]]
    .assign(sohee_same=sohee_same)
    .rename(columns={"bbox_area": "w×h", "area_diff": "차이", "area_diff_pct": "차이율(%)"})
)

In [ ]:
fig, axes = image_grid(len(area_df), cols=3, cell=(4.5, 4.8))
for ax, (_, row) in zip(axes, area_df.iterrows(), strict=True):
    title = (
        f"[원본] {row['category_id']} {row['dl_name']}\n"
        f"area={row['area']:g}, w×h={row['bbox_area']:g} ({row['area_diff_pct']:+.2f}%)"
    )
    show_image(ax, TRAIN_IMG_DIR / row["file_name"], title)
    draw_bbox(ax, row[BBOX_COLS])
    zoom_to(ax, [row[BBOX_COLS]], margin=0.3, limit=(row["width"], row["height"]))
plt.tight_layout()
plt.show()

In [ ]:
# 잘린 캡슐 (JJAM 셀 47–48): 박스 중앙의 색상 덩어리 전체를 감싸는 bbox를 다시 제안한다.
# 메모리에만 두고 5.8에서 미검수 보정 기록으로 다룬다.
CUT_CAPSULE = (
    "K-003351-019232-029667_json/K-019232/K-003351-019232-029667_0_2_1_2_70_000_200.json",
    5178,
)
capsule_row = ann_df[
    (ann_df["source_json"] == CUT_CAPSULE[0]) & (ann_df["annotation_id"] == CUT_CAPSULE[1])
]
capsule_proposal = None
if len(capsule_row) != 1:
    print(f"[원본] 잘린 캡슐 제안: {NOT_RUN} (대상 라벨 {len(capsule_row)}개)")
else:
    row = capsule_row.iloc[0]
    old_bbox = tuple(int(v) for v in row[BBOX_COLS])
    capsule_proposal, note = propose_bbox_at_center(TRAIN_IMG_DIR / row["file_name"], old_bbox)
    print(
        f"[원본] {row['category_id']} 라벨 {CUT_CAPSULE[1]}: {list(old_bbox)} "
        f"→ 제안 {capsule_proposal} ({note})"
    )
    fig, axes = plt.subplots(1, 2, figsize=(8, 4.8))
    panels = [("[원본] bbox", old_bbox, "red"), ("[제안] JJAM 방식", capsule_proposal, "lime")]
    for ax, (title, bbox, color) in zip(axes, panels, strict=True):
        show_image(ax, TRAIN_IMG_DIR / row["file_name"], f"{title}\n{bbox}")
        if bbox is not None:
            draw_bbox(ax, bbox, color=color)
        boxes = [b for b in (old_bbox, capsule_proposal) if b is not None]
        zoom_to(ax, boxes, margin=0.2, limit=(row["width"], row["height"]))
    fig.suptitle(f"{row['file_name']} | {row['dl_name']}", fontsize=10)
    plt.tight_layout()
    plt.show()

- area 불일치는 6개로 JJAM 셀 46과 같습니다. sohee 결과(5개)와의 차이는 계산 조건 때문입니다. sohee는 상대 오차
  0.1%까지 같다고 봐서 차이가 작은 1개가 빠졌습니다(위 표 `sohee_same`). 원본 상태 관측값입니다.
- `K-003351-019232-029667` 70° 사진의 19232(라벨 5178)는 기록된 area(79112)가 w×h(55912)보다 크고,
  박스가 캡슐 아래쪽을 자릅니다. JJAM은 bbox를 `[527, 211, 224, 323]`으로 키우고 area를 w×h로 맞춘 수정본을
  `data/processed/bbox_area/`에 저장했습니다(JJAM 셀 47–48). 이 노트북은 같은 방법으로 제안만 다시 계산하며,
  결과가 JJAM 값과 같습니다.
- 나머지 5개는 그림에서 박스가 알약을 감싸고 있어 bbox가 아니라 `area` 값만 다른 것으로 보입니다.

### 5.5 면적·가로세로 비율 이상치 후보 (IQR, 오류 아님)

> 출처: JJAM99_eda.ipynb 셀 19, 21–24 (potato) 기준. eda_sohee.ipynb 셀 19–22 (bravedolphinhere)의
> 가로세로 비율 기준을 합쳤습니다.

- 면적 비율 = `w × h ÷ (width × height)`, 가로세로 비율 = `w ÷ h`
- 5.1의 좌표 오류 박스를 뺀 나머지에서 Q1, Q3을 구하고, `Q1 − 1.5×IQR`보다 작거나 `Q3 + 1.5×IQR`보다 크면 후보로 둡니다.
- 정답의 허용 범위가 아니라 눈으로 볼 후보를 추리는 통계 기준입니다. 크기 분포 자체는 6.2에서 봅니다.

In [ ]:
valid = ann_df[
    ~(flag_df["out_of_image"] | flag_df["non_positive"]) & ann_df[BBOX_COLS].notna().all(axis=1)
]
ratio_df = pd.DataFrame(
    {
        "area_ratio": valid["w"] * valid["h"] / (valid["width"] * valid["height"]),
        "aspect": valid["w"] / valid["h"],
    }
)
ratio_names = {"area_ratio": "면적 비율", "aspect": "가로세로 비율"}
bounds = {col: iqr_bounds(ratio_df[col]) for col in ratio_df}
outlier_flags = pd.DataFrame({col: ~ratio_df[col].between(*bounds[col]) for col in ratio_df})

summary = pd.DataFrame(
    {
        "하한": [bounds[c][0] for c in ratio_df],
        "상한": [bounds[c][1] for c in ratio_df],
        "하한 미만": [(ratio_df[c] < bounds[c][0]).sum() for c in ratio_df],
        "상한 초과": [(ratio_df[c] > bounds[c][1]).sum() for c in ratio_df],
        "후보": outlier_flags.sum().to_numpy(),
    },
    index=[ratio_names[c] for c in ratio_df],
)
print(f"[원본] IQR 후보 (검사한 bbox {len(valid)}개, 좌표 오류 제외)")
display(summary.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, col in zip(axes, ratio_df, strict=True):
    ax.hist(ratio_df[col], bins=40, color="steelblue")
    for value, color in zip(bounds[col], ["orange", "red"], strict=True):
        ax.axvline(value, color=color, ls="--")
    ax.set(
        title=f"[원본] {ratio_names[col]} (점선: IQR 하한·상한)",
        xlabel=ratio_names[col],
        ylabel="bbox 수",
    )
plt.tight_layout()
plt.show()

# 후보가 어떤 약에 몰려 있는지 본다.
by_drug = (
    outlier_flags.join(ann_df[["category_id", "dl_name"]])
    .groupby(["category_id", "dl_name"])[["area_ratio", "aspect"]]
    .sum()
    .join(cat_df.set_index("category_id")["n_boxes"], on="category_id")
)
by_drug = by_drug[by_drug[["area_ratio", "aspect"]].sum(axis=1) > 0].sort_values(
    ["area_ratio", "aspect"], ascending=False
)
print("[원본] 약별 후보 수 (n_boxes: 그 약의 전체 bbox 수)")
display(by_drug.rename(columns=ratio_names))

In [ ]:
# 기준에서 가장 멀리 벗어난 후보를 기준마다 4개씩 본다.
examples = []
for col in ratio_df:
    low, high = bounds[col]
    distance = np.maximum(low - ratio_df[col], ratio_df[col] - high)
    examples += [(col, i) for i in distance[outlier_flags[col]].nlargest(4).index]
fig, axes = image_grid(len(examples), cols=4, cell=(3.5, 4.2))
for ax, (col, i) in zip(axes, examples, strict=True):
    row = ann_df.loc[i]
    value = ratio_df.loc[i, col]
    title = f"[원본] {ratio_names[col]} {value:.3f}\n{row['category_id']} {row['dl_name']}"
    show_image(ax, TRAIN_IMG_DIR / row["file_name"], title)
    draw_bbox(ax, row[BBOX_COLS])
    zoom_to(ax, [row[BBOX_COLS]], margin=0.3, limit=(row["width"], row["height"]))
plt.tight_layout()
plt.show()

- 면적 비율 후보는 24개이고 모두 상한(약 14.2%)을 넘는 큰 박스입니다(JJAM 셀 24, eda_sohee 셀 20과 같은 수).
  가로세로 비율 후보는 82개입니다(eda_sohee 셀 22). 원본 상태 관측값입니다.
- JJAM과 sohee가 후보 그림을 모두 본 결과, 박스가 큰 캡슐이나 길쭉한 알약을 잘 감싸고 있어 크기만으로 잘못된 라벨이라
  볼 근거는 없었습니다(JJAM 셀 23–24). 기울어진 긴 알약은 축에 평행한 bbox 안에 배경이 더 들어갑니다.
- 후보가 몇몇 약에 몰려 있으므로(위 약별 표), 전체 분포가 아니라 같은 약끼리 비교해야 이상한 박스가 드러납니다.
  같은 약끼리 비교하는 검사는 5.6의 `area_ratio`입니다.

### 5.6 붙은·부서진 알약

> 출처: eda_kwon.ipynb 셀 54–66, 84–86 (gwon753-dev)

1. 알약 상태 값: `drug_S`에 정상 외의 값이 있는가
2. 붙어 있는 알약: 같은 사진에서 간격 10px 이하이고 IoU가 0.9 미만인 박스 쌍 (IoU 0.9 이상은 5.2)
3. 모양 이상: 박스 안 알약 픽셀 넓이를 같은 약끼리 비교하고(`area_ratio` = 같은 약 중앙값 대비),
   윤곽이 패였는지(`solidity`), 덩어리가 여럿인지(`n_blobs`) 봅니다.

**주의**: 투명한 캡슐, 어두운 캡슐, 배경과 색이 비슷한 흰 알약은 OpenCV가 알약을 조각내 잡아서 정상이어도
의심으로 나옵니다. 그래서 그림으로 반드시 확인합니다(kwon 셀 61).

In [ ]:
print("[원본] drug_S 값")
display(ann_df["drug_S"].value_counts(dropna=False).rename("bbox 수").to_frame().T)

touch_df = pair_df[pair_df["iou"] < DUP_IOU]
print(f"[원본] 맞닿은 박스 쌍 (간격 ≤ {TOUCH_GAP}px, IoU < {DUP_IOU}): {len(touch_df)}개")
display(touch_df[["file_name", "category_id_1", "category_id_2", "gap_px", "iou"]].round(3))
if not touch_df.empty:
    fig, axes = image_grid(len(touch_df), cols=4)
    for ax, pair in zip(axes, touch_df.itertuples(), strict=True):
        boxes = ann_df[ann_df["file_name"] == pair.file_name]
        is_pair = boxes["category_id"].isin([pair.category_id_1, pair.category_id_2])
        title = (
            f"[원본] {pair.file_name.split('_')[0]} ({boxes['camera_la'].iloc[0]}°)\n"
            f"{pair.category_id_1} ↔ {pair.category_id_2}, IoU {pair.iou:.3f}"
        )
        show_image(ax, TRAIN_IMG_DIR / pair.file_name, title)
        draw_bboxes(ax, boxes[~is_pair], color="gray")
        draw_bboxes(ax, boxes[is_pair], color="red")
    plt.tight_layout()
    plt.show()

In [ ]:
AREA_LOW, AREA_HIGH = 0.7, 1.3  # 같은 약 중앙값 대비 알약 넓이가 이 범위 밖이면 의심
SOLIDITY_MIN = 0.9  # 이보다 낮으면 윤곽이 패인 것(깨짐, 이 빠짐, 두 알 붙음)으로 의심
EMPTY_MAX = 0.1  # 알약 넓이가 같은 약 중앙값의 10% 미만이면 박스 안에 알약이 없는 것

shape_df = ann_df.loc[measured, ["file_name", "category_id", *BBOX_COLS, "width", "height"]].join(
    pixel_df[["pill_area", "solidity", "n_blobs"]]
)
median_area = shape_df.groupby("category_id")["pill_area"].transform("median")
shape_df["area_ratio"] = shape_df["pill_area"] / median_area
reasons = pd.DataFrame(
    {
        "알약 없음(라벨 오류?)": shape_df["area_ratio"] < EMPTY_MAX,
        "작음(부서짐?)": shape_df["area_ratio"].between(EMPTY_MAX, AREA_LOW, inclusive="left"),
        "큼(붙음?)": shape_df["area_ratio"] > AREA_HIGH,
        "윤곽 패임": shape_df["solidity"] < SOLIDITY_MIN,
        "덩어리 여러 개": shape_df["n_blobs"] >= 2,
    }
)
shape_df["reason"] = reasons.apply(lambda r: ", ".join(r.index[r]), axis=1)
shape_suspect_df = shape_df[shape_df["reason"] != ""].sort_values("solidity")

print(
    f"[원본] 모양 검사 bbox: {len(shape_df)}개 / 의심: {len(shape_suspect_df)}개 "
    "(이유는 겹칠 수 있음)"
)
display(reasons.sum().rename("bbox 수").to_frame().T)
display(
    shape_suspect_df[
        ["file_name", "category_id", "area_ratio", "solidity", "n_blobs", "reason"]
    ].round(3)
)

In [ ]:
# 의심 박스(빨간 제목)와 같은 약의 정상 박스(초록 제목)를 나란히 본다. 앞의 12개만 그린다.
shown = shape_suspect_df.head(12)
normal = shape_df[shape_df["reason"] == ""]
fig, axes = image_grid(2 * len(shown), cols=4, cell=(2.8, 3.3))
for i, s in enumerate(shown.itertuples()):
    ax_s, ax_r = axes[2 * i], axes[2 * i + 1]
    show_image(ax_s, TRAIN_IMG_DIR / s.file_name)
    zoom_to(ax_s, [(s.x, s.y, s.w, s.h)], margin=0.02, limit=(s.width, s.height))
    ax_s.set_title(
        f"의심 {s.category_id}\n" + s.reason.replace(", ", "\n"), fontsize=8, color="red"
    )
    same = normal[normal["category_id"] == s.category_id]
    if not same.empty:
        ref = same.loc[(same["area_ratio"] - 1).abs().idxmin()]
        show_image(ax_r, TRAIN_IMG_DIR / ref["file_name"])
        zoom_to(ax_r, [ref[BBOX_COLS]], margin=0.02, limit=(ref["width"], ref["height"]))
        ax_r.set_title(f"정상 예시 {s.category_id}", fontsize=8, color="green")
    else:
        ax_r.set_title(f"{s.category_id}: 정상 판정 박스 없음", fontsize=8)
fig.suptitle(f"[원본] 모양 의심 {len(shown)}개 / 전체 {len(shape_suspect_df)}개", x=0.01, ha="left")
plt.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

- `drug_S`는 모두 `정상알약`입니다.
- 맞닿은 박스 쌍은 `K-003351-033880-035206`의 75°, 90° 사진 2개입니다(같은 장면). IoU 0.02 정도로 살짝 겹칠 뿐
  라벨은 정상입니다(kwon 셀 83). 원본 상태 관측값입니다.
- 모양 의심은 kwon이 라벨 누락을 채운 771개 기준으로 30개였고, 라벨 오류 3개(3351 빈 배경, 18357 이미지 밖,
  16548 다른 약 자리)를 빼면 부서지거나 붙은 알약은 0개, 나머지 27개는 투명·어두운·흰 알약의 오탐이었습니다
  (kwon 셀 65–66).
- 이 노트북은 원본 763개에서 이미지 밖 박스(18357)를 빼고 재서 의심이 29개입니다. 라벨 오류 2개(3351 빈 배경은
  `알약 없음`, 16548 다른 약 자리는 `큼`)를 빼면 27개이고, 모두 약 6종(16688, 2483, 27733, 31885, 27926, 35206)에
  몰려 있어 kwon이 오탐으로 본 유형과 같습니다. 원본 상태 관측값입니다.
- 맞닿은 알약 처리(kwon 셀 85–86에서 정함): 데이터는 그대로 둡니다. 대신 추론 NMS가 클래스별로 적용되는지,
  IoU 기준이 너무 낮지 않은지 확인하고, 같은 장면 2장은 그룹 분할로 한쪽에만 둡니다. 학습 후 붙은 알약을 놓치면
  그때 Copy-Paste 증강을 추가합니다. 9장 "학습 파이프라인으로 넘길 것"에 옮깁니다.

### 5.7 bbox 여백 (GrabCut 추정) `[RUN_HEAVY]`

> 출처: JJAM99_eda.ipynb 셀 53–60 (potato). 함수는 `data_process/data_processing.py`의
> `measure_bbox_padding`입니다.

알약마다 GrabCut으로 몸체를 추정하고 `padding_ratio = 1 − 몸체 넓이 ÷ bbox 넓이`를 잽니다.

- 그림자 후보를 배경으로 지정하고 초기 조건 두 가지로 GrabCut을 돌려, 두 윤곽을 모두 포함하는 사각형을 몸체로 봅니다.
  몸체에 최소 4px 여유를 더한 것이 `tight_bbox`(새 bbox 후보)입니다.
- `status`가 `estimated`인 박스만 여백을 집계합니다. 탐색 경계에 닿거나, 두 추정이 다르거나, 윤곽이 원본 박스 밖으로
  이어지면 `review_required`·`extends_outside`로 따로 둡니다.
- 원형이나 기울어진 알약은 축에 평행한 박스 모서리에 빈 공간이 생기는 게 정상이라, 여백 비율만으로 박스가 틀렸다고 보지 않습니다.
- 전체 박스에 1–2분 걸려 `RUN_HEAVY=True`일 때만 계산합니다. 아니면 입력·방식·seed가 같은 캐시만 씁니다.

이식하지 않은 셀: 셀 61–62(이전 색상 전용 방식과 비교), 63–66(보정 전후 개수와 그림), 67–74(보정본 저장, 미분류
재보정). 저장 셀은 계획 6장 원칙에 따라 옮기지 않았고, 여백 20% 이상 후보는 5.8에서 미검수 보정 기록으로 다룹니다.

In [ ]:
# [RUN_HEAVY] GrabCut 여백 추정. 입력 지문·방식·seed가 저장 당시와 같은 캐시만 재사용한다.
padding_cache = CACHE_DIR / "bbox_padding.json"
padding_key = {
    "version": PADDING_VERSION,
    "seed": SEED,
    "data_state": "원본",
    "inputs": fingerprint_files([*img_paths["train"], *TRAIN_ANN_DIR.rglob("*.json")], RAW_DIR),
}
if RUN_HEAVY:
    start = time.perf_counter()
    padding_df = measure_bbox_padding(ann_df, TRAIN_IMG_DIR, seed=SEED)
    padding_status = f"실행 ({time.perf_counter() - start:.0f}초)"
    if SAVE_OUTPUTS:
        save_cache(padding_cache, padding_key, padding_df)
else:
    padding_df, padding_status = load_cache(padding_cache, padding_key)
    if padding_df is None:
        padding_status = f"{NOT_RUN} ({padding_status})"

# padding_df가 None이면 미실행이다. 뒤 셀은 None을 확인하고, 여백 후보 0개로 바꿔 세지 않는다.
print(f"[원본] GrabCut 여백 추정: {padding_status}")
if padding_df is not None:
    estimated = padding_df[padding_df["status"] == "estimated"]
    display(padding_df["status"].value_counts().rename("bbox 수").to_frame().T)
    threshold_df = pd.DataFrame(
        [
            {
                "추정 여백 기준": f"{t:.0%} 이상",
                "bbox 수": (estimated["padding_ratio"] >= t).sum(),
                "사진 수": estimated.loc[estimated["padding_ratio"] >= t, "file_name"].nunique(),
                "클래스 수": estimated.loc[
                    estimated["padding_ratio"] >= t, "category_id"
                ].nunique(),
            }
            for t in (0.1, 0.2, 0.3)
        ]
    ).set_index("추정 여백 기준")
    print(f"[원본] 여백 기준별 bbox (estimated {len(estimated)}개 중, 누적이므로 행을 더하지 않음)")
    display(threshold_df)
    padding_review_df = padding_df[padding_df["status"] != "estimated"]
    print(f"[원본] 경계 확인 필요: {len(padding_review_df)}개")
    display(
        padding_review_df[
            ["file_name", "annotation_id", "category_id", "bbox", "status", "review_reason"]
        ]
    )

In [ ]:
# 여백 10~20%, 20~30%, 30% 이상 구간에서 하한에 가장 가까운 박스를 하나씩 본다(JJAM 셀 56).
# 빨강: 원본 bbox, 초록: 경계 여유를 포함한 추정 bbox.
# 아래 표는 원본·추정 bbox IoU 분포다(JJAM 셀 60).
if padding_df is None:
    print(f"[원본] 여백 예시와 IoU 분포: {padding_status}")
else:
    fig, axes = plt.subplots(1, 3, figsize=(10, 4.2))
    for ax, (low, high) in zip(axes, [(0.1, 0.2), (0.2, 0.3), (0.3, 1.01)], strict=True):
        band = estimated[estimated["padding_ratio"].between(low, high, inclusive="left")]
        if band.empty:
            ax.set_title(f"{low:.0%} 이상: 예시 없음")
            ax.axis("off")
            continue
        ex = band.sort_values(["padding_ratio", "file_name", "annotation_id"]).iloc[0]
        title = (
            f"[원본] 여백 {ex['padding_ratio']:.1%}\n{ex['category_id']}, IoU {ex['bbox_iou']:.3f}"
        )
        show_image(ax, TRAIN_IMG_DIR / ex["file_name"], title)
        draw_bbox(ax, ex["bbox"], color="red")
        draw_bbox(ax, ex["tight_bbox"], color="lime")
        zoom_to(ax, [ex["bbox"], ex["tight_bbox"]], margin=0.15, limit=(976, 1280))
    plt.tight_layout()
    plt.show()

    edges = [i / 10 for i in range(11)]
    labels = [f"{lo:.1f}–{hi:.1f}" for lo, hi in zip(edges[:-1], edges[1:], strict=True)]
    bins = edges[:-1] + [1.000001]
    per_box = pd.cut(estimated["bbox_iou"], bins=bins, labels=labels, right=False)
    per_image = pd.cut(
        estimated.groupby("file_name")["bbox_iou"].min(), bins=bins, labels=labels, right=False
    )
    iou_table = pd.DataFrame(
        {
            "bbox 수": per_box.value_counts(sort=False),
            "사진 수 (사진별 최저 IoU)": per_image.value_counts(sort=False),
        }
    ).rename_axis("원본↔추정 IoU")
    display(iou_table[iou_table.sum(axis=1) > 0])

- 여백 10% 이상 747개, 20% 이상 173개(사진 120장), 30% 이상 7개로 JJAM 셀 54와 같습니다. 원본·추정 IoU 분포도
  JJAM 셀 60과 거의 같습니다(원본 상태 관측값).
- 경계 확인은 9개로 JJAM보다 1개 많습니다. JJAM은 보정본(`bbox_area`, `bbox_range`)이 있으면 그것을 입력으로 써서
  18357이 고쳐진 좌표로 들어갔고, 이 노트북은 원본을 입력으로 써서 18357이 `invalid_bbox`(이미지 밖)로 남습니다.
- 대부분의 박스가 10% 이상 여백을 갖습니다. kwon이 본 것처럼 기존 박스는 알약 윤곽보다 사방 약 8px 넉넉하게
  그려져 있습니다(kwon 셀 16). 여백이 일관되면 모델도 일관되게 배우므로 그 자체로 오류는 아닙니다.
- 20% 이상만 줄이면 다른 박스와 여백 기준이 달라집니다(kwon 셀 85의 같은 지적). 정답 박스의 여백 기준을 바꿀지는
  9장 "결정할 것"으로 넘깁니다.

### 5.8 라벨 오류 요약과 보정 기록 통합

> 출처: eda_kwon.ipynb 셀 14–23, 67–83 (gwon753-dev), JJAM99_eda.ipynb 셀 25–26, 47–48, 63–68 (potato),
> EDA_jang.ipynb 셀 10–11 (younghoon1215). 보정 기록 함수는 `data_process/make_labels.py`에 있습니다.

먼저 4–5장에서 찾은 문제를 한 표로 모읍니다. 건수는 이번 실행의 `[원본]` 값입니다.

In [ ]:
n_padding = NOT_RUN if padding_df is None else int((estimated["padding_ratio"] >= 0.2).sum())
error_summary = pd.DataFrame(
    [
        ("4.1", "bbox 값 결측", len(bbox_missing), "오류", "-"),
        ("4.2", "라벨 누락", missing_df["missing_codes"].str.len().sum(), "오류", "추가 (kwon)"),
        (
            "5.1",
            "이미지 밖 좌표",
            flag_df["out_of_image"].sum(),
            "오류",
            "위치 수정 (kwon, JJAM 값이 다름)",
        ),
        ("5.1", "크기 0 이하", flag_df["non_positive"].sum(), "오류", "-"),
        ("5.2", "다른 약과 좌표 중복 (쌍)", len(dup_pair_df), "오류", "틀린 쪽 위치 수정 (kwon)"),
        ("5.3", "빈 배경 박스", len(empty_df), "오류", "위치 수정 (kwon)"),
        ("5.4", "area ≠ w×h", len(area_df), "확인 대상", "19232만 bbox 확대 제안 (JJAM)"),
        ("5.5", "면적 비율 IQR 후보", outlier_flags["area_ratio"].sum(), "후보", "보정 안 함"),
        ("5.5", "가로세로 비율 IQR 후보", outlier_flags["aspect"].sum(), "후보", "보정 안 함"),
        ("5.6", "맞닿은 알약 (쌍)", len(touch_df), "정상 라벨", "보정 안 함 (NMS·그룹 분할)"),
        ("5.6", "모양 이상 의심", len(shape_suspect_df), "확인 대상", "보정 안 함"),
        ("5.7", "여백 20% 이상", n_padding, "추정", "여백 보정 제안 (JJAM)"),
    ],
    columns=["절", "문제", "[원본] 건수", "판정", "보정 기록"],
).set_index("절")
display(error_summary)

**보정 기록 통합 규칙** (계획 6장)

- 같은 문제를 여러 노트북이 다른 방식으로 고쳤으므로, 모든 보정을 한 형식의 표(`correction_df`, 열은 `CORRECTION_COLS`)로 모읍니다.
  한 행이 보정 하나입니다. 주요 열: `stage`(추가 → 위치 수정 → 여백 보정 순으로 적용), `source_json`·`annotation_id`(대상 라벨),
  `old_bbox`(보정할 때 본 좌표), `new_bbox`, `method`, `author`, `reviewed`, `source_record`(원래 기록),
  `input_sha256`(원본 JSON 해시).
- kwon CSV는 `(file_name, category_id)`로 라벨을 가리키므로 원본 라벨로 역매핑합니다. 후보가 하나가 아니면 적용하지 않습니다.
- 새 라벨 ID는 원본 ID와 겹치지 않게 1,000,000부터 `correction_id` 순서로 붙입니다. 같은 기록이면 실행마다 같은 ID입니다.
- 적용 조건: 검수됨(`reviewed`), 같은 단계에 같은 대상을 가리키는 기록이 하나뿐, 현재 bbox = `old_bbox`, segmentation 없음,
  새 bbox가 이미지 안이고 크기가 양수. 하나라도 어기면 적용하지 않고 이유를 남깁니다(`미검수`, `거부`).
  적용한 박스는 `area = w×h`로 맞춥니다. 이미 적용된 기록은 건너뛰므로 다시 적용해도 결과가 같습니다.
- 적용은 메모리의 `ann_corrected_df`에만 합니다. `ann_df`와 `data/raw/`는 그대로이고, 이 노트북은 기록 파일을 쓰지 않습니다.

| 이번 실행의 기록 출처 | 내용 | `reviewed` |
|---|---|---|
| kwon `data/processed/added_annotations.csv`, `fixed_annotations.csv` | 라벨 누락 8개 추가, 위치 수정 5개 | `KWON_REVIEWED` (기본 True: kwon이 전후 그림으로 확인, kwon 셀 79–80) |
| JJAM 색상 윤곽 제안 (5.1에서 다시 계산) | 18357 위치 | False |
| JJAM 잘린 캡슐 제안 (5.4에서 다시 계산) | 19232 bbox 확대 | False |
| JJAM GrabCut (5.7) | 여백 20% 이상 박스의 `tight_bbox` | False (자동 추정이며 수동 검수가 아님, JJAM 셀 67) |

JJAM 저장본(`data/processed/bbox_area/`, `bbox_grabcut/`)은 팀에 공유되지 않아 같은 방법으로 다시 계산한 값을 씁니다.

In [ ]:
KWON_ADDED_PATH = PROCESSED_DIR / "added_annotations.csv"
KWON_FIXED_PATH = PROCESSED_DIR / "fixed_annotations.csv"
KWON_REVIEWED = True  # kwon이 전후 그림으로 확인함(kwon 셀 79–80). 팀 검수 기준이 정해지면 바꾼다.
PADDING_MIN = 0.2  # 여백이 이 이상인 박스만 여백 보정 후보로 둔다(JJAM 셀 64)

kwon_records = corrections_from_kwon(
    KWON_ADDED_PATH, KWON_FIXED_PATH, ann_df, TRAIN_ANN_DIR, reviewed=KWON_REVIEWED
)
jjam_rows = []
for (source_json, annotation_id), bbox in shape_proposals.items():
    if bbox is not None:
        jjam_rows.append(
            correction_for_label(
                ann_df,
                source_json,
                annotation_id,
                bbox,
                TRAIN_ANN_DIR,
                correction_id=f"jjam_shape:{source_json}#{annotation_id}",
                stage="위치 수정",
                case="이미지 밖 좌표",
                author="potato",
                reviewed=False,
                method="HSV 색상 윤곽 중 y·크기가 비슷한 것 + 사방 8px",
                source_record="JJAM99_eda.ipynb 셀 26 (5.1에서 다시 계산)",
            )
        )
if capsule_proposal is not None:
    jjam_rows.append(
        correction_for_label(
            ann_df,
            *CUT_CAPSULE,
            capsule_proposal,
            TRAIN_ANN_DIR,
            correction_id=f"jjam_capsule:{CUT_CAPSULE[0]}#{CUT_CAPSULE[1]}",
            stage="위치 수정",
            case="잘린 bbox",
            author="potato",
            reviewed=False,
            method="박스 중앙의 HSV 색상 덩어리 + 사방 8px",
            source_record="JJAM99_eda.ipynb 셀 48 (5.4에서 다시 계산)",
        )
    )
padding_rows = []
if padding_df is not None:
    candidates = estimated[estimated["padding_ratio"] >= PADDING_MIN]
    for r in candidates.itertuples():
        padding_rows.append(
            correction_for_label(
                ann_df,
                r.source_json,
                r.annotation_id,
                r.tight_bbox,
                TRAIN_ANN_DIR,
                correction_id=f"jjam_grabcut:{r.source_json}#{r.annotation_id}",
                stage="여백 보정",
                case="bbox 여백",
                author="potato",
                reviewed=False,
                method=f"GrabCut 몸체 + 경계 여유 ({PADDING_VERSION})",
                source_record="JJAM99_eda.ipynb 셀 54, 64 (5.7에서 다시 계산)",
            )
        )

parts = [kwon_records, pd.DataFrame(jjam_rows + padding_rows, columns=CORRECTION_COLS)]
parts = [p for p in parts if not p.empty]
correction_df = (
    pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=CORRECTION_COLS)
)
correction_df = assign_new_ids(correction_df, ann_df)

kwon_sources = kwon_records["source_record"].str.split("#").str[0]
source_df = pd.DataFrame(
    [
        *[
            (
                p.relative_to(PROJECT_ROOT).as_posix(),
                "읽음" if p.is_file() else "파일 없음",
                (kwon_sources == p.name).sum(),
            )
            for p in (KWON_ADDED_PATH, KWON_FIXED_PATH)
        ],
        ("JJAM 색상 제안 (5.1, 5.4)", "다시 계산", len(jjam_rows)),
        (
            "JJAM GrabCut (5.7)",
            padding_status,
            NOT_RUN if padding_df is None else len(padding_rows),
        ),
    ],
    columns=["출처", "상태", "기록 수"],
)
print(f"보정 기록: {len(correction_df)}개")
display(source_df)
display(
    correction_df[correction_df["stage"] != "여백 보정"].drop(
        columns=["input_sha256", "source_json"]
    )
)

In [ ]:
# 같은 라벨에 값이 다른 기록이 둘 이상이면 나란히 그린다(계획 6장: 두 값을 비교해 하나를 고른다).
fixes = correction_df[correction_df["stage"] != "추가"]
conflict_keys = ["stage", "source_json", "annotation_id"]
conflict_df = fixes[fixes.duplicated(conflict_keys, keep=False)]
if conflict_df.empty:
    print("같은 라벨에 값이 다른 보정 기록이 없습니다.")
else:
    groups = list(conflict_df.groupby(conflict_keys))
    fig, axes = image_grid(len(groups), cols=2, cell=(6, 6.5))
    for ax, ((stage, _, annotation_id), group) in zip(axes, groups, strict=True):
        first = group.iloc[0]
        boxes = ann_df[ann_df["file_name"] == first["file_name"]]
        lines = []
        show_image(ax, TRAIN_IMG_DIR / first["file_name"])
        draw_bboxes(ax, boxes[boxes["annotation_id"] != annotation_id], color="gray")
        styles = [("tab:blue", "-", "실선"), ("tab:orange", "--", "점선"), ("magenta", ":", "점")]
        for (color, ls, name), rec in zip(styles, group.itertuples(), strict=False):
            draw_bbox(ax, rec.new_bbox, color=color, linestyle=ls)
            lines.append(f"{name}: {rec.author} {list(rec.new_bbox)} (reviewed={rec.reviewed})")
        iou = bbox_iou(group["new_bbox"].iloc[0], group["new_bbox"].iloc[1])
        title = f"[제안 비교] {first['file_name'].split('_')[0]} {first['category_id']} ({stage})\n"
        ax.set_title(title + "\n".join(lines) + f"\n두 제안의 IoU {iou:.3f}", fontsize=8)
        size = (boxes["width"].iloc[0], boxes["height"].iloc[0])
        zoom_to(ax, list(group["new_bbox"]), margin=0.8, limit=size)
    plt.tight_layout()
    plt.show()

In [ ]:
ann_corrected_df, apply_log_df = apply_corrections(ann_df, correction_df)
n_applied = int((apply_log_df["status"] == "적용").sum())
print(
    f"[보정] 기록 {len(correction_df)}개 중 적용 {n_applied}개, "
    f"bbox {len(ann_df)}개 → {len(ann_corrected_df)}개"
)
status_table = pd.crosstab(apply_log_df["stage"], apply_log_df["status"])
display(status_table.reindex([s for s in STAGES if s in status_table.index]))
not_applied = apply_log_df[apply_log_df["status"] != "적용"]
print("[보정] 적용하지 않은 기록 (이유별)")
display(not_applied.groupby(["stage", "status", "reasons"]).size().rename("기록 수").reset_index())

# 같은 기록을 보정본에 다시 적용해도 새로 적용되는 것이 없어야 한다(중복 추가 방지).
again_df, again_log = apply_corrections(ann_corrected_df, correction_df)
print(
    f"[보정] 다시 적용: 새로 적용 {(again_log['status'] == '적용').sum()}개, "
    f"이미 적용 {(again_log['status'] == '이미 적용').sum()}개, "
    f"bbox 수 변화 {len(again_df) - len(ann_corrected_df)}개"
)

In [ ]:
# 원본과 보정본을 같은 검사로 비교한다. 바뀐 박스만 픽셀을 다시 잰다.
correction_status = "완료" if n_applied else f"{NOT_RUN} (적용된 보정 없음)"
print(f"[보정] 원본·보정 비교: {correction_status}")
if n_applied:
    changed = ann_corrected_df["correction_ids"] != ""
    flags_after = bbox_flags(ann_corrected_df, area_tol=AREA_TOL)
    pairs_after = find_box_pairs(ann_corrected_df, max_gap=TOUCH_GAP)
    diff_after = pixel_df["diff_p90"].reindex(ann_corrected_df.index)
    diff_after[changed] = measure_box_pixels(ann_corrected_df[changed], TRAIN_IMG_DIR)["diff_p90"]
    check_after = compare_name_codes(img_df, ann_corrected_df)
    compare_df = pd.DataFrame(
        [
            ("bbox 수", len(ann_df), len(ann_corrected_df)),
            (
                "라벨 누락 (bbox)",
                label_check["missing_codes"].str.len().sum(),
                check_after["missing_codes"].str.len().sum(),
            ),
            (
                "파일명에 없는 약의 bbox",
                label_check["extra_codes"].str.len().sum(),
                check_after["extra_codes"].str.len().sum(),
            ),
            ("이미지 밖 좌표", flag_df["out_of_image"].sum(), flags_after["out_of_image"].sum()),
            ("크기 0 이하", flag_df["non_positive"].sum(), flags_after["non_positive"].sum()),
            ("area ≠ w×h", flag_df["area_mismatch"].sum(), flags_after["area_mismatch"].sum()),
            ("좌표 중복 쌍", len(dup_pair_df), (pairs_after["iou"] >= DUP_IOU).sum()),
            ("맞닿은 쌍", len(touch_df), (pairs_after["iou"] < DUP_IOU).sum()),
            ("빈 배경 박스", len(empty_df), (diff_after < EMPTY_DIFF_MAX).sum()),
        ],
        columns=["검사", "[원본]", "[보정]"],
    ).set_index("검사")
    display(compare_df)

    counts = (
        pd.DataFrame(
            {
                "[원본]": ann_df["category_id"].value_counts(),
                "[보정]": ann_corrected_df["category_id"].value_counts(),
            }
        )
        .fillna(0)
        .astype(int)
    )
    counts = counts[counts["[원본]"] != counts["[보정]"]].join(
        cat_df.set_index("category_id")["dl_name"]
    )
    print("[보정] bbox 수가 바뀐 클래스")
    display(counts)

    remaining = ann_corrected_df[flags_after.any(axis=1)].join(flags_after)
    print("[보정] 남은 좌표·area 오류 (correction_ids가 비면 적용된 보정이 없는 라벨)")
    display(
        remaining[
            [
                "file_name",
                "annotation_id",
                "category_id",
                *BBOX_COLS,
                "area",
                "correction_ids",
                *flags_after.columns,
            ]
        ]
    )

In [ ]:
# 적용된 보정의 전후. 빨간 점선: 고치기 전, 초록: 고친 뒤(또는 추가한) 박스, 회색: 나머지 라벨
applied = apply_log_df[apply_log_df["status"] == "적용"]
records = correction_df.set_index("correction_id")
if applied.empty:
    print(f"[보정] 전후 그림: {correction_status}")
else:
    fig, axes = image_grid(len(applied), cols=4)
    for ax, rec in zip(axes, applied.itertuples(), strict=True):
        record = records.loc[rec.correction_id]
        boxes = ann_corrected_df[ann_corrected_df["file_name"] == rec.file_name]
        is_target = boxes["correction_ids"].str.contains(rec.correction_id, regex=False)
        angle = boxes["camera_la"].iloc[0]
        title = f"[보정] {rec.file_name.split('_')[0]} ({angle}°)\n{rec.case}: {rec.category_id}"
        show_image(ax, TRAIN_IMG_DIR / rec.file_name, title)
        draw_bboxes(ax, boxes[~is_target], color="gray")
        if isinstance(record["old_bbox"], tuple):
            draw_bbox(ax, record["old_bbox"], color="red", linestyle="--")
        draw_bbox(ax, record["new_bbox"], str(rec.category_id), color="lime")
    plt.tight_layout()
    plt.show()

아래는 kwon CSV가 있는 환경에서 관측한 값입니다(원본 상태 관측값과 보정 기록 기준).

- kwon 기록 13개 중 12개(추가 8개, 위치 수정 4개)가 적용되어 bbox는 763개 → 771개입니다. 라벨 누락 8개 → 0개,
  좌표 중복 3쌍 → 0쌍, 빈 배경 1개 → 0개입니다. 위치 수정은 bbox 수를 바꾸지 않으므로 늘어난 8개는 모두 라벨 누락 추가입니다.
- 이미지 밖 좌표 1개(18357)는 남습니다. kwon과 JJAM 방식이 같은 라벨에 다른 값(IoU 약 0.98)을 내서 충돌로 보류했기
  때문입니다. 값을 하나 고르고 다른 기록을 빼면 적용됩니다.
- area 불일치 6개도 남습니다. 19232 제안은 미검수이고, 나머지 5개는 보정 기록이 없습니다.
- 여백 보정 기록은 모두 미검수라 적용하지 않았습니다. 위치가 고쳐진 박스의 여백 기록은 현재 bbox와 `old_bbox`가 달라
  거부 이유가 함께 남습니다. 고친 좌표로 다시 계산해야 합니다(계획 6장).

**9장으로 넘길 결정**

1. 18357 bbox 값: kwon `[574, 626, 300, 309]`과 JJAM 색상 윤곽 제안 중 하나
2. 검수 기준: kwon 기록을 검수됨으로 볼지(`KWON_REVIEWED`), JJAM 19232 제안을 적용할지
3. 정답 박스의 여백 기준: 여백 20% 이상 박스만 줄일지, 그대로 둘지
4. 보정 기록 파일: `correction_df` 형식으로 `data/processed/`에 저장하는 작업을 `data_process/`에 만들지
   (저장하면 새 라벨 ID도 기록에 고정됩니다)

## 6. 분포

> 출처: eda_kwon.ipynb 셀 10, 34–37 (gwon753-dev), EDA_jang.ipynb 셀 13–17 (younghoon1215),
> sohee_eda.ipynb 셀 13 (bravedolphinhere)

- 클래스 하나는 `category_id` 하나, 곧 약 한 종류입니다.
- 분포는 `[원본]` 기준으로 봅니다. 보정 기록을 적용한 `[보정]`은 클래스별 bbox 수만 나란히 둡니다.
  위치 수정은 bbox 수를 바꾸지 않으므로 `[보정]`의 차이는 라벨 누락 추가(4.2, 5.8)에서만 생깁니다.

### 6.1 클래스별 bbox 수와 불균형 단계

> 기준본: eda_kwon.ipynb 셀 35, 37. EDA_jang.ipynb 셀 15의 요약 통계와 sohee_eda.ipynb 셀 13의
> 클래스별 사진 수·조합 수를 합쳤습니다.

- 막대는 bbox가 많은 클래스부터 위에서 아래로 그립니다. 파랑은 원본 라벨, 주황은 보정 기록으로 추가한 라벨입니다.
- 불균형 단계는 클래스를 bbox 수에 따라 5단계로 나누고, 각 단계가 클래스 수와 bbox 수에서 차지하는 비중을 비교합니다.
  같은 색의 위아래 막대 길이가 다를수록 불균형이 큽니다.

In [ ]:
# 클래스별 bbox 수·사진 수·조합 수. 조합은 파일 이름 첫 밑줄 앞의 알약 조합이다(3.2).
class_df = (
    ann_df.assign(combo=ann_df["file_name"].str.split("_").str[0])
    .groupby("category_id")
    .agg(
        n_boxes=("annotation_id", "size"),
        n_images=("file_name", "nunique"),
        n_combos=("combo", "nunique"),
    )
    .join(cat_df.set_index("category_id")["dl_name"].str.strip())
)
class_df["ratio(%)"] = class_df["n_boxes"] / class_df["n_boxes"].sum() * 100
# [보정]은 보정 기록이 하나라도 적용됐을 때만 센다(5.8). 아니면 미실행으로 둔다.
if n_applied:
    class_df["[보정] n_boxes"] = (
        ann_corrected_df["category_id"].value_counts().reindex(class_df.index, fill_value=0)
    )
class_df = class_df.sort_values("n_boxes", ascending=False)

counts = class_df["n_boxes"]
class_summary = pd.Series(
    {
        "클래스 수": len(class_df),
        "bbox 수": counts.sum(),
        "클래스당 평균": counts.mean(),
        "클래스당 중앙값": counts.median(),
        "최소": counts.min(),
        "최대": counts.max(),
        "최대 / 최소": counts.max() / counts.min(),
    },
    name="[원본]",
)
print(f"[보정] 클래스별 bbox 수: {correction_status}")
display(class_summary.to_frame().round(2))
with pd.option_context("display.max_rows", None):
    display(class_df.round(2))

In [ ]:
ORIGINAL_COLOR = "#2a78d6"  # 파랑: 원본 라벨
ADDED_COLOR = "#eb6834"  # 주황: 보정 기록으로 추가한 라벨
TEXT_COLOR = "#52514e"

# 가로 막대는 아래부터 그려지므로 오름차순으로 두면 많은 클래스가 위에 온다.
bar_df = class_df.sort_values("n_boxes")
added = pd.Series(0, index=bar_df.index)
if n_applied:
    added_labels = ann_corrected_df[ann_corrected_df["label_origin"] == "추가"]
    added = added_labels["category_id"].value_counts().reindex(bar_df.index, fill_value=0)
total = bar_df["n_boxes"] + added
y_labels = [f"{cid}  {name}" for cid, name in zip(bar_df.index, bar_df["dl_name"], strict=True)]

fig, ax = plt.subplots(figsize=(11, 15))
ax.barh(y_labels, bar_df["n_boxes"], color=ORIGINAL_COLOR, height=0.7, label="원본 라벨")
if added.sum():
    ax.barh(
        y_labels,
        added,
        left=bar_df["n_boxes"],
        color=ADDED_COLOR,
        height=0.7,
        edgecolor="white",
        linewidth=1,
        label="[보정] 추가한 라벨",
    )
for i, (n, ratio) in enumerate(zip(total, total / total.sum() * 100, strict=True)):
    ax.text(n + 1.5, i, f"{n} ({ratio:.1f}%)", va="center", fontsize=8, color=TEXT_COLOR)
mean = bar_df["n_boxes"].mean()
ax.axvline(mean, color=TEXT_COLOR, linestyle="--", linewidth=1)
ax.text(mean + 1, len(bar_df) - 0.5, f"[원본] 평균 {mean:.1f}개", fontsize=9, color=TEXT_COLOR)
state = "[원본]+[보정] 추가" if added.sum() else "[원본]"
ax.set_title(f"{state} 클래스별 bbox 수 (전체 {total.sum()}개, {len(bar_df)}종)", loc="left")
ax.set_xlabel("bbox 수")
ax.set_xlim(0, total.max() * 1.15)
ax.margins(y=0.01)
ax.tick_params(axis="y", labelsize=8)
ax.grid(axis="x", color="#e5e5e5", linewidth=0.8)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
ax.legend(loc="lower right", frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# 클래스를 bbox 수에 따라 5단계로 나누고 각 단계가 클래스 수와 bbox 수에서 차지하는 비중을 본다.
TIERS = [(1, 5, "1~5개"), (6, 10, "6~10개"), (11, 20, "11~20개"), (21, 50, "21~50개")]
TIERS.append((51, 10_000, "51개 이상"))
TIER_COLORS = ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#0d366b"]  # bbox가 많을수록 진하게


def tier_table(totals: pd.Series) -> pd.DataFrame:
    """단계마다 클래스 수·bbox 수와 전체 대비 비중(%)을 센다."""
    rows = []
    for low, high, name in TIERS:
        in_tier = totals[totals.between(low, high)]
        rows.append(
            {
                "단계": name,
                "클래스 수": len(in_tier),
                "bbox 수": in_tier.sum(),
                "클래스 비중(%)": len(in_tier) / len(totals) * 100,
                "bbox 비중(%)": in_tier.sum() / totals.sum() * 100,
            }
        )
    return pd.DataFrame(rows).set_index("단계")


tier_df = tier_table(class_df["n_boxes"])
print("[원본] 불균형 단계")
display(tier_df.round(1))
if n_applied:
    print("[보정] 불균형 단계")
    display(tier_table(class_df["[보정] n_boxes"]).round(1))

rows = [("클래스", "클래스 비중(%)", "클래스 수", "종"), ("bbox", "bbox 비중(%)", "bbox 수", "개")]
fig, ax = plt.subplots(figsize=(12, 4))
for y, (_, share_col, count_col, unit) in enumerate(rows):
    left = 0
    for (_, t), color in zip(tier_df.iterrows(), TIER_COLORS, strict=True):
        share, count = t[share_col], int(t[count_col])
        ax.barh(y, share, left=left, color=color, height=0.6, edgecolor="white", linewidth=2)
        label = f"{count}{unit}\n{share:.0f}%"
        if share >= 6:
            text_color = "white" if color in TIER_COLORS[2:] else "#0b0b0b"
            ax.text(left + share / 2, y, label, ha="center", va="center", color=text_color)
        else:  # 좁은 칸은 글자가 안 들어가므로 막대 위에 적는다.
            ax.text(
                left + share / 2,
                y + 0.42,
                label.replace("\n", " "),
                ha="center",
                va="bottom",
                fontsize=9,
                color=TEXT_COLOR,
            )
        left += share
ax.set_yticks(range(len(rows)), [r[0] for r in rows], fontsize=11)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("비중 (%)")
for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
ax.tick_params(axis="y", length=0)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in TIER_COLORS]
ax.legend(
    handles,
    [f"클래스당 bbox {name}" for _, _, name in TIERS],
    ncol=5,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.28),
    frameon=False,
    fontsize=9,
)
many, few = tier_df.iloc[-2:], tier_df.iloc[:2]  # bbox 21개 이상, 10개 이하 단계
ax.set_title(
    f"[원본] 상위 {many['클래스 수'].sum()}종({many['클래스 비중(%)'].sum():.0f}%)이 bbox의 "
    f"{many['bbox 비중(%)'].sum():.0f}%를 차지하고, "
    f"{few['클래스 수'].sum()}종({few['클래스 비중(%)'].sum():.0f}%)은 bbox가 10개 이하입니다",
    loc="left",
    fontsize=12,
)
plt.tight_layout()
plt.show()

아래는 원본 상태 관측값입니다.

- 56종, bbox 763개입니다. `3351`(일양하이트린정 2mg) 한 종이 153개(20.1%)로 전체의 5분의 1이고, 다음은
  `3483` 45개(5.9%)입니다. 클래스당 중앙값 9개, 최소 3개이며 최대/최소는 51배입니다(jang 셀 15와 같음).
- 불균형 단계: bbox 5개 이하 18종(32%)이 bbox의 7%만 갖고, 21개 이상 6종(11%)이 bbox의 39%를 갖습니다.
  10개 이하로 넓히면 29종(52%)이 bbox 18%를 나눠 갖습니다(kwon 셀 37).
- `3351`은 조합 114개 중 89개에 들어 있습니다. 반대로 조합 하나에만 나오는 클래스가 17종입니다.
  이런 약은 사진이 1–3장, 곧 장면 하나뿐이라 분할 방식에 따라 학습이나 검증 한쪽에서 통째로 빠집니다(8.1).
- `[보정]`은 라벨 누락 8개가 추가되어 `3351` 153 → 157개 등이 바뀝니다. `4543`이 5 → 6개가 되어
  5개 이하 단계가 18종 → 17종으로 줄어듭니다. 불균형의 모양은 그대로입니다.
- 불균형 대응 방안(kwon 셀 39: 그룹·층화 분할, 증강, 손실 가중치, 2단계 검출 등)은 9장의
  "학습 파이프라인으로 넘길 것"에서 정리합니다.

### 6.2 bbox 크기 분포, 클래스별 크기

> 기준본: EDA_jang.ipynb 셀 13, 17 (younghoon1215)

- 크기는 사진 넓이 대비 bbox 넓이 비율(%)로 봅니다. 모든 사진이 976×1280이라 픽셀 넓이와 순서가 같습니다.
- jang 기준으로 3% 미만을 Small, 3% 이상 8% 미만을 Medium, 8% 이상을 Large로 나눕니다. COCO의 small/medium/large
  (32², 96² px 기준)와 다른 기준입니다. COCO 기준으로는 가장 작은 bbox도 96² px보다 커서 모두 large입니다.
- 크기가 0 이하이거나 좌표가 없는 bbox만 뺍니다. 이미지 밖 좌표(5.1)는 위치만 틀리고 크기는 있으므로
  포함합니다(jang과 같은 기준). jang 셀 17은 보정본을 읽으려 했지만 경로가 달라 원본을 읽었습니다(계획 3장 4번).

In [ ]:
SIZE_BINS = [0, 3, 8, float("inf")]  # 사진 대비 bbox 넓이(%) 경계 (jang 셀 13)
SIZE_LABELS = ["Small", "Medium", "Large"]

size_valid = ann_df[(ann_df["w"] > 0) & (ann_df["h"] > 0)]
bbox_size_df = pd.DataFrame(
    {
        "w": size_valid["w"],
        "h": size_valid["h"],
        "area_px": size_valid["w"] * size_valid["h"],
        "area_ratio(%)": size_valid["w"]
        * size_valid["h"]
        / (size_valid["width"] * size_valid["height"])
        * 100,
    }
)
bbox_size_df["size"] = pd.cut(
    bbox_size_df["area_ratio(%)"], bins=SIZE_BINS, labels=SIZE_LABELS, right=False
)
print(f"[원본] 크기를 잰 bbox: {len(bbox_size_df)}개 (뺀 bbox: {len(ann_df) - len(size_valid)}개)")
display(bbox_size_df.drop(columns="size").describe().round(2))
size_counts = bbox_size_df["size"].value_counts().reindex(SIZE_LABELS)
display(
    pd.DataFrame({"bbox 수": size_counts, "비율(%)": size_counts / len(bbox_size_df) * 100}).round(
        1
    )
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(bbox_size_df["area_ratio(%)"], bins=40, color="steelblue")
for edge, color in zip(SIZE_BINS[1:3], ["orange", "red"], strict=True):
    axes[0].axvline(edge, color=color, linestyle="--", label=f"{edge}%")
axes[0].set(
    title="[원본] 사진 대비 bbox 넓이 (점선: 크기 구간 경계)",
    xlabel="bbox 넓이 / 사진 넓이 (%)",
    ylabel="bbox 수",
)
axes[0].legend()
axes[1].bar(SIZE_LABELS, size_counts, color=["#86b6ef", "#3987e5", "#0d366b"])
for i, count in enumerate(size_counts):
    axes[1].text(i, count, f"{count} ({count / len(bbox_size_df):.0%})", ha="center", va="bottom")
axes[1].set(title="[원본] 크기 구간별 bbox 수", ylabel="bbox 수")
for ax in axes:
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 클래스별 크기: 구간별 bbox 수와 넓이 비율 분포. 넓이 비율 중앙값이 큰 클래스가 위에 온다.
class_size = bbox_size_df.join(ann_df["category_id"])
class_size_df = (
    pd.crosstab(class_size["category_id"], class_size["size"])
    .reindex(columns=SIZE_LABELS, fill_value=0)
    .assign(
        median_area_ratio=class_size.groupby("category_id")["area_ratio(%)"].median(),
        n_size_bins=lambda d: (d[SIZE_LABELS] > 0).sum(axis=1),
    )
    .join(class_df["dl_name"])
    .sort_values("median_area_ratio")
)
print(
    f"[원본] 한 크기 구간에만 있는 클래스: {(class_size_df['n_size_bins'] == 1).sum()}종 / "
    f"두 구간 이상에 걸친 클래스: {(class_size_df['n_size_bins'] > 1).sum()}종"
)
display(class_size_df[class_size_df["n_size_bins"] > 1].round(2))
small_share = class_size.loc[class_size["size"] == "Small", "category_id"].value_counts(
    normalize=True
)
print(f"[원본] Small bbox 중 가장 많은 클래스: {small_share.index[0]} ({small_share.iloc[0]:.0%})")

fig, ax = plt.subplots(figsize=(10, 15))
order = class_size_df.index
ax.boxplot(
    [class_size.loc[class_size["category_id"] == cid, "area_ratio(%)"] for cid in order],
    orientation="horizontal",
    widths=0.6,
    flierprops={"markersize": 3},
)
ax.set_yticks(
    range(1, len(order) + 1),
    [f"{cid}  {name}" for cid, name in zip(order, class_size_df["dl_name"], strict=True)],
    fontsize=8,
)
for edge, color in zip(SIZE_BINS[1:3], ["orange", "red"], strict=True):
    ax.axvline(edge, color=color, linestyle="--", linewidth=1)
ax.set(
    title="[원본] 클래스별 bbox 넓이 비율 (점선: Small/Medium, Medium/Large 경계)",
    xlabel="bbox 넓이 / 사진 넓이 (%)",
)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

아래는 원본 상태 관측값입니다.

- bbox 763개 모두 크기가 있습니다. 사진 대비 넓이는 평균 5.6%, 중앙값 4.3%, 최소 1.9%, 최대 21.8%입니다.
  Small 221개(29%), Medium 374개(49%), Large 168개(22%)로 jang 셀 13과 같습니다.
- 56종 중 44종은 bbox가 한 크기 구간에만 있습니다. 크기는 약마다 거의 정해져 있습니다.
  Small 221개 중 152개(69%)가 `3351`이라 "작은 객체"는 사실상 `3351`입니다.
- 두 구간에 걸친 12종은 대부분 경계 근처이거나(`16232` Small 14 / Medium 7), 길쭉한 알약이 기울어져
  bbox가 커진 경우(`35206` Medium 18 / Large 19)입니다. 같은 약의 bbox 크기 차이는 7.3에서 실제 크기로 다시 봅니다.
- 객체가 사진에 비해 작지 않고 크기 범위도 좁습니다(최대/최소 넓이 약 12배). 작은 객체 대응보다 클래스 불균형(6.1)이
  더 큰 문제입니다.

## 7. 촬영 조건

> 출처: eda_kwon.ipynb 셀 40–44 (gwon753-dev), EDA_jang.ipynb 셀 18–20 (younghoon1215),
> dongwook_eda.ipynb 셀 10–17 (def-uk), JJAM99_eda.ipynb 셀 49–52와 과거 셀 75–85 (potato),
> sohee_eda.ipynb 셀 15 (bravedolphinhere)

- 파일 이름은 `<알약 조합>_0_2_0_2_<카메라 각도>_000_200.png` 형식입니다. 예를 들어
  `K-003351-013900-021325_0_2_0_2_70_000_200.png`에서 `K-003351-013900-021325`가 알약 조합, `70`이 카메라
  각도입니다.
- test 이미지에는 JSON이 없으므로 촬영 조건을 모델 입력으로 쓸 수 없습니다(dongwook 셀 10).
  여기서는 분할과 증강을 정할 근거로만 봅니다.

### 7.1 조합별 각도 구성

> 기준본: eda_kwon.ipynb 셀 41, 43. EDA_jang.ipynb 셀 19의 조합 묶음 확인과 JJAM99_eda.ipynb 셀 50, 52의
> 반복·단독 조합 집계를 합쳤습니다.

- 같은 조합을 카메라 각도(70°, 75°, 90°)만 바꿔 찍은 사진이 있습니다. 조합마다 사진 수와 각도 구성을 셉니다.
- 그림은 사진 수(3장, 2장, 1장)마다 조합을 하나씩 `SEED`로 골라 한 줄에 70° / 75° / 90° 순서로 놓습니다.
  그 각도의 사진이 없으면 빈 칸입니다. 숫자는 `category_id`입니다.

In [ ]:
ANGLES = [70, 75, 90]

# 파일 이름의 각도와 JSON의 camera_la가 같은지 본다. JSON이 없는 사진은 따로 센다.
name_angle = img_df["file_name"].str.split("_").str[5].astype(int)
has_meta = img_df["camera_la"].notna()
angle_mismatch = has_meta & (name_angle != img_df["camera_la"])
print(
    f"[원본] 파일 이름의 각도 ≠ camera_la: {angle_mismatch.sum()}장 "
    f"(camera_la 없음: {(~has_meta).sum()}장)"
)
# jang 셀 19는 각도 외의 이름 부분까지 같은 사진을 묶었다. 같은 조합·각도가 둘 이상이면 차이가 난다.
per_angle = img_df.assign(angle=name_angle).groupby(["combo", "angle"]).size()
print(f"[원본] 같은 조합·같은 각도 사진이 2장 이상인 경우: {(per_angle > 1).sum()}건")

print("[원본] 각도별 사진 수")
display(name_angle.value_counts().sort_index().rename_axis("각도").rename("사진 수").to_frame().T)

combo_angles = (
    img_df.assign(angle=name_angle).groupby("combo")["angle"].agg(lambda s: tuple(sorted(s)))
)
combo_size_df = (
    combo_angles.str.len()
    .value_counts()
    .sort_index(ascending=False)
    .rename_axis("조합당 사진 수")
    .to_frame("조합 수")
)
combo_size_df["사진 수"] = combo_size_df.index * combo_size_df["조합 수"]
combo_size_df.loc["합계"] = combo_size_df.sum()
print("[원본] 조합당 사진 수")
display(combo_size_df)
angle_set_df = (
    combo_angles.value_counts()
    .rename_axis("각도 구성")
    .reset_index(name="조합 수")
    .assign(n=lambda d: d["각도 구성"].str.len())
    .sort_values(["n", "조합 수"], ascending=False)
    .drop(columns="n")
)
print("[원본] 각도 구성별 조합 수")
display(angle_set_df.set_index("각도 구성").T)

In [ ]:
def show_combo_angles(combos: list[str], title: str) -> None:
    """조합마다 한 줄씩 70°/75°/90° 칸에 사진과 bbox를 놓는다. 없는 각도는 빈 칸으로 둔다."""
    fig, axes = plt.subplots(
        len(combos), len(ANGLES), figsize=(12, 5.3 * len(combos)), squeeze=False
    )
    for row_axes, combo in zip(axes, combos, strict=True):
        for ax, angle in zip(row_axes, ANGLES, strict=True):
            match = img_df.loc[(img_df["combo"] == combo) & (name_angle == angle), "file_name"]
            if match.empty:
                ax.set_title(f"{combo}\n{angle}°", fontsize=9)
                ax.text(
                    0.5,
                    0.5,
                    "사진 없음",
                    ha="center",
                    va="center",
                    color="gray",
                    transform=ax.transAxes,
                )
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_linestyle("--")
                    spine.set_color("#c3c2b7")
                continue
            file_name = match.iloc[0]
            show_image(ax, TRAIN_IMG_DIR / file_name, f"{combo}\n{angle}°", fontsize=9)
            draw_bboxes(ax, ann_df[ann_df["file_name"] == file_name], color="green")
    fig.suptitle(title, x=0.01, ha="left", fontsize=12)
    plt.tight_layout(rect=(0, 0, 1, 0.98))
    plt.show()


n_per_combo = combo_angles.str.len()
sample_combos = [
    n_per_combo[n_per_combo == n].sample(1, random_state=SEED).index[0] for n in (3, 2, 1)
]
show_combo_angles(sample_combos, "[원본] 사진 3장 / 2장 / 1장 조합 예시 (초록: bbox)")

아래는 원본 상태 관측값입니다.

- 파일 이름의 각도와 `camera_la`는 모두 같고, 같은 조합·같은 각도 사진은 없습니다. 그래서 조합 이름만으로 묶어도
  jang 셀 19(각도 외 이름 부분까지 같은 사진 묶기)와 결과가 같습니다.
- 조합 114개: 3장(70°·75°·90°) 40개 120장, 2장 38개 76장, 1장 36개 36장, 합계 232장입니다(kwon 셀 44, jang 셀 19,
  JJAM 셀 50의 반복 조합 78개 196장, 셀 52의 단독 조합 36개와 같음).
- 2장 조합의 각도 구성은 (70°, 90°) 15개, (75°, 90°) 14개, (70°, 75°) 9개이고, 1장 조합은 90° 16개, 75° 12개,
  70° 8개입니다. 사진 수는 70° 72장, 75° 75장, 90° 85장으로 90°가 조금 많습니다.
- 같은 조합의 사진은 각도만 다른 같은 장면입니다. 알약 배치가 같아 보이므로 학습과 검증에 나눠 넣으면 점수가
  부풀려질 수 있습니다. 유사도는 8.2에서 잽니다.

**조합 하나를 학습·검증에 어떻게 넣느냐에 따라 검증이 재는 것** (jang 셀 20 요약)

| 조합당 사진 | 배치 | 검증이 재는 것 |
|---|---|---|
| 1장 | 학습 1 / 검증 0 | 이 조합으로는 검증하지 못함 |
| 1장 | 학습 0 / 검증 1 | 처음 보는 조합. 조합 속 알약 종류는 다른 사진에서 배웠어야 함 |
| 2–3장 | 같은 조합을 양쪽에 나눔 | 같은 장면을 다른 각도에서 알아보는지. 사진이 비슷하면 점수가 낙관적 |
| 2–3장 | 조합을 통째로 한쪽에 | 처음 보는 조합에 대한 일반화 |

- jang 결론: 처음 보는 조합의 일반화를 재려면 조합을 쪼개지 않습니다. 검증에만 있는 조합의 알약 종류가 학습에도
  있는지 확인해야 합니다. 조합이 적어 어떤 조합을 검증에 넣느냐에 따라 점수가 크게 달라질 수 있으므로, 여러 번
  나눠 모델을 매번 처음부터 학습해 비교합니다. 분할 기준은 8.4와 이슈 #26에서 정합니다.

### 7.2 각도·배경·조명별 bbox 크기와 비율

> 기준본: dongwook_eda.ipynb 셀 11. sohee_eda.ipynb 셀 15의 결측·빈 문자열 구분을 합쳤고,
> 같은 알약을 각도끼리 짝지어 비교하는 표를 더했습니다.

- 촬영 조건 값이 사진마다 다른지 먼저 봅니다. 값이 하나뿐인 조건은 학습에 정보를 주지 않습니다.
- 각도별 평균은 각도마다 찍힌 약 구성이 달라 약 차이가 섞입니다. 그래서 같은 조합의 같은 약을 각도끼리 짝지어
  90° 대비 bbox 너비·높이 비율도 봅니다. 비율이 1이면 각도가 bbox 크기를 바꾸지 않은 것입니다.

In [ ]:
IMG_COND_COLS = ["camera_la", "camera_lo", "size", "back_color", "light_color"]  # 사진 단위
LABEL_COND_COLS = ["drug_dir", "drug_S"]  # 알약 단위


def value_counts_text(values: pd.Series) -> dict:
    """결측과 빈 문자열을 구분해 값별 수를 센다(sohee 셀 15)."""
    labeled = values.astype(object).where(values.notna(), "<결측>").replace("", "<빈 문자열>")
    return labeled.value_counts().sort_index().to_dict()


cond_df = pd.DataFrame(
    [("사진", col, value_counts_text(img_df[col])) for col in IMG_COND_COLS]
    + [("bbox", col, value_counts_text(ann_df[col])) for col in LABEL_COND_COLS],
    columns=["단위", "조건", "값별 수"],
).set_index("조건")
cond_df["값 종류"] = cond_df["값별 수"].str.len()
print("[원본] 촬영 조건 값")
with pd.option_context("display.max_colwidth", None):
    display(cond_df)
back_side = ann_df[ann_df["drug_dir"] == "뒷면"]
print("[원본] 뒷면으로 찍힌 bbox의 약")
display(back_side.groupby(["category_id", "dl_name"]).size().rename("bbox 수").to_frame())

In [ ]:
# 각도별 bbox 크기(dongwook 셀 11). ratio = 긴 변 / 짧은 변, 좌표·area 오류 bbox는 뺀다.
ok_box = ~flag_df.any(axis=1) & ann_df[BBOX_COLS].notna().all(axis=1)
angle_box = ann_df[ok_box].assign(
    combo=lambda d: d["file_name"].str.split("_").str[0],
    ratio=lambda d: d[["w", "h"]].max(axis=1) / d[["w", "h"]].min(axis=1),
    area_ratio=lambda d: d["w"] * d["h"] / (d["width"] * d["height"]) * 100,
)
print(f"[원본] 각도별 bbox 크기 (bbox {len(angle_box)}개)")
display(
    angle_box.groupby("camera_la")
    .agg(
        n_images=("file_name", "nunique"),
        n_boxes=("annotation_id", "size"),
        w_mean=("w", "mean"),
        h_mean=("h", "mean"),
        ratio_mean=("ratio", "mean"),
        area_ratio_mean=("area_ratio", "mean"),
    )
    .round(3)
)

# 같은 조합의 같은 약을 각도끼리 짝짓는다. 같은 사진에 같은 약이 둘 이상이면 평균을 쓴다.
paired = angle_box.groupby(["combo", "category_id", "camera_la"])[["w", "h"]].mean()
paired = paired.unstack("camera_la")
pair_rows, pair_values = [], {}
for angle in [70, 75]:
    for col in ["w", "h"]:
        ratio = (paired[(col, angle)] / paired[(col, 90)]).dropna()
        pair_values[f"{angle}°/90° {col}"] = ratio
        pair_rows.append(
            {
                "비교": f"{angle}°/90° {col}",
                "짝 수": len(ratio),
                "중앙값": ratio.median(),
                "하위 10%": ratio.quantile(0.1),
                "상위 10%": ratio.quantile(0.9),
            }
        )
print("[원본] 같은 조합·같은 약의 90° 대비 bbox 크기 비율")
display(pd.DataFrame(pair_rows).set_index("비교").round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].boxplot(
    [angle_box.loc[angle_box["camera_la"] == a, "ratio"] for a in ANGLES],
    tick_labels=[f"{a}°" for a in ANGLES],
)
axes[0].set(title="[원본] 각도별 bbox 긴 변 / 짧은 변", ylabel="비율")
axes[1].boxplot(list(pair_values.values()), tick_labels=list(pair_values))
axes[1].axhline(1, color="gray", linestyle="--")
axes[1].set(title="[원본] 같은 약의 90° 대비 크기 (점선: 같음)", ylabel="비율")
for ax in axes:
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

아래는 원본 상태 관측값입니다.

- 배경(`연회색 배경`), 조명(`주백색`), `camera_lo`(0), `size`(200), `drug_S`(`정상알약`)는 값이 하나뿐입니다.
  사진마다 다른 촬영 조건은 카메라 각도뿐입니다. 결측이나 빈 문자열은 없습니다.
- `drug_dir`은 bbox 763개 중 751개가 `앞면`, 12개가 `뒷면`입니다(5종). 앞뒷면은 학습에서 따로 다룰 만큼 많지 않습니다.
- 각도별 평균 bbox 너비·높이·긴 변/짧은 변 비율은 거의 같습니다(dongwook 셀 11).
  같은 약끼리 짝지어도(182쌍) 90° 대비 중앙값이 70° 너비 1.005·높이 1.007, 75° 너비 1.005·높이 1.017로
  1–2% 차이입니다.
  카메라를 70°로 기울여도 bbox가 눈에 띄게 납작해지지 않습니다.
- 그래서 각도 차이를 흉내 내는 원근 변환 증강은 우선순위가 낮습니다. 배경·조명이 하나뿐이므로 test 사진의 배경·조명이
  다르면 색·밝기 증강이 필요합니다. test 사진의 촬영 조건은 메타데이터가 없어 사진으로 확인해야 합니다.

### 7.3 알약 실제 크기와 픽셀 크기

> 기준본: dongwook_eda.ipynb 셀 13–17 (def-uk). JJAM99_eda.ipynb 과거 셀 75–85 (potato, 커밋 `ac96ac4`)의
> GrabCut 장단축 측정, 조합 단위 배율 보정·검증, 검수 후보를 합쳤습니다. 이 셀들은 현재 JJAM 노트북에는 없습니다.

- `thick`(두께), `leng_long`(장축), `leng_short`(단축)은 mm 단위이고 클래스마다 고정값입니다. 0.0은 값이 없다는 뜻입니다.
- 크기끼리의 상관은 클래스 단위(클래스당 한 행), bbox 픽셀 크기와의 상관은 bbox 단위로 봅니다.
  크기가 0인 라벨과 좌표·area 오류 bbox(5.1, 5.4)는 뺍니다.
- bbox의 긴 변·짧은 변은 알약이 기울어져 있으면 실제 장축·단축보다 깁니다. 그래서 `[RUN_HEAVY]`에서는 bbox 안을
  GrabCut으로 분리하고 최소 외접 회전 사각형으로 장축·단축을 다시 잽니다(JJAM 과거 셀 78).
- 1mm당 픽셀 수(배율)는 조합 75%로 정하고 나머지 25% 조합으로 검증합니다. 같은 사진으로 배율을 맞추고 평가하지
  않기 위해서입니다(JJAM 과거 셀 76, 80). 검증 오차는 메타데이터와의 차이이며 실측 정확도가 아닙니다.

In [ ]:
SIZE_COLS = ["thick", "leng_long", "leng_short"]  # mm, 클래스마다 고정
PX_COLS = ["long_px", "short_px"]  # bbox 긴 변·짧은 변

has_zero = (ann_df[SIZE_COLS] == 0).any(axis=1)
print(f"[원본] 크기가 0인 bbox: {has_zero.sum()}개")
display(
    ann_df.loc[has_zero, ["category_id", "dl_name"]].value_counts().rename("bbox 수").to_frame()
)
size_df = ann_df[ok_box & ~has_zero].assign(
    long_px=lambda d: d[["w", "h"]].max(axis=1),
    short_px=lambda d: d[["w", "h"]].min(axis=1),
)
class_mm_df = size_df.drop_duplicates("category_id")

fig, axes = plt.subplots(
    1, 2, figsize=(13, 5.5), gridspec_kw={"width_ratios": [3, 5]}, layout="constrained"
)
plot_corr_heatmap(
    axes[0], class_mm_df[SIZE_COLS].corr(), f"[원본] 실제 크기끼리 (클래스 {len(class_mm_df)}개)"
)
mesh = plot_corr_heatmap(
    axes[1],
    size_df[SIZE_COLS + PX_COLS].corr(),
    f"[원본] 실제 크기 vs bbox 픽셀 (bbox {len(size_df)}개)",
)
fig.colorbar(mesh, ax=axes, shrink=0.8, label="상관계수")
plt.show()

# 1mm가 몇 픽셀인지: 각도마다 거의 같으면 촬영 거리가 고정된 것이다.
size_df = size_df.assign(
    long_px_per_mm=size_df["long_px"] / size_df["leng_long"],
    short_px_per_mm=size_df["short_px"] / size_df["leng_short"],
)
print("[원본] 각도별 1mm당 픽셀 수 (bbox 긴 변·짧은 변 기준)")
display(
    size_df.groupby("camera_la")[["long_px_per_mm", "short_px_per_mm"]]
    .agg(["median", "std"])
    .round(2)
)

아래는 원본 상태 관측값입니다.

- 크기가 0인 bbox는 18개이고 모두 `16548`(가바토파정 100mg)입니다(4.3의 크기 0.0과 같음). 이 약은 크기 분석에서 빠집니다.
- 실제 크기끼리는 두께와 장축의 상관이 0.84로 높습니다. 긴 약이 대체로 두껍습니다.
- 장축(mm)과 bbox 긴 변(px)의 상관은 0.98로 거의 비례합니다. 단축과 bbox 짧은 변은 0.60으로 낮습니다.
  기울어진 길쭉한 알약은 bbox 짧은 변이 실제 단축보다 길어지기 때문입니다.
- 1mm당 픽셀 수는 각도마다 거의 같습니다(긴 변 기준 중앙값 약 25px). 촬영 거리가 고정되어 있어 픽셀 크기로
  실제 크기를 짐작할 수 있습니다. 짧은 변 기준은 위와 같은 이유로 크고 흩어집니다.

In [ ]:
# [RUN_HEAVY] GrabCut 장단축 측정. 입력 지문·방식·seed가 저장 당시와 같은 캐시만 재사용한다.
LABEL_KEYS = ["source_json", "annotation_id"]
axes_cache = CACHE_DIR / "pill_axes.json"
axes_key = {
    "version": AXIS_VERSION,
    "seed": SEED,
    "data_state": "원본",
    "inputs": padding_key["inputs"],  # 5.7과 같은 입력(train 이미지·JSON)
}
if RUN_HEAVY:
    start = time.perf_counter()
    axes_df = measure_pill_axes(ann_df, TRAIN_IMG_DIR, seed=SEED)
    axes_status = f"실행 ({time.perf_counter() - start:.0f}초)"
    if SAVE_OUTPUTS:
        save_cache(axes_cache, axes_key, axes_df)
else:
    axes_df, axes_status = load_cache(axes_cache, axes_key)
    if axes_df is None:
        axes_status = f"{NOT_RUN} ({axes_status})"

# axes_df가 None이면 미실행이다. 뒤 셀은 None을 확인하고 건너뛴다.
print(f"[원본] GrabCut 장단축 측정: {axes_status}")
if axes_df is not None:
    # 캐시에서 읽으면 index가 없으므로 라벨 식별자로 ann_df에 맞춘다.
    axes_df = ann_df[LABEL_KEYS].merge(axes_df, on=LABEL_KEYS, how="left").set_index(ann_df.index)
    display(axes_df["status"].value_counts(dropna=False).rename("bbox 수").to_frame().T)

In [ ]:
AXIS_VALIDATION_FRACTION = 0.25  # 배율 검증에 떼어 둘 조합 비율 (JJAM 과거 셀 76)
AXIS_ERROR_MAX = 0.20  # 장축·단축 상대오차가 이 이상이면 검수 후보. 오류 확정이 아니다.

if axes_df is None:
    print(f"[원본] 배율 보정·검증: {NOT_RUN} (GrabCut 장단축 측정 없음)")
else:
    info_cols = ["file_name", "category_id", "dl_name", "camera_la", "width", "height"]
    axis_df = ann_df[[*info_cols, *BBOX_COLS]].join(
        ann_df[["leng_long", "leng_short"]].rename(columns=lambda c: c.split("_")[1] + "_mm")
    )
    axis_df = axis_df.join(axes_df.drop(columns=LABEL_KEYS))
    axis_df["combo"] = axis_df["file_name"].str.split("_").str[0]
    axis_df["metadata_valid"] = (
        (axis_df["long_mm"] > 0)
        & (axis_df["short_mm"] > 0)
        & (axis_df["long_mm"] >= axis_df["short_mm"])
    )
    axis_usable = (axis_df["status"] == "estimated") & axis_df["metadata_valid"] & ok_box

    # 조합 단위로 보정·검증을 나눈다. 같은 조합의 사진은 한쪽에만 들어간다.
    combos = np.sort(axis_df["combo"].unique())
    n_val = round(len(combos) * AXIS_VALIDATION_FRACTION)
    val_combos = set(np.random.default_rng(SEED).permutation(combos)[:n_val])
    axis_df["split"] = np.where(axis_df["combo"].isin(val_combos), "validation", "calibration")
    calib = axis_df[axis_usable & (axis_df["split"] == "calibration")]
    # 사진이 많은 조합에 끌리지 않도록 조합별 중앙값의 중앙값을 배율로 쓴다.
    PX_PER_MM = (calib["long_px"] / calib["long_mm"]).groupby(calib["combo"]).median().median()
    for axis in ["long", "short"]:
        axis_df[f"expected_{axis}_px"] = axis_df[f"{axis}_mm"] * PX_PER_MM
        axis_df[f"{axis}_error"] = (
            axis_df[f"{axis}_px"] / axis_df[f"expected_{axis}_px"] - 1
        ).where(axis_usable)
    axis_df["worst_error"] = axis_df[["long_error", "short_error"]].abs().max(axis=1)

    def error_summary(by: list[str]) -> pd.DataFrame:
        """쓸 수 있는 측정의 상대오차를 묶음별로 요약한다. bias는 부호 있는 중앙값이다."""
        valid = axis_df[axis_usable]
        return valid.groupby(by).agg(
            n_boxes=("long_error", "size"),
            n_combos=("combo", "nunique"),
            long_abs_median=("long_error", lambda v: v.abs().median()),
            long_abs_p90=("long_error", lambda v: v.abs().quantile(0.9)),
            long_bias=("long_error", "median"),
            short_abs_median=("short_error", lambda v: v.abs().median()),
            short_bias=("short_error", "median"),
        )

    print(
        f"[원본] 보정 조합 {len(combos) - n_val}개로 정한 배율: 1mm = {PX_PER_MM:.2f}px "
        f"(검증 조합 {n_val}개, 쓸 수 있는 측정 {axis_usable.sum()}개)"
    )
    display(error_summary(["split"]).round(3))
    angle_errors = error_summary(["split", "camera_la"])
    display(angle_errors.round(3))

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), gridspec_kw={"width_ratios": [4, 4, 5]})
    valid = axis_df[axis_usable]
    bins = np.linspace(0, valid["worst_error"].max() * 100, 30)
    for split, rows in valid.groupby("split"):
        axes[0].hist(rows["short_error"].abs() * 100, bins=bins, alpha=0.6, label=split)
    axes[0].axvline(AXIS_ERROR_MAX * 100, color="red", linestyle="--")
    axes[0].set(
        title="[원본] 단축 상대오차 (점선: 검수 기준)", xlabel="|오차| (%)", ylabel="bbox 수"
    )
    axes[0].legend()
    for split, rows in angle_errors.reset_index().groupby("split"):
        axes[1].plot(rows["camera_la"].astype(str), rows["long_bias"] * 100, "o-", label=split)
    axes[1].axhline(0, color="gray", linestyle="--")
    axes[1].set(title="[원본] 각도별 장축 오차 중앙값", xlabel="camera_la", ylabel="오차 (%)")
    axes[1].legend()
    compare = (
        valid[["long_mm", "short_mm"]]
        .join(valid[["long_px", "short_px"]].add_prefix("grabcut_"))
        .join(valid[["w", "h"]].max(axis=1).rename("bbox_long_px"))
        .join(valid[["w", "h"]].min(axis=1).rename("bbox_short_px"))
    )
    plot_corr_heatmap(axes[2], compare.corr(), f"[원본] mm vs 픽셀 (bbox {len(valid)}개)")
    plt.tight_layout()
    plt.show()

In [ ]:
SIZE_TOL_MM = 0.5  # 장축·단축이 모두 이 이내로 같으면 크기만으로 구별하기 어려운 쌍으로 본다.

if axes_df is None:
    print(f"[원본] 클래스별 크기 비교: {NOT_RUN} (GrabCut 장단축 측정 없음)")
else:
    valid = axis_df[axis_usable].assign(
        measured_long_mm=lambda d: d["long_px"] / PX_PER_MM,
        measured_short_mm=lambda d: d["short_px"] / PX_PER_MM,
    )
    class_axis_df = (
        valid.groupby(["category_id", "dl_name"])
        .agg(
            n_boxes=("long_px", "size"),
            n_combos=("combo", "nunique"),
            long_mm=("long_mm", "first"),
            short_mm=("short_mm", "first"),
            measured_long_median=("measured_long_mm", "median"),
            measured_short_median=("measured_short_mm", "median"),
            measured_long_iqr=("measured_long_mm", lambda v: v.quantile(0.75) - v.quantile(0.25)),
        )
        .sort_values("long_mm")
    )
    meta = class_axis_df.reset_index()[["category_id", "long_mm", "short_mm"]]
    pairs = meta.merge(meta, how="cross", suffixes=("_a", "_b"))
    pairs = pairs[
        (pairs["category_id_a"] < pairs["category_id_b"])
        & ((pairs["long_mm_a"] - pairs["long_mm_b"]).abs() <= SIZE_TOL_MM)
        & ((pairs["short_mm_a"] - pairs["short_mm_b"]).abs() <= SIZE_TOL_MM)
    ]
    n_close = pd.concat([pairs["category_id_a"], pairs["category_id_b"]]).nunique()
    print(
        f"[원본] 장축·단축이 모두 ±{SIZE_TOL_MM}mm 이내인 클래스 쌍: {len(pairs)}쌍 "
        f"(관련 클래스 {n_close}종 / {len(class_axis_df)}종)"
    )
    with pd.option_context("display.max_rows", None):
        display(class_axis_df.round(2))

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(
        valid["measured_long_mm"],
        valid["measured_short_mm"],
        s=8,
        alpha=0.4,
        color="gray",
        label="측정 (bbox 하나)",
    )
    ax.scatter(
        class_axis_df["long_mm"],
        class_axis_df["short_mm"],
        s=40,
        marker="x",
        color="red",
        label="메타데이터 (클래스 하나)",
    )
    ax.plot([0, 25], [0, 25], color="#c3c2b7", linestyle="--", linewidth=1)
    ax.set(title="[원본] 알약 장축·단축 (점선 위: 원형)", xlabel="장축 (mm)", ylabel="단축 (mm)")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# 검수 후보: 측정 실패·불확실, 크기 메타데이터 없음, 장축 또는 단축 상대오차 AXIS_ERROR_MAX 이상.
# 초록·하늘 실선은 메타데이터 × 배율로 그린 예상 장축·단축, 주황 점선은 GrabCut으로 잰 장단축이다.
if axes_df is None:
    print(f"[원본] 크기 검수 후보: {NOT_RUN} (GrabCut 장단축 측정 없음)")
else:
    measured = axis_df["status"].notna()  # 좌표가 없는 라벨은 측정 대상이 아니다.
    reasons = pd.DataFrame(
        {
            "측정 불확실": measured & (axis_df["status"] != "estimated"),
            "크기 정보 없음": measured & ~axis_df["metadata_valid"],
            "좌표·area 오류": measured & ~ok_box,
            "장축 오차": axis_df["long_error"].abs() >= AXIS_ERROR_MAX,
            "단축 오차": axis_df["short_error"].abs() >= AXIS_ERROR_MAX,
        }
    )
    axis_df["review_reason"] = reasons.apply(lambda r: ", ".join(r.index[r]), axis=1)
    axis_review_df = axis_df[axis_df["review_reason"] != ""]
    print(
        f"[원본] 크기 검수 후보: {len(axis_review_df)}개 / 측정 {measured.sum()}개 "
        "(이유는 겹칠 수 있음)"
    )
    display(reasons.sum().rename("bbox 수").to_frame().T)
    big_error = axis_review_df[axis_review_df["worst_error"] >= AXIS_ERROR_MAX]
    display(
        big_error.sort_values("worst_error", ascending=False)[
            ["file_name", "category_id", "dl_name", "camera_la", "long_mm", "short_mm"]
            + ["long_error", "short_error", "status"]
        ].round(3)
    )

    # 오차가 큰 후보 3개와 측정이 불확실한 후보 3개를 확대해 본다. 이미지 밖 좌표는 5.1에서 봤다.
    preview = pd.concat(
        [
            big_error.sort_values("worst_error", ascending=False).head(3),
            axis_review_df[
                (axis_review_df["status"] != "estimated") & axis_review_df["center_x"].notna()
            ].head(3),
        ]
    )
    fig, axes = image_grid(len(preview), cols=3, cell=(4.5, 4.8))
    for ax, row in zip(axes, preview.itertuples(), strict=True):
        error_text = (
            f"장축 {row.long_error:+.0%}, 단축 {row.short_error:+.0%}"
            if pd.notna(row.worst_error)
            else row.status
        )
        show_image(
            ax, TRAIN_IMG_DIR / row.file_name, f"{row.category_id} {row.camera_la}°\n{error_text}"
        )
        draw_bbox(ax, (row.x, row.y, row.w, row.h))
        if pd.notna(row.angle_rad):
            for axis, offset, color in [("long", 0, "lime"), ("short", np.pi / 2, "cyan")]:
                angle = row.angle_rad + offset
                for length, line_color, style in [
                    (getattr(row, f"expected_{axis}_px"), color, "-"),
                    (getattr(row, f"{axis}_px"), "orange", "--"),
                ]:
                    if pd.isna(length):
                        continue
                    dx, dy = length / 2 * np.cos(angle), length / 2 * np.sin(angle)
                    ax.plot(
                        [row.center_x - dx, row.center_x + dx],
                        [row.center_y - dy, row.center_y + dy],
                        color=line_color,
                        linestyle=style,
                        linewidth=1.8,
                    )
        zoom_to(ax, [(row.x, row.y, row.w, row.h)], margin=0.3, limit=(row.width, row.height))
    fig.legend(
        handles=[
            plt.Line2D([], [], color="lime", label="예상 장축 (메타데이터 × 배율)"),
            plt.Line2D([], [], color="cyan", label="예상 단축"),
            plt.Line2D([], [], color="orange", linestyle="--", label="측정 장단축 (GrabCut)"),
            plt.Line2D([], [], color="red", label="원본 bbox"),
        ],
        loc="lower center",
        ncol=4,
    )
    plt.tight_layout(rect=(0, 0.05, 1, 1))
    plt.show()

아래는 `RUN_HEAVY=True`로 실행해 관측한 원본 상태 값입니다. 미실행이면 위 세 셀은 결과를 내지 않습니다.

- GrabCut 측정 763개 중 740개가 자동 검사를 통과했습니다(`estimated`). 윤곽이 bbox에 닿은 것 16개, 두 seed의
  윤곽이 다른 것 5개, 이미지 밖 좌표 1개, 분리 실패 1개는 검수 후보로 남깁니다.
- 보정 조합 86개로 정한 배율은 1mm ≈ 23.6px입니다. 검증 조합 28개에서 장축 오차 중앙값 2.0%, 상위 10% 5.8%,
  단축 오차 중앙값 2.4%로 보정 조합(1.8%, 6.4%, 2.7%)과 비슷합니다. JJAM 과거 표본(12조합, 1mm = 23.24px,
  검증 장축 오차 중앙값 2.7%)과도 같은 수준입니다. bbox 긴 변으로 잰 dongwook 값(약 25px)이 더 큰 것은 bbox에
  여백(5.7)이 있기 때문입니다.
- GrabCut 장축·단축은 메타데이터와 상관 0.99·0.96으로, bbox 긴 변·짧은 변(0.98·0.60)보다 단축이 크게 좋아집니다.
- 70°에서는 장축이 메타데이터보다 약 2% 짧게(보정 −1.7%, 검증 −2.6%), 오차 중앙값도 4%로 90°(1–2%)보다 큽니다.
  기울어진 카메라의 원근 영향이 작게 보입니다. 7.2에서 bbox 크기가 거의 같았던 것과 어긋나지 않는 작은 차이입니다.
- 오차 20% 이상 18개는 모두 단축을 메타데이터보다 20–65% 길게 잰 경우입니다. 캡슐(`41768`, `2483`)과 길쭉한 정제가
  많습니다. 확대해 보면 `27777`(카나브정 60mg)은 알약 폭이 메타데이터 단축 4.0mm보다 분명히 넓고, `41768`은 측정 선이
  캡슐 폭과 맞습니다. 메타데이터 단축이 실제 폭과 다르게 기록됐거나 그림자를 함께 잡았을 수 있습니다.
  원인을 확인하기 전에는 라벨 오류로 보지 않습니다. 이 단계의 bbox는 정답 그대로 둡니다.
- 메타데이터 기준으로 장축·단축이 모두 ±0.5mm 이내인 클래스 쌍이 46쌍이고, 55종 중 34종이 한 쌍 이상에 들어갑니다.
  대부분 지름 7–8.5mm 원형 정제끼리입니다.
  측정 오차(상위 10% 약 6%, 8mm 알약에서 약 0.5mm)를 생각하면 크기만으로 구별할 수 없는 약이 많습니다.
  크기는 색·모양과 함께 쓰는 보조 정보이고, 분류에 쓸지는 이미지만 쓴 모델과 비교해 정합니다(JJAM 과거 셀 85).
- 여기서는 정답 bbox로 쟀습니다. 학습한 모델에 쓰려면 예측 bbox로 다시 검증해야 합니다. 거리·줌이 다른 test 사진에는
  이 배율을 그대로 쓸 수 없습니다.

## 8. 데이터 누수와 분할

> 출처: eda_kwon.ipynb 셀 38–39, 45–53, 87–120 (gwon753-dev),
> sohee_eda.ipynb 셀 14–17 (bravedolphinhere),
> EDA_jang.ipynb 셀 20 (younghoon1215)

- train 232장은 조합 114개를 1–3장씩 찍은 것입니다(7.1). 사진을 무작위로 나누면 같은 장면이 학습과 검증에 함께
  들어갈 수 있습니다. 이 장에서는 그런 누수가 얼마나 있는지 재고 분할 후보를 비교할 근거를 정리합니다.
- 용어(kwon 셀 111, 117)

| 용어 | 뜻 |
|---|---|
| 조합 | 사진에 찍힌 약 이름 목록. 파일 이름 첫 밑줄 앞 부분이며 사진 1–3장 |
| 세팅 | 그 약들을 판 위 어디에 놓았는가 |
| 세팅 공유 | 조합은 다른데 공통 약 2개 이상이 같은 자리에 있음. 알약 일부만 바꿔 찍은 사진 |
| 묶음 | 세팅을 공유하는 조합을 이어 만든 모임. 묶음 크기는 조합 수 |

- 분석은 `[원본]` 기준입니다. 세팅 묶음은 박스 위치로 정하므로 `[보정]`(5.8) 결과를 나란히 둡니다.
- 분할 방식은 여기서 정하지 않습니다. 근거만 정리하고 결정은 이슈 #26에서 합니다.

### 8.1 희귀 약은 사실상 장면 하나

> 기준본: eda_kwon.ipynb 셀 45–53

- bbox가 `RARE_MAX`(5)개 이하인 약을 희귀 약으로 봅니다. 6.1의 가장 낮은 불균형 단계와 같습니다.
- 약마다 bbox가 나온 조합 수를 셉니다. 아래 첫 그림은 한 줄이 약 하나, 칸 하나가 bbox 하나이고 칸 안의 숫자는 각도입니다.
  칸 색은 조합입니다. 한 줄이 한 색이면 그 약의 bbox가 모두 한 장면에서 나온 것입니다.

In [ ]:
RARE_MAX = 5  # bbox가 이 개수 이하인 약을 희귀 약으로 본다 (kwon 셀 46)


def rare_classes(df: pd.DataFrame) -> pd.DataFrame:
    """bbox가 RARE_MAX개 이하인 약마다 bbox 수, 나온 조합 수, 조합 목록을 센다."""
    boxes = df.assign(combo=df["file_name"].str.split("_").str[0])
    counts = boxes["category_id"].value_counts()
    rare = boxes[boxes["category_id"].isin(counts[counts <= RARE_MAX].index)]
    return (
        rare.groupby("category_id")
        .agg(
            n_boxes=("file_name", "size"),
            n_combos=("combo", "nunique"),
            combos=("combo", lambda s: sorted(set(s))),
        )
        .join(cat_df.set_index("category_id")["dl_name"].str.strip())
        .sort_values(["n_combos", "n_boxes"], ascending=False)
    )


rare_df = rare_classes(ann_df)
rare_states = {"[원본]": rare_df}
if n_applied:
    rare_states["[보정]"] = rare_classes(ann_corrected_df)
print(f"[보정] 희귀 약: {correction_status}")
display(
    pd.DataFrame(
        {
            state: {
                f"bbox {RARE_MAX}개 이하 약": len(df),
                "그중 조합 1개에서만 나옴": int((df["n_combos"] == 1).sum()),
                "희귀 약 bbox 수": int(df["n_boxes"].sum()),
            }
            for state, df in rare_states.items()
        }
    )
)
print("[원본] 희귀 약 목록")
with pd.option_context("display.max_colwidth", None):
    display(rare_df[["dl_name", "n_boxes", "n_combos", "combos"]])

In [ ]:
# 칸 하나 = bbox 하나, 칸 색 = 조합, 숫자 = 각도 (kwon 셀 48)
COMBO_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]  # 첫 번째 조합: 파랑, 두 번째: 주황, 세 번째: 초록
rare_boxes = ann_df[ann_df["category_id"].isin(rare_df.index)].assign(
    combo=lambda d: d["file_name"].str.split("_").str[0],
    angle=lambda d: d["file_name"].str.split("_").str[5].astype(int),
)
n_cols = int(rare_df["n_boxes"].max())
fig, ax = plt.subplots(figsize=(13, 0.5 * len(rare_df) + 1.5))
for row, (cid, r) in enumerate(rare_df.iloc[::-1].iterrows()):
    boxes = rare_boxes[rare_boxes["category_id"] == cid].sort_values(["combo", "angle"])
    for col, b in enumerate(boxes.itertuples()):
        color = COMBO_COLORS[r["combos"].index(b.combo) % len(COMBO_COLORS)]
        ax.add_patch(plt.Rectangle((col + 0.05, row + 0.1), 0.9, 0.8, color=color))
        ax.text(col + 0.5, row + 0.5, f"{b.angle}°", ha="center", va="center", color="white")
    ax.text(n_cols + 0.3, row + 0.5, "  /  ".join(r["combos"]), va="center", fontsize=8)
ax.set_xlim(0, n_cols + 4.5)
ax.set_ylim(0, len(rare_df))
ax.set_yticks(
    [k + 0.5 for k in range(len(rare_df))],
    [f"{cid}  {name}" for cid, name in rare_df["dl_name"].iloc[::-1].items()],
    fontsize=9,
)
ax.set_xticks([k + 0.5 for k in range(n_cols)], [f"bbox {k + 1}" for k in range(n_cols)])
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in COMBO_COLORS[:2]]
ax.legend(
    handles,
    ["첫 번째 조합", "두 번째 조합"],
    loc="upper center",
    bbox_to_anchor=(0.35, -0.04),
    frameon=False,
    ncol=2,
)
n_single = int((rare_df["n_combos"] == 1).sum())
ax.set_title(
    f"[원본] bbox {RARE_MAX}개 이하 약 {len(rare_df)}종 중 {n_single}종은 "
    "한 조합(같은 색)에서만 나온다",
    loc="left",
    fontsize=12,
)
plt.tight_layout()
plt.show()

In [ ]:
# 조합 2개에서 나오는 희귀 약과 1개에서만 나오는 희귀 약을 하나씩 사진으로 본다 (kwon 셀 50, 52).
# 빨강: 그 약의 박스, 회색: 같이 찍힌 다른 약. 제목 색 = 조합(첫 번째 파랑, 두 번째 주황)
show_ids = [rare_df[rare_df["n_combos"] > 1].index[0], rare_df[rare_df["n_combos"] == 1].index[0]]
n_cols = int(rare_df.loc[show_ids, "n_boxes"].max())
fig, axes = plt.subplots(
    len(show_ids), n_cols, figsize=(3.4 * n_cols, 5 * len(show_ids)), squeeze=False
)
for row_axes, cid in zip(axes, show_ids, strict=True):
    boxes = rare_boxes[rare_boxes["category_id"] == cid].sort_values(["combo", "angle"])
    combos_of = rare_df.at[cid, "combos"]
    for ax in row_axes:
        ax.axis("off")
    for ax, b in zip(row_axes, boxes.itertuples(), strict=False):
        color = COMBO_COLORS[combos_of.index(b.combo) % len(COMBO_COLORS)]
        show_image(ax, TRAIN_IMG_DIR / b.file_name)
        others = ann_df[(ann_df["file_name"] == b.file_name) & (ann_df["category_id"] != cid)]
        draw_bboxes(ax, others, color="gray")
        draw_bboxes(ax, ann_df.loc[[b.Index]], color="red")
        ax.set_title(f"{b.combo}\n{b.angle}°", fontsize=9, color=color)
    row_axes[0].text(
        -0.05,
        0.5,
        f"{cid} {rare_df.at[cid, 'dl_name']}\n조합 {len(combos_of)}개",
        transform=row_axes[0].transAxes,
        rotation=90,
        ha="right",
        va="center",
        fontsize=11,
    )
fig.suptitle("[원본] 희귀 약의 bbox가 나온 사진 (빨강: 그 약, 제목 색: 조합)", x=0.01, ha="left")
plt.tight_layout(rect=(0.02, 0, 1, 0.96))
plt.show()

아래는 원본 상태 관측값입니다.

- bbox 5개 이하 약은 18종이고, 그중 17종은 조합 하나에서만 나옵니다(kwon 셀 46과 같음). 17종은 모두 bbox 3개로,
  한 조합을 70°·75°·90°로 찍은 사진 3장에서 나온 것입니다. 조합 2개에서 나오는 약은 `4543`(에어탈정) 하나입니다.
- `[보정]`은 라벨 누락 추가로 `4543`이 6개가 되어 18종 → 17종입니다. 조합 하나에서만 나오는 17종은 그대로입니다.
- 희귀 약의 데이터는 bbox 3개가 아니라 사실상 장면 1개입니다(kwon 셀 53). 그래서 어떻게 나눠도 손해가 있습니다.
  - 사진 단위로 나누면 검증 사진과 같은 장면이 거의 항상 학습에 있습니다. 모델이 약이 아니라 장면을 외워도 점수가 높게 나옵니다.
  - 조합 단위로 나누면 누수는 막지만, 유일한 장면이 검증으로 가면 그 약은 학습 데이터가 0개가 되어 AP가 0이 됩니다.
    mAP는 클래스 평균이라 약 하나당 1/56씩 깎입니다.
  - 각도만 다른 같은 장면 3장은 서로 독립된 데이터 3개가 아닙니다. 모델이 배울 수 있는 다양성은 1장에 가깝습니다.
- 손해의 크기는 8.4에서 분할 후보별로 잽니다.

### 8.2 같은 조합 사진의 유사도

> 기준본: eda_kwon.ipynb 셀 87–107. sohee_eda.ipynb 셀 14–15의 aHash 유사 쌍 후보(거리 3 이하)와 비교했습니다.
> 해시 함수는 `data_process/data_processing.py`의 `image_hashes`입니다.

- pHash는 사진을 흑백 32x32로 줄여 밝기 구도를 64비트로 요약한 값입니다. 두 해시에서 같은 비트의 비율(`phash_sim`)이
  1이면 구도가 같고 0.5 근처면 무관한 사진입니다. aHash는 8x8로 줄여 평균보다 밝은지를 64비트로 만든 값이며
  다른 비트 수(해밍 거리, `ahash_dist`)로 비교합니다.
- 모든 사진의 배경과 조명이 같아서 무관한 사진끼리도 0.5보다 조금 높게 나옵니다. 그래서 다른 조합끼리의 유사도를
  기준선으로 둡니다.
- 회의 쟁점(kwon 셀 87): 70° 사진은 75°·90°와 배치가 달라 보이지만, 180° 돌리면 거의 같아 보입니다. 70° 사진을 돌려
  다시 비교하고, 다른 조합의 70° 사진도 똑같이 돌려 대조군으로 씁니다. 사진 가운데의 둥근 조명 얼룩처럼 돌려도
  비슷한 무늬 때문에 유사도가 오를 수 있기 때문입니다.
- 사진 232장의 해시는 10초 안팎이라 `RUN_HEAVY`와 관계없이 계산합니다.

In [ ]:
AHASH_MAX = 3  # aHash 거리가 이 이하이면 유사 쌍 후보 (sohee 셀 15)
GROUP_ORDER = ["같은 조합 75°-90°", "같은 조합 70°-75°", "같은 조합 70°-90°", "다른 조합"]
GROUP_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#898781"]  # 다른 조합(기준선)은 회색

start = time.perf_counter()
hash_df = image_hashes([TRAIN_IMG_DIR / f for f in img_df["file_name"]])
print(f"[원본] 해시 계산: {len(hash_df)}장 ({time.perf_counter() - start:.0f}초)")
photo_files = img_df["file_name"].to_numpy()
photo_combo = img_df["combo"].to_numpy()
photo_angle = img_df["file_name"].str.split("_").str[5].astype(int).to_numpy()
phash = np.stack(hash_df["phash"])
ahash = np.stack(hash_df["ahash"])
phash_sim = hash_similarity(phash, phash)
# [a, b]: 사진 a를 180° 돌린 해시와 사진 b의 원래 해시를 비교한 값
phash_rot_sim = hash_similarity(np.stack(hash_df["phash_rot180"]), phash)
ahash_dist = np.rint((1 - hash_similarity(ahash, ahash)) * ahash.shape[1]).astype(int)

# 모든 사진 쌍. 같은 조합은 각도 쌍별로, 다른 조합은 하나로 묶는다.
pi, pj = np.triu_indices(len(img_df), k=1)
low, high = (
    np.minimum(photo_angle[pi], photo_angle[pj]),
    np.maximum(photo_angle[pi], photo_angle[pj]),
)
hash_pairs = pd.DataFrame(
    {
        "i": pi,
        "j": pj,
        "group": [
            f"같은 조합 {a}°-{b}°" if same else "다른 조합"
            for a, b, same in zip(low, high, photo_combo[pi] == photo_combo[pj], strict=True)
        ],
        "phash_sim": phash_sim[pi, pj],
        "ahash_dist": ahash_dist[pi, pj],
    }
)
print("[원본] 사진 쌍의 pHash 유사도")
display(hash_pairs.groupby("group")["phash_sim"].describe().reindex(GROUP_ORDER).round(3))

near_pairs = hash_pairs[hash_pairs["ahash_dist"] <= AHASH_MAX]
print(f"[원본] aHash 거리 {AHASH_MAX} 이하 유사 쌍 후보: {len(near_pairs)}쌍")
display(near_pairs["group"].value_counts().rename("쌍 수").to_frame().T)

In [ ]:
# 왼쪽: 그룹별 상자(가운데 50%)와 점(사진 쌍), 오른쪽: 그룹 안 비율로 맞춘 분포 (kwon 셀 92)
baseline = hash_pairs.loc[hash_pairs["group"] == "다른 조합", "phash_sim"].median()
bins = np.arange(0.125, 1.0 + 1 / 64, 1 / 32)  # pHash 유사도는 1/64 단위라 2칸씩 묶는다
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [1.1, 1]})
ax = axes[0]
for k, (group, color) in enumerate(zip(GROUP_ORDER, GROUP_COLORS, strict=True)):
    values = hash_pairs.loc[hash_pairs["group"] == group, "phash_sim"]
    y = len(GROUP_ORDER) - 1 - k
    ax.boxplot(
        values,
        positions=[y],
        orientation="horizontal",
        widths=0.5,
        showfliers=False,
        patch_artist=True,
        boxprops={"facecolor": color, "alpha": 0.25, "edgecolor": color},
        medianprops={"color": color, "linewidth": 2.5},
        whiskerprops={"color": color},
        capprops={"color": color},
    )
    if group != "다른 조합":  # 다른 조합은 점이 너무 많아 상자만 그린다.
        jitter = rng.uniform(-0.18, 0.18, len(values))
        ax.scatter(values, y + jitter, s=14, color=color, alpha=0.7, zorder=3)
    ax.text(
        1.02,
        y,
        f"중앙값 {values.median():.3f}\nn={len(values)}",
        va="center",
        fontsize=9,
        color=TEXT_COLOR,
        transform=ax.get_yaxis_transform(),
    )
ax.axvline(baseline, color="#898781", linestyle="--", linewidth=1)
ax.set_yticks(range(len(GROUP_ORDER)), GROUP_ORDER[::-1])
ax.set_xlim(0.1, 1.0)
ax.set_xlabel("pHash 유사도 (1 = 같은 해시, 0.5 = 무관)")
ax.set_title(f"[원본] 그룹별 분포 (점선: 다른 조합 중앙값 {baseline:.3f})", loc="left", fontsize=11)
ax = axes[1]
for group, color in zip(GROUP_ORDER, GROUP_COLORS, strict=True):
    values = hash_pairs.loc[hash_pairs["group"] == group, "phash_sim"]
    weights = np.full(len(values), 100 / len(values))
    if group == "다른 조합":
        ax.hist(values, bins=bins, weights=weights, color=color, alpha=0.35, label=group)
    else:
        ax.hist(values, bins=bins, weights=weights, histtype="step", color=color, lw=2, label=group)
ax.axvline(baseline, color="#898781", linestyle="--", linewidth=1)
ax.set_xlabel("pHash 유사도")
ax.set_ylabel("그룹 안 비율 (%)")
ax.set_title("[원본] 회색(다른 조합)에서 오른쪽으로 벗어날수록 더 닮았다", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=9)
for ax in axes:
    ax.grid(axis="x" if ax is axes[0] else "y", color="#e5e5e5", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# 70° 사진을 180° 돌려 75°·90° 사진과 다시 비교한다 (kwon 셀 94, 98).
# 같은 조합은 실험군, 다른 조합은 대조군이다.
one_70 = (photo_angle[pi] == 70) ^ (photo_angle[pj] == 70)
src = np.where(photo_angle[pi] == 70, pi, pj)[one_70]  # 돌리는 70° 사진
dst = np.where(photo_angle[pi] == 70, pj, pi)[one_70]  # 비교 대상 75°·90° 사진
rot_pairs = pd.DataFrame(
    {
        "group": np.where(photo_combo[src] == photo_combo[dst], "같은 조합", "다른 조합"),
        "file1": photo_files[src],
        "file2": photo_files[dst],
        "before": phash_sim[src, dst],
        "after": phash_rot_sim[src, dst],
    }
)
rot_pairs["group"] += " 70°-" + pd.Series(photo_angle[dst]).astype(str) + "°"
rot_pairs["change"] = rot_pairs["after"] - rot_pairs["before"]
ROT_GROUPS = ["같은 조합 70°-75°", "같은 조합 70°-90°", "다른 조합 70°-75°", "다른 조합 70°-90°"]
rot_summary = (
    rot_pairs.groupby("group")
    .agg(
        n=("change", "size"),
        before=("before", "median"),
        after=("after", "median"),
        change=("change", "median"),
        up_rate=("change", lambda c: (c > 0).mean()),
    )
    .reindex(ROT_GROUPS)
)
print("[원본] 70° 사진 180° 회전 전후 pHash 유사도 중앙값, up_rate = 회전 후 오른 쌍 비율")
display(rot_summary.round(3))

# 정답 박스로도 확인한다. 같은 조합·같은 약의 박스 중심이 POS_DIST 이내면 같은 자리다 (kwon 셀 101).
POS_DIST = 60
img_w, img_h = img_df["width"].iloc[0], img_df["height"].iloc[0]  # 모든 사진이 같은 크기다(3.1).
pos = ann_df.dropna(subset=BBOX_COLS).assign(
    combo=lambda d: d["file_name"].str.split("_").str[0],
    angle=lambda d: d["file_name"].str.split("_").str[5].astype(int),
)
rotated = pos.where(pos["angle"] != 70, rotate_bboxes_180(pos, img_w, img_h))
same_spot_rows = []
for label, boxes in [("그대로", pos), ("70° 사진 180° 회전", rotated)]:
    boxes = boxes.assign(cx=boxes["x"] + boxes["w"] / 2, cy=boxes["y"] + boxes["h"] / 2)
    matched = boxes.merge(boxes, on=["combo", "category_id"], suffixes=("1", "2"))
    matched = matched[matched["angle1"] < matched["angle2"]]
    dist = np.hypot(matched["cx1"] - matched["cx2"], matched["cy1"] - matched["cy2"])
    angle_pair = matched["angle1"].astype(str) + "°-" + matched["angle2"].astype(str) + "°"
    for pair, d in dist.groupby(angle_pair):
        same_spot_rows.append(
            {
                "각도 쌍": pair,
                "비교": label,
                "박스 쌍": len(d),
                "같은 자리 비율": (d <= POS_DIST).mean(),
                "중심 거리 중앙값(px)": d.median(),
            }
        )
same_spot_df = pd.DataFrame(same_spot_rows).set_index(["각도 쌍", "비교"]).sort_index()
print(f"[원본] 같은 조합·같은 약 박스 중심이 {POS_DIST}px 이내인 비율")
display(same_spot_df.round(3))

In [ ]:
# 왼쪽: 회전 전(연함)과 후(진함)의 중앙값, 오른쪽: 회전 후 유사도가 오른 쌍의 비율 (kwon 셀 100)
ROT_COLORS = ["#eb6834", "#1baf7a", "#c3c2b7", "#898781"]  # 같은 조합: 주황·초록, 다른 조합: 회색
ref_75_90 = hash_pairs.loc[hash_pairs["group"] == "같은 조합 75°-90°", "phash_sim"].median()
labels = [g.replace(" 70°", "\n70°") for g in ROT_GROUPS]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
x = np.arange(len(ROT_GROUPS))
ax.bar(x - 0.2, rot_summary["before"], width=0.38, color=ROT_COLORS, alpha=0.35, label="회전 전")
bars = ax.bar(x + 0.2, rot_summary["after"], width=0.38, color=ROT_COLORS, label="회전 후")
ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)
ax.axhline(ref_75_90, color="#2a78d6", linestyle="--", linewidth=1)
ax.text(3.5, ref_75_90 + 0.01, f"같은 조합 75°-90° {ref_75_90:.3f}", ha="right", color="#2a78d6")
ax.set_xticks(x, labels, fontsize=9)
ax.set_ylim(0.4, 0.95)
ax.set_ylabel("pHash 유사도 중앙값")
ax.legend(frameon=False, fontsize=9, loc="upper right")
ax.set_title("[원본] 회전 전 → 후: 같은 조합만 75°-90° 수준으로 오른다", loc="left", fontsize=11)
ax = axes[1]
bars = ax.barh(labels[::-1], rot_summary["up_rate"][::-1] * 100, color=ROT_COLORS[::-1], height=0.6)
ax.bar_label(bars, fmt="%.0f%%", padding=3)
ax.axvline(50, color="#898781", linestyle="--", linewidth=1)
ax.set_xlim(0, 105)
ax.set_xlabel("회전 후 유사도가 오른 쌍 (%)")
ax.set_title(
    "[원본] 오른 쌍의 비율: 같은 조합은 대부분, 다른 조합은 절반 이하", loc="left", fontsize=11
)
for ax in axes:
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# 같은 조합인데 돌려도 pHash가 오르지 않은 쌍 (kwon 셀 103).
# 한 줄에 70° 원본, 70° 회전본, 비교 사진을 놓는다.
# 가운데와 오른쪽에서 약 코드별 위치가 같으면 같은 배치를 180° 뒤집어 찍은 것이다.
not_up = rot_pairs[rot_pairs["group"].str.startswith("같은") & (rot_pairs["change"] <= 0)]
not_up = not_up.sort_values("change").reset_index(drop=True)
n_same_rot = int(rot_pairs["group"].str.startswith("같은").sum())
print(f"[원본] 돌려도 pHash가 오르지 않은 같은 조합 쌍: {len(not_up)} / {n_same_rot}")
if len(not_up):
    fig, axes = plt.subplots(len(not_up), 3, figsize=(10, 4.4 * len(not_up)), squeeze=False)
    for (ax1, ax2, ax3), r in zip(axes, not_up.itertuples(), strict=True):
        boxes1 = ann_df[ann_df["file_name"] == r.file1]
        show_image(ax1, TRAIN_IMG_DIR / r.file1, f"{r.file1.split('_')[0]}\n70° 원본", fontsize=9)
        draw_bboxes(ax1, boxes1, color="green")
        show_image(ax2, TRAIN_IMG_DIR / r.file1, rotate180=True)
        ax2.set_title(
            f"70° 180° 회전\npHash {r.before:.3f} → {r.after:.3f}", fontsize=9, color="red"
        )
        draw_bboxes(ax2, rotate_bboxes_180(boxes1, img_w, img_h), color="green")
        show_image(ax3, TRAIN_IMG_DIR / r.file2, f"\n{r.file2.split('_')[5]}° 원본 (비교 대상)", 9)
        draw_bboxes(ax3, ann_df[ann_df["file_name"] == r.file2], color="green")
    fig.suptitle("[원본] 돌려도 pHash가 오르지 않은 같은 조합 쌍 (초록: bbox)", x=0.01, ha="left")
    plt.tight_layout(rect=(0, 0, 1, 1 - 0.6 / fig.get_figheight()))
    plt.show()

In [ ]:
# 다른 조합인데 pHash가 매우 높은 쌍 (kwon 셀 106).
# 제목에 두 조합의 공통 약과 한쪽에만 있는 약을 적는다.
HIGH_SIM = 0.9
N_SHOW = 6
diff_pairs = hash_pairs[hash_pairs["group"] == "다른 조합"]
high_pairs = diff_pairs[diff_pairs["phash_sim"] >= HIGH_SIM].sort_values(
    "phash_sim", ascending=False
)
n_exact = (high_pairs["phash_sim"] == 1).sum()
n_095 = (high_pairs["phash_sim"] >= 0.95).sum()
print(
    f"[원본] 다른 조합 쌍 {len(diff_pairs)}개 중 pHash 1.000: {n_exact}쌍, "
    f"0.95 이상: {n_095}쌍, {HIGH_SIM} 이상: {len(high_pairs)}쌍"
)
n_same_angle = (photo_angle[high_pairs["i"]] == photo_angle[high_pairs["j"]]).sum()
print(f"[원본] 그중 같은 각도: {n_same_angle}쌍")


def drug_codes(file_name: str) -> set[str]:
    """파일 이름 앞부분(K-003351-016262-041768)에서 약 코드 집합을 뽑는다."""
    return set(file_name.split("_")[0].split("-")[1:])


fig, axes = plt.subplots(N_SHOW, 2, figsize=(8, 5.6 * N_SHOW), squeeze=False)
shown = zip(axes, high_pairs.head(N_SHOW).itertuples(), strict=False)
for k, ((ax1, ax2), r) in enumerate(shown, start=1):
    f1, f2 = photo_files[r.i], photo_files[r.j]
    d1, d2 = drug_codes(f1), drug_codes(f2)
    title1 = f"[{k}] pHash {r.phash_sim:.3f} · 공통 약: {', '.join(sorted(d1 & d2))}\n"
    title1 += f"{f1.split('_')[0]} ({f1.split('_')[5]}°)\n이 사진에만: {', '.join(sorted(d1 - d2))}"
    title2 = (
        f"\n{f2.split('_')[0]} ({f2.split('_')[5]}°)\n이 사진에만: {', '.join(sorted(d2 - d1))}"
    )
    for ax, file_name, title in [(ax1, f1, title1), (ax2, f2, title2)]:
        show_image(ax, TRAIN_IMG_DIR / file_name)
        ax.set_title(title, fontsize=9, loc="left")
        draw_bboxes(ax, ann_df[ann_df["file_name"] == file_name], color="green")
fig.suptitle(f"[원본] 다른 조합인데 pHash가 높은 쌍 상위 {N_SHOW}개", x=0.01, ha="left")
plt.tight_layout(rect=(0, 0, 1, 1 - 0.6 / fig.get_figheight()))
plt.show()

아래는 원본 상태 관측값입니다.

- pHash 유사도 중앙값은 같은 조합 75°-90° 0.781, 70°-75° 0.531, 70°-90° 0.562, 다른 조합(기준선) 0.562로 kwon 셀 90과
  같습니다. 그대로 비교하면 75°·90°만 확실히 닮았고 70°는 무관한 사진처럼 나옵니다.
- 70° 사진을 180° 돌리면 같은 조합은 70°-75°, 70°-90° 모두 0.812로 75°-90° 수준까지 오르고, 오른 쌍이 96%·84%입니다.
  대조군(다른 조합)은 0.531 → 0.562, 0.562 → 0.531로 그대로이고 오른 쌍도 48%·37%로 우연 수준입니다(kwon 셀 98).
  조명 얼룩 때문이 아니라 같은 장면이라서 오른 것입니다.
- 박스 위치로도 같은 결과입니다. 같은 조합·같은 약의 박스 중심이 60px 이내인 비율은 70° 사진을 그대로 두면 0%이고,
  180° 돌리면 70°-75° 64.5%, 70°-90° 95.7%입니다. kwon 셀 101의 64%는 70°-75° 값입니다. 75°-90°는 52.4%로 오히려
  낮습니다(중심 거리 중앙값 52px). 75° 사진의 박스가 다른 각도와 더 어긋나는 이유는 확인하지 못했습니다.
- 돌려도 pHash가 오르지 않은 같은 조합 쌍 11개는 사진으로 보면 모두 같은 배치를 180° 뒤집어 찍은 것입니다(kwon 셀 104).
  알약 3개가 삼각형으로 놓여 뒤집어도 밝기 구도가 비슷하고, 돌리기 전 값이 이미 기준선보다 높았습니다. 배치가 달라서가
  아니라 pHash의 한계입니다.
- **결론**: 같은 조합의 사진 1–3장은 같은 알약 배치를 방향과 각도만 바꿔 찍은 것이고, 70° 사진은 모두 180° 뒤집혀 있습니다.
  알약을 다시 놓고 찍은 70° 사진은 찾지 못했습니다.
- 다른 조합인데 pHash가 높은 쌍(1.000 1쌍, 0.95 이상 4쌍, 0.9 이상 62쌍. kwon 셀 106과 같음)은 중복 사진이 아니라
  같은 촬영 세팅에서 알약 일부만 바꿔 찍은 사진입니다(kwon 셀 107). 바뀐 약끼리 닮은 경우가 많아(16262 ↔ 20238 분홍 원형,
  16232 ↔ 21325 흰 원형, 18357 ↔ 32310 노란 타원) 학습 후 혼동 행렬에서 확인할 후보입니다. 규모는 8.3에서 박스 위치로 셉니다.
- aHash 거리 3 이하 후보는 38쌍으로 sohee 결과와 같습니다. 그중 35쌍이 다른 조합이고 같은 조합은 75°-90° 3쌍뿐입니다.
  aHash는 70° 뒤집힘을 보지 못하고 같은 조합 사진 쌍 158개 중 3쌍만 잡습니다. 같은 장면 후보로 쓰기에는 너무 엄격합니다.
- 해시는 흑백 변환 방식에도 민감합니다. 이식 중 aHash를 pHash처럼 컬러로 읽어 흑백으로 바꿔 보니 거리 3 이하가 38쌍에서
  25쌍으로 줄었습니다. 해시는 경향을 보는 데만 쓰고, 쌍 하나하나는 사진이나 박스 위치로 판정합니다.

### 8.3 다른 조합의 세팅 공유와 묶음

> 기준본: eda_kwon.ipynb 셀 108–119. 함수는 `data_process/data_processing.py`의 `setup_pair_distances`,
> `group_setups`입니다.

- 8.2에서 다른 조합인데 pHash가 높은 쌍은 일부 알약만 바꿔 찍은 사진이었습니다. pHash는 이 데이터에서 둔하므로
  규모는 정답 박스 위치로 셉니다.
- 다른 조합·같은 각도 사진 쌍에서 공통 약의 박스 중심이 `SAME_DIST`(30px) 이내면 같은 자리, 같은 자리인 공통 약이
  `MIN_SAME`(2개) 이상이면 세팅 공유입니다. 거의 모든 사진에 있는 3351 하나가 우연히 겹치는 경우를 거르려고 2개로 둡니다.
- 세팅을 공유하는 조합끼리 이어 묶음을 만듭니다. A-B, B-C가 공유하면 A·B·C가 한 묶음입니다.
- 박스 위치로 정하므로 라벨 오류(5장)와 라벨 누락(4.2)이 결과를 바꿀 수 있습니다. `[원본]`과 `[보정]`을 함께 셉니다.

In [ ]:
SAME_DIST = 30  # 공통 약의 박스 중심 거리가 이 px 이내면 같은 자리 (kwon 셀 109)
MIN_SAME = 2  # 같은 자리인 공통 약이 이 개수 이상이면 세팅 공유
combos = sorted(img_df["combo"].unique())


def summarize_setups(pairs: pd.DataFrame, shared: pd.Series, group: pd.Series) -> dict:
    """세팅 공유 쌍 수와 묶음 수·크기를 센다."""
    sizes = group.value_counts()
    return {
        "다른 조합·같은 각도·공통 약 있는 사진 쌍": len(pairs),
        "공통 약 1개 이상이 같은 자리": int(pairs["dists"].map(min).le(SAME_DIST).sum()),
        f"세팅 공유 쌍 (공통 약 {MIN_SAME}개 이상)": int(shared.sum()),
        "묶음 수": len(sizes),
        "혼자인 조합": int((sizes == 1).sum()),
        "다른 조합과 세팅을 공유하는 조합": int(sizes[sizes > 1].sum()),
        "가장 큰 묶음 (조합 수)": int(sizes.max()),
        "사진이 가장 많은 묶음 (사진 수)": int(img_df["combo"].map(group).value_counts().max()),
    }


setup_pairs = setup_pair_distances(ann_df)
setup_shared, setup_group = group_setups(setup_pairs, combos, SAME_DIST, MIN_SAME)
setup_states = {"[원본]": summarize_setups(setup_pairs, setup_shared, setup_group)}
if n_applied:
    corrected_pairs = setup_pair_distances(ann_corrected_df)
    corrected_shared, corrected_group = group_setups(corrected_pairs, combos, SAME_DIST, MIN_SAME)
    setup_states["[보정]"] = summarize_setups(corrected_pairs, corrected_shared, corrected_group)
print(f"[보정] 세팅 묶음: {correction_status}")
display(pd.DataFrame(setup_states))
size_counts = setup_group.value_counts().value_counts().sort_index()
print("[원본] 묶음 크기(조합 수)별 묶음 수")
display(size_counts.rename_axis("묶음 크기").rename("묶음 수").to_frame().T)


def members(group: pd.Series) -> set[frozenset]:
    """묶음마다 조합 집합을 만든다."""
    return {frozenset(s.index) for _, s in group.groupby(group)}


if n_applied:
    # 묶음 구성이 바뀐 곳과 그 원인이 된 세팅 공유 쌍을 본다.
    changed = members(corrected_group) - members(setup_group)
    for combo_set in changed:
        print(f"[보정] 바뀐 묶음 (조합 {len(combo_set)}개): {sorted(combo_set)}")
    key = ["file1", "file2"]
    new_shared = corrected_pairs[corrected_shared].merge(
        setup_pairs[setup_shared][key], on=key, how="left", indicator=True
    )
    new_shared = new_shared[new_shared["_merge"] == "left_only"].drop(columns="_merge")
    print(f"[보정] 원본에는 없고 보정본에서 새로 생긴 세팅 공유 쌍: {len(new_shared)}개")
    new_shared["dists"] = new_shared["dists"].map(lambda dists: [round(d, 1) for d in dists])
    display(new_shared[["angle", "combo1", "combo2", "shared", "dists"]])

# 8.2의 해시 후보가 세팅 묶음과 맞는지 본다. 다른 조합 쌍 중 같은 묶음에 든 쌍의 수
photo_group = img_df["combo"].map(setup_group).to_numpy()
check_rows = []
for label, pairs in [
    (f"aHash 거리 {AHASH_MAX} 이하", near_pairs),
    (f"pHash {HIGH_SIM} 이상", high_pairs),
]:
    diff = pairs[pairs["group"] == "다른 조합"]
    same_group = photo_group[diff["i"]] == photo_group[diff["j"]]
    check_rows.append(
        {"후보": label, "다른 조합 쌍": len(diff), "같은 묶음": int(same_group.sum())}
    )
print("[원본] 해시 유사 후보(다른 조합)와 세팅 묶음")
display(pd.DataFrame(check_rows).set_index("후보"))

In [ ]:
# ① 단계별로 좁혀지는 사진 쌍, ② 같은 자리인 공통 약 수, ③ 묶음 크기별 조합 수 (kwon 셀 110)
fig, axes = plt.subplots(1, 3, figsize=(17, 5.2), gridspec_kw={"width_ratios": [1.2, 1, 1.2]})
n_same = setup_pairs["dists"].map(lambda dists: sum(d <= SAME_DIST for d in dists))
ax = axes[0]
steps = [
    "다른 조합·같은 각도\n공통 약 있음",
    "공통 약 1개 이상\n같은 자리",
    f"공통 약 {MIN_SAME}개 이상\n같은 자리 (세팅 공유)",
]
counts = [len(setup_pairs), int((n_same >= 1).sum()), int(setup_shared.sum())]
bars = ax.barh(steps[::-1], counts[::-1], color=["#eb6834", "#86b6ef", "#c3c2b7"], height=0.6)
ax.bar_label(bars, labels=[f"{c:,}쌍 ({c / counts[0]:.1%})" for c in counts[::-1]], padding=4)
ax.set_xlim(0, counts[0] * 1.4)
ax.set_title("① 사진 쌍이 좁혀지는 과정", loc="left", fontsize=11)
ax = axes[1]
n_same_counts = n_same.value_counts().sort_index()
colors = ["#eb6834" if n >= MIN_SAME else "#c3c2b7" for n in n_same_counts.index]
bars = ax.bar(n_same_counts.index.astype(str), n_same_counts.values, color=colors, width=0.65)
ax.bar_label(bars, fmt="%d", padding=3)
ax.set_xlabel("같은 자리에 있는 공통 약 수")
ax.set_ylabel("사진 쌍 수")
ax.set_title(
    f"② 같은 자리인 공통 약 수 (주황: {MIN_SAME}개 이상 = 세팅 공유)", loc="left", fontsize=11
)
ax = axes[2]
combos_in = size_counts.index * size_counts.values  # 묶음 크기 → 그 크기 묶음에 속한 조합 수
colors = ["#c3c2b7" if size == 1 else "#2a78d6" for size in size_counts.index]
bars = ax.bar([f"{s}개" for s in size_counts.index], combos_in, color=colors, width=0.65)
ax.bar_label(
    bars,
    labels=[
        f"조합 {c}개\n(묶음 {n}개)" for c, n in zip(combos_in, size_counts.values, strict=True)
    ],
    padding=3,
    fontsize=9,
)
ax.set_ylim(0, max(combos_in) * 1.3)
ax.set_xlabel("묶음 크기 (묶음 안 조합 수)")
ax.set_ylabel("조합 수")
ax.set_title(
    f"③ 조합 {len(combos)}개 → 묶음 {setup_group.nunique()}개 (회색: 혼자인 조합)",
    loc="left",
    fontsize=11,
)
for ax in axes:
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
fig.suptitle("[원본] 세팅 공유와 묶음", x=0.01, ha="left")
plt.tight_layout()
plt.show()

**기준을 바꿔도 묶음이 유지되는가** (kwon 셀 112–116)

`SAME_DIST`와 `MIN_SAME`을 바꿔 묶음 수와 가장 큰 묶음을 다시 셉니다. kwon 표(15–80px)에 5px, 10px를 더했습니다.
kwon 셀 116의 제안대로, 세팅 공유 판정을 가르는 거리(쌍마다 두 번째로 가까운 공통 약의 중심 거리) 분포도 봅니다.
"진짜 같은 세팅"(거의 0px)과 "우연히 비슷한 자리"(넓게 퍼짐) 사이에 경계가 있으면 그 거리가 기준값 후보입니다.

In [ ]:
SAME_DISTS = [5, 10, 15, 20, 30, 50, 80]
MIN_SAMES = [1, 2, 3]
sens_rows = []
for same_dist in SAME_DISTS:
    for min_same in MIN_SAMES:
        shared, group = group_setups(setup_pairs, combos, same_dist, min_same)
        sizes = group.value_counts()
        sens_rows.append(
            {
                "SAME_DIST": same_dist,
                "MIN_SAME": min_same,
                "세팅 공유 쌍": int(shared.sum()),
                "묶음 수": len(sizes),
                "혼자인 조합": int((sizes == 1).sum()),
                "가장 큰 묶음": int(sizes.max()),
            }
        )
sens_df = pd.DataFrame(sens_rows)
print(f"[원본] 묶음 수 / 가장 큰 묶음 (기준: SAME_DIST={SAME_DIST}, MIN_SAME={MIN_SAME})")
display(sens_df.pivot(index="SAME_DIST", columns="MIN_SAME", values=["묶음 수", "가장 큰 묶음"]))

# 공통 약이 2개 이상인 쌍에서 두 번째로 가까운 공통 약의 거리.
# 이 값이 SAME_DIST 이하이면 세팅 공유다.
second_dist = setup_pairs["dists"].map(lambda dists: sorted(dists)[1] if len(dists) > 1 else np.nan)
second_dist = second_dist.dropna()
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), gridspec_kw={"width_ratios": [1, 1.4]})
ax = axes[0]
for min_same, color in zip(MIN_SAMES, ["#c3c2b7", "#2a78d6", "#eb6834"], strict=True):
    sub = sens_df[sens_df["MIN_SAME"] == min_same]
    ax.plot(
        sub["SAME_DIST"],
        sub["묶음 수"],
        marker="o",
        color=color,
        lw=2,
        label=f"MIN_SAME {min_same}개",
    )
ax.scatter([SAME_DIST], [setup_group.nunique()], s=200, facecolors="none", edgecolors="red", lw=2.5)
ax.set_xticks(SAME_DISTS)
ax.set_xlabel("SAME_DIST (px)")
ax.set_ylabel(f"묶음 수 (조합 {len(combos)}개 중)")
ax.set_ylim(0, len(combos) + 10)
ax.legend(frameon=False, fontsize=9)
ax.set_title(
    "[원본] 거리 기준을 넓히면 묶음이 합쳐진다 (빨간 원: 현재 기준)", loc="left", fontsize=11
)
ax = axes[1]
ax.hist(second_dist[second_dist <= 80], bins=np.arange(0, 82, 2), color="#2a78d6")
for d in [5, 20, SAME_DIST]:
    ax.axvline(d, color="red" if d == SAME_DIST else "#898781", linestyle="--", linewidth=1)
    ax.text(d + 0.5, ax.get_ylim()[1] * 0.92, f"{d}px", fontsize=9, color=TEXT_COLOR)
ax.set_xlabel(
    f"두 번째로 가까운 공통 약의 중심 거리 (px, 80 초과 {(second_dist > 80).sum()}쌍은 뺌)"
)
ax.set_ylabel("사진 쌍 수")
ax.set_title(
    f"[원본] 공통 약 2개 이상인 다른 조합 쌍 {len(second_dist)}개의 거리 분포",
    loc="left",
    fontsize=11,
)
for ax in axes:
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()
print("[원본] 두 번째 공통 약 거리 구간별 쌍 수")
display(
    pd.cut(second_dist, [0, 5, 10, 15, 20, 30, 50, 80, np.inf], right=True, include_lowest=True)
    .value_counts(sort=False)
    .rename("쌍 수")
    .to_frame()
    .T
)

In [ ]:
# 두 번째 공통 약 거리 구간마다 세팅 공유 후보 쌍을 SEED로 2개씩 골라 본다.
# 초록: 공통 약, 주황: 한쪽에만 있는 약. 알약의 자리뿐 아니라 놓인 방향까지 같은지 본다.
DIST_BANDS = [(0, 5), (20, 30), (30, 50)]
band_pairs = setup_pairs.assign(second=second_dist).dropna(subset=["second"])
band_pairs = pd.concat(
    [
        band_pairs[(band_pairs["second"] > lo) & (band_pairs["second"] <= hi)].sample(
            2, random_state=SEED
        )
        for lo, hi in DIST_BANDS
    ]
)
fig, axes = plt.subplots(len(band_pairs), 2, figsize=(8, 5.2 * len(band_pairs)), squeeze=False)
for (ax1, ax2), r in zip(axes, band_pairs.itertuples(), strict=True):
    for ax, file_name in [(ax1, r.file1), (ax2, r.file2)]:
        title = f"{file_name.split('_')[0]} ({r.angle}°)\n두 번째 공통 약 거리 {r.second:.0f}px"
        show_image(ax, TRAIN_IMG_DIR / file_name, title, fontsize=9)
        boxes = ann_df[ann_df["file_name"] == file_name]
        draw_bboxes(ax, boxes[boxes["category_id"].isin(r.shared)], color="green")
        draw_bboxes(ax, boxes[~boxes["category_id"].isin(r.shared)], color="#eb6834")
fig.suptitle("[원본] 거리 구간별 세팅 공유 후보 (구간마다 2쌍)", x=0.01, ha="left")
plt.tight_layout(rect=(0, 0, 1, 1 - 0.6 / fig.get_figheight()))
plt.show()

In [ ]:
# 세팅 공유 예시(같은 자리 공통 약이 가장 많은 쌍)와 가장 큰 묶음 (kwon 셀 118 ②, ⑥)
# 초록: 같은 자리 그대로인 공통 약, 주황: 바뀐 약
example = setup_pairs[setup_shared].assign(n_same=n_same).sort_values("n_same").iloc[-1]
same_spot = {c for c, d in zip(example["shared"], example["dists"], strict=True) if d <= SAME_DIST}
fig, axes = plt.subplots(1, 2, figsize=(9, 6))
for ax, file_name, combo in [
    (axes[0], example["file1"], example["combo1"]),
    (axes[1], example["file2"], example["combo2"]),
]:
    show_image(ax, TRAIN_IMG_DIR / file_name, f"조합 {combo} ({example['angle']}°)", fontsize=9)
    boxes = ann_df[ann_df["file_name"] == file_name]
    draw_bboxes(ax, boxes[boxes["category_id"].isin(same_spot)], color="green")
    draw_bboxes(ax, boxes[~boxes["category_id"].isin(same_spot)], color="#eb6834")
fig.suptitle(
    "[원본] 조합은 다르지만 세팅은 같다 (초록: 같은 자리 / 주황: 바뀐 약)", x=0.01, ha="left"
)
plt.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

largest = setup_group.value_counts().index[0]
largest_combos = sorted(setup_group[setup_group == largest].index)
largest_photos = img_df[img_df["combo"].isin(largest_combos)].assign(
    angle=lambda d: d["file_name"].str.split("_").str[5].astype(int)
)
common = int(largest_photos["angle"].value_counts().idxmax())  # 같은 각도끼리 놓아야 비교하기 쉽다.
fig, axes = image_grid(len(largest_combos), cols=5, cell=(3.2, 4.4))
for ax, combo in zip(axes, largest_combos, strict=True):
    photos = largest_photos[largest_photos["combo"] == combo]
    photo = photos[photos["angle"] == common] if (photos["angle"] == common).any() else photos
    photo = photo.iloc[0]
    show_image(ax, TRAIN_IMG_DIR / photo["file_name"], f"{combo}\n({photo['angle']}°)")
    draw_bboxes(ax, ann_df[ann_df["file_name"] == photo["file_name"]], color="green")
fig.suptitle(
    f"[원본] 가장 큰 묶음: 조합 {len(largest_combos)}개 · 사진 {len(largest_photos)}장 "
    f"(train {len(img_df)}장의 {len(largest_photos) / len(img_df):.0%})",
    x=0.01,
    ha="left",
)
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

아래는 원본 상태 관측값입니다. `[보정]`은 5.8의 보정 기록을 적용한 결과입니다.

- 다른 조합·같은 각도·공통 약이 있는 사진 쌍 4,560개 중 세팅 공유는 103쌍입니다. 조합 114개가 묶음 67개가 되고,
  다른 조합과 세팅을 공유하는 조합은 65개(57%), 가장 큰 묶음은 조합 13개·사진 26장입니다(kwon 셀 109와 같음).
  사진이 가장 많은 묶음은 조합 12개·사진 36장(train의 16%)입니다.
- `[보정]`에서는 세팅 공유 쌍이 6개 늘어 109쌍이 되고, 조합 13개 묶음과 3개 묶음이 합쳐져 묶음 66개, 가장 큰 묶음은
  조합 16개가 됩니다. 라벨 누락 추가와 좌표 수정으로 공통 약이 생기거나 제자리를 찾은 쌍이 새로 이어졌기 때문입니다.
  세팅 묶음은 어느 라벨로 만들었는지에 따라 달라지므로 분할에 쓸 때 라벨 상태를 함께 기록해야 합니다.
- 해시 후보와 맞춰 보면 aHash 후보(다른 조합 35쌍) 중 16쌍, pHash 0.9 이상(다른 조합 62쌍) 중 25쌍만 같은 묶음입니다.
  pHash 상위 6쌍 중 3쌍도 묶이지 않습니다. 묶이지 않은 고유사 쌍 37개 중 23개는 공통 약이 1개뿐이라 `MIN_SAME=2`로는
  세팅 공유를 판정할 수 없습니다. 알약 3개 중 2개를 바꿔 찍은 세팅은 이 방법이 놓칩니다.
- 기준 민감도는 kwon 셀 113과 같습니다. 공통 약 1개 기준은 3351 때문에 거의 다 이어지고(5px에서도 묶음 63개, 15px 이상 14개 이하),
  3개 기준은 거의 묶이지 않습니다(108–112개). 공통 약 2개 기준에서 거리만 바꾸면 5px 103개, 20px 92개, 30px 67개, 50px 17개입니다.
  가장 엄격한 5px에서도 조합 17개가 세팅을 공유하므로, 이름이 달라도 같은 세팅인 사진이 있다는 것은 기준과 관계없이 확실합니다.
- 두 번째 공통 약 거리(공통 약 2개 이상인 쌍 848개)는 5px 이하에 42쌍이 뚜렷이 몰려 있고, 5–20px는 21쌍으로 드물며,
  20px부터 다시 늘어납니다. 구간별 예시(구간마다 2쌍)에서 5px 이하 쌍은 알약의 자리와 놓인 방향까지 같습니다. 20–30px 쌍은
  같은 약이 비슷한 자리에 있지만 방향이 다른 알약이 있고, 30px를 넘으면 "위 1개, 아래 2개" 같은 비슷한 배치일 뿐입니다.
- 그래서 현재 기준(30px) 묶음 67개에는 세팅을 그대로 두고 알약만 바꾼 쌍(5px 이하)과, 알약을 비슷한 자리에 다시 놓은 쌍
  (20–30px 40쌍)이 섞여 있습니다. 뒤의 경우도 같은 약이 같은 자리 근처에 있어 누수가 될 수 있습니다. 어디까지 묶을지는
  9장 "결정할 것"으로 넘깁니다(kwon 셀 120의 "20–30px가 적절해 보임"과 비교).

### 8.4 분할 후보 비교

> 기준본: eda_kwon.ipynb 셀 39, 120. sohee_eda.ipynb 셀 16–17의 `suggest_split`(조합의 20%를 seed 42로 검증 후보로
> 뽑기)을
> 그대로 재현해 비교안으로 두고, EDA_jang.ipynb 셀 20의 "여러 번 나눠 비교하자"는 제안에 따라 seed를 바꿔 반복했습니다.

학습 없이 나누기만 해서 비교합니다. 후보는 나누는 단위만 다릅니다.

| 후보 | 나누는 단위 | 출처 |
|---|---|---|
| A. 사진 | 사진 232장 | 비교 기준 |
| B. 조합 | 조합 114개 | kwon 셀 39 A-1, sohee 셀 17, jang 셀 20 |
| C. 세팅 묶음 | 묶음(8.3, `SAME_DIST`·`MIN_SAME` 기준) | kwon 셀 120 |
| C+. 세팅 묶음, 희귀 약 묶음은 학습에 고정 | C와 같고, 희귀 약이 든 묶음은 항상 학습 | kwon 셀 120 추천안 |

- A–C+는 bbox를 표본, `category_id`를 층, 나누는 단위를 그룹으로 `StratifiedGroupKFold(5)`에 넣습니다. 같은 그룹은
  같은 fold에 들어가고 클래스 비율은 fold마다 비슷해지도록 맞춥니다. seed를 `SEED`부터 5개 바꿔 fold 25개를 봅니다.
- 지표는 fold마다 검증 쪽 기준입니다.
  - **검증에만 있는 클래스**: 학습에 0개라 배울 수 없는 약
  - **검증에 없는 클래스**: 그 fold에서 평가하지 못하는 약
  - **같은 조합 누수**: 같은 조합의 다른 사진이 학습에 있는 검증 사진의 비율
  - **같은 묶음 누수**: 같은 세팅 묶음의 사진이 학습에 있는 검증 사진의 비율

In [ ]:
N_FOLDS = 5
N_REPEATS = 5  # seed를 SEED, SEED+1, ...로 바꿔 반복한다.
split_units = img_df[["file_name", "combo"]].assign(setup=img_df["combo"].map(setup_group))
split_boxes = ann_df[["file_name", "category_id"]].merge(split_units, on="file_name")
rare_setups = set(split_boxes.loc[split_boxes["category_id"].isin(rare_df.index), "setup"])


def split_metrics(val_files: set[str]) -> dict:
    """검증 사진 집합에 대해 사진·bbox 수, 클래스 누락, 같은 조합·묶음 누수를 잰다."""
    is_val = split_boxes["file_name"].isin(val_files)
    train_cls = set(split_boxes.loc[~is_val, "category_id"])
    val_cls = set(split_boxes.loc[is_val, "category_id"])
    val_units = split_units[split_units["file_name"].isin(val_files)]
    train_units = split_units[~split_units["file_name"].isin(val_files)]
    return {
        "검증 사진": len(val_units),
        "검증 bbox": int(is_val.sum()),
        "검증에만 있는 클래스": len(val_cls - train_cls),
        "검증에 없는 클래스": len(train_cls - val_cls),
        "같은 조합 누수": val_units["combo"].isin(set(train_units["combo"])).mean(),
        "같은 묶음 누수": val_units["setup"].isin(set(train_units["setup"])).mean(),
    }


# sohee 셀 17 재현: 정렬한 조합 목록에서 20%를 default_rng(42)로 뽑아 검증 후보로 둔다.
sohee_groups = sorted(split_units["combo"].unique())
sohee_val = set(
    np.random.default_rng(42).choice(sohee_groups, round(len(sohee_groups) * 0.2), replace=False)
)
sohee_files = set(split_units.loc[split_units["combo"].isin(sohee_val), "file_name"])
sohee_split = split_metrics(sohee_files)
sohee_is_val = split_boxes["file_name"].isin(sohee_files)
sohee_val_only = sorted(
    set(split_boxes.loc[sohee_is_val, "category_id"])
    - set(split_boxes.loc[~sohee_is_val, "category_id"])
)
print(f"[원본] sohee 분할 재현: 검증 조합 {len(sohee_val)}개, {sohee_split}")
print(f"[원본] sohee 분할의 검증에만 있는 클래스: {sohee_val_only}")

CANDIDATES = {
    "A. 사진": ("file_name", False),
    "B. 조합": ("combo", False),
    "C. 세팅 묶음": ("setup", False),
    "C+. 세팅 묶음, 희귀 약 학습 고정": ("setup", True),
}
fold_rows = []
for name, (group_col, fix_rare) in CANDIDATES.items():
    boxes = split_boxes[~split_boxes["setup"].isin(rare_setups)] if fix_rare else split_boxes
    for seed in range(SEED, SEED + N_REPEATS):
        splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        with warnings.catch_warnings():
            # bbox가 N_FOLDS개보다 적은 클래스가 있다는 경고. 희귀 약이 있어 당연히 나온다.
            warnings.simplefilter("ignore", UserWarning)
            folds = list(splitter.split(boxes, boxes["category_id"], boxes[group_col]))
        for fold, (_, val_idx) in enumerate(folds):
            val_files = set(boxes["file_name"].iloc[val_idx])
            fold_rows.append({"후보": name, "seed": seed, "fold": fold, **split_metrics(val_files)})
fold_df = pd.DataFrame(fold_rows)
fixed = split_units["setup"].isin(rare_setups)
n_fixed_combos = split_units.loc[fixed, "combo"].nunique()
print(
    f"[원본] 희귀 약이 든 묶음 {len(rare_setups)}개 = 조합 {n_fixed_combos}개, "
    f"사진 {fixed.sum()}장 (C+에서 항상 학습)"
)
METRICS = ["검증 사진", "검증 bbox", "검증에만 있는 클래스", "검증에 없는 클래스"]
METRICS += ["같은 조합 누수", "같은 묶음 누수"]
split_summary = fold_df.groupby("후보", sort=False)[METRICS].agg(["mean", "min", "max"])
print(f"[원본] 후보별 fold {N_FOLDS * N_REPEATS}개의 평균·최소·최대 (누수는 비율)")
display(split_summary.T.round(2))

In [ ]:
# 후보별로 fold 25개의 지표를 점으로 찍는다. 가로선은 평균이다.
SPLIT_COLORS = ["#898781", "#eb6834", "#2a78d6", "#1baf7a"]
panels = [
    ("같은 묶음 누수", "같은 묶음 사진이 학습에 있는 검증 사진 (%)", 100),
    ("검증에만 있는 클래스", "검증에만 있는 클래스 수 (배울 수 없음)", 1),
    ("검증에 없는 클래스", "검증에 없는 클래스 수 (평가 못 함)", 1),
]
rng = np.random.default_rng(SEED)
names = list(CANDIDATES)
fig, axes = plt.subplots(1, len(panels), figsize=(17, 5))
for ax, (metric, title, scale) in zip(axes, panels, strict=True):
    for k, (name, color) in enumerate(zip(names, SPLIT_COLORS, strict=True)):
        values = fold_df.loc[fold_df["후보"] == name, metric] * scale
        ax.scatter(k + rng.uniform(-0.15, 0.15, len(values)), values, s=18, color=color, alpha=0.7)
        ax.hlines(values.mean(), k - 0.3, k + 0.3, color=color, linewidth=2.5)
        ax.text(k + 0.32, values.mean(), f"{values.mean():.1f}", va="center", fontsize=9)
    ax.set_xticks(range(len(names)), [n.replace(", ", ",\n") for n in names], fontsize=9)
    ax.set_title(title, loc="left", fontsize=11)
    ax.grid(axis="y", color="#e5e5e5", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
fig.suptitle(
    f"[원본] 분할 후보별 fold {N_FOLDS * N_REPEATS}개 (점: fold, 가로선: 평균)", x=0.01, ha="left"
)
plt.tight_layout()
plt.show()

아래는 원본 상태 관측값입니다. 학습하지 않고 나누기만 한 결과이며 fold 25개의 평균(괄호는 최소–최대)입니다.

| 후보 | 같은 조합 누수 | 같은 묶음 누수 | 검증에만 있는 클래스 | 검증에 없는 클래스 | 검증 사진 |
|---|---:|---:|---:|---:|---:|
| A. 사진 | 82% (76–94%) | 91% | 0 | 6.8 | 46 |
| B. 조합 | 0% | 59% (40–78%) | 3.4 (1–6) | 20.2 | 46 (41–52) |
| C. 세팅 묶음 | 0% | 0% | 4.4 (2–6) | 25.6 | 46 (41–57) |
| C+. 세팅 묶음, 희귀 약 학습 고정 | 0% | 0% | 0.2 (0–1) | 35.5 | 33 (30–40) |
| sohee 분할 재현 (B, 1회) | 0% | 44% | 5 | 25 | 50 |

- **A. 사진**: 검증 사진의 82%가 같은 조합의 다른 사진을 학습에 둡니다(kwon 셀 120의 76%는 한 번 나눈 값으로 이 범위 안).
  검증 점수가 장면 암기로 부풀려지므로 비교 기준으로만 둡니다.
- **B. 조합**: 같은 조합 누수는 없지만 검증 사진의 59%가 세팅을 공유하는 다른 조합의 사진을 학습에 둡니다. sohee 분할을 그대로
  재현하면 검증 조합 23개·사진 50장, 검증에만 있는 클래스 5개(`3544, 4543, 12247, 27993, 31885`)로 `sohee_eda_results.md`와
  같고, 같은 묶음 누수는 44%입니다.
- **C. 세팅 묶음**: 묶음 누수까지 없앱니다. 대신 희귀 약의 유일한 장면이 검증으로 가면서 fold마다 평균 4.4개 클래스를 배울 수
  없고, 묶음 크기가 고르지 않아 검증 크기가 41–57장, bbox 127–206개로 흔들립니다.
- **C+**: 희귀 약이 든 묶음 10개(조합 23개, 사진 69장, train의 30%)를 항상 학습에 두면 배울 수 없는 클래스는 거의 사라집니다.
  사진이 가장 많은 묶음(36장)도 여기에 들어갑니다. 그 대가로 검증은 33장 정도로 줄고 fold마다 56종 중 35종 정도를 평가하지 못합니다.
  희귀 약의 성능은 검증으로 잴 수 없다는 점을 기록해야 합니다(kwon 셀 39 A-3).
- fold마다 숫자가 크게 흔들리므로 한 번 나눈 점수로 모델을 고르지 않습니다. 같은 fold들에서 매번 처음부터 학습해 평균과
  표준편차로 비교합니다(jang 셀 20, kwon 셀 39 A-4).
- 어느 후보를 쓸지와 세팅 묶음 기준(8.3)은 9장 "결정할 것"과 이슈 #26에서 정합니다.

## 9. 결론과 다음 작업

> 출처: 3–8장의 해석 셀. 노트북마다 결론이 다른 내용은 지우지 않고 9.2 "결정할 것"으로 옮겼습니다.

수치는 모두 `[원본]` 상태 관측값입니다. `RUN_HEAVY`가 필요한 항목은 미실행이면 이 노트북 실행만으로는 다시 확인되지 않습니다.

### 9.1 확정된 사실

| 주제 | 사실 | 절 |
|---|---|---|
| 규모 | train 232장, test 842장. annotation JSON 763개 = bbox 763개, 클래스 56종 | 2, 3.1 |
| 구조 | JSON 하나 = 사진 1장 × 알약 1개. 같은 사진의 라벨이 약 코드별 폴더에 흩어져 있어 사진 단위로 모아야 함 | 3.2 |
| 중복 | JSON 내용·라벨 중복 없음, train/test 같은 파일명 없음. 파일·픽셀 중복은 `RUN_HEAVY` | 3.3 |
| 라벨 누락 | 사진 속 알약 하나의 bbox가 빠진 경우 8개 | 4.2 |
| 메타데이터 | `null` 18건은 모두 `16548` 약 정보. bbox 학습에는 영향 없음 | 4.3 |
| 라벨 오류 | 이미지 밖 좌표 1개(18357), 다른 약과 좌표 중복 3쌍, 빈 배경 박스 1개, area ≠ w×h 6개(bbox가 틀린 것은 19232 1개) | 5.1–5.4 |
| 이상치 | 면적·비율 IQR 후보와 모양 의심 박스는 라벨 오류가 아님(위 오류 제외) | 5.5, 5.6 |
| 여백 | 정답 박스는 알약보다 사방 약 8px 넉넉하며, 대부분 10% 이상 여백 (`RUN_HEAVY`) | 5.7 |
| 불균형 | `3351` 한 종이 bbox의 20%. bbox 5개 이하 18종, 최대/최소 51배 | 6.1 |
| 크기 | 사진 대비 bbox 넓이 1.9–21.8%. Small 구간의 69%가 `3351`. 작은 객체보다 불균형이 더 큰 문제 | 6.2 |
| 촬영 조건 | 배경·조명은 한 가지, 사진마다 다른 것은 각도(70°·75°·90°)뿐. 각도별 bbox 크기 차이 1–2% | 7.2 |
| 실제 크기 | 1mm ≈ 23.6px. 장축·단축이 ±0.5mm 이내인 클래스 쌍이 46쌍이라 크기만으로 구별 못 하는 약이 많음 (`RUN_HEAVY`) | 7.3 |
| 같은 조합 | 조합 114개(3장 40, 2장 38, 1장 36). 같은 조합 사진은 같은 배치를 각도만 바꿔 찍었고, 70° 사진은 180° 뒤집혀 있음 | 7.1, 8.2 |
| 희귀 약 | bbox 5개 이하 18종 중 17종은 조합 하나(사진 3장)에서만 나옴 | 8.1 |
| 세팅 공유 | 조합이 달라도 알약 일부만 바꿔 찍은 사진이 있음. 가장 엄격한 5px 기준에서도 조합 17개 | 8.3 |
| 누수 | 사진 무작위 분할은 검증 사진의 82%, 조합 분할도 59%가 같은 장면·세팅의 사진을 학습에 둠 | 8.4 |

### 9.2 결정할 것

| # | 항목 | 선택지 | 근거 | 절 |
|---:|---|---|---|---|
| 1 | 18357 bbox 값 | kwon `[574, 626, 300, 309]` / JJAM 색상 윤곽 `[572, 625, 303, 312]` | 두 값의 IoU 약 0.98. 하나를 고르면 이미지 밖 좌표가 해소됨 | 5.1, 5.8 |
| 2 | 보정 기록 검수 기준 | kwon 기록을 검수됨으로 볼지, JJAM 19232 제안을 적용할지 | 미검수 기록은 적용하지 않음 | 5.4, 5.8 |
| 3 | 라벨 누락 8개 처리 | 추가 / 사진 제외 / 검증에서만 제외 / 그대로 | kwon 셀 12. 추가하면 희귀 약 수와 세팅 묶음도 바뀜 | 4.2, 8.1, 8.3 |
| 4 | 정답 박스 여백 | 그대로 / 20% 이상만 줄이기 / 전체 기준을 새로 정하기 | 20% 이상만 줄이면 다른 박스와 여백 기준이 달라짐 | 5.7 |
| 5 | 보정 기록 파일 | `correction_df` 형식으로 `data/processed/`에 저장하는 작업을 `data_process/`에 만들지 | 저장하면 새 라벨 ID가 기록에 고정됨 | 5.8 |
| 6 | 분할 단위 | B 조합 / C 세팅 묶음 / C+ 희귀 약 묶음 학습 고정 | 누수와 평가 범위가 서로 맞바뀜. 이슈 #26 | 8.4 |
| 7 | 세팅 묶음 기준 | `SAME_DIST` 5px(묶음 103개) / 20px(92개) / 30px(67개), 원본·보정 라벨 중 어느 것으로 묶을지 | 거리 분포가 5px 이하에 몰리고 20px부터 다시 늘어남. kwon은 20–30px 제안 | 8.3 |
| 8 | 공통 약 1개 세팅 공유 | 사진으로 확인해 묶음에 더할지 | `MIN_SAME=2`는 알약 2개를 바꿔 찍은 세팅을 놓침 | 8.3 |
| 9 | 70° 사진 방향 | 학습 전에 돌려 맞추기 / 회전 증강으로 처리 | 70° 사진은 모두 180° 뒤집혀 있음. test 사진 방향은 모름 | 8.2 |
| 10 | 실제 크기 정보 | 분류 보조로 쓸지 | 이미지만 쓴 모델과 비교해 정함. test 배율은 확인 필요 | 7.3 |

### 9.3 학습 파이프라인으로 넘길 것

**분할과 평가**

- 같은 조합의 사진은 반드시 한쪽에 둡니다. 세팅 묶음까지 묶을지는 9.2의 6–7번 결정을 따릅니다.
- 한 번 나눈 점수로 모델을 고르지 않습니다. 같은 fold들에서 매번 처음부터 학습해 평균과 표준편차를 기록합니다(jang 셀 20).
  B안과 C안의 검증 점수 차이가 크면 세팅 공유 누수가 실제로 있다는 근거입니다(kwon 셀 120).
- 전체 mAP와 함께 클래스별 AP, 클래스 그룹별(bbox 21개 이상 / 6–20개 / 5개 이하) 평균 AP, 혼동 행렬을 봅니다(kwon 셀 39 F).
  검증에 없는 클래스가 있으면 그 수를 함께 적습니다.
- 혼동 후보: 일부만 바꿔 찍은 닮은 약(16262 ↔ 20238, 16232 ↔ 21325, 18357 ↔ 32310, 8.2), 크기가 같은 지름 7–8.5mm 원형
  정제(7.3).

**불균형과 증강** (kwon 셀 39의 추천 순서)

1. 사전학습 가중치와 기본 증강으로 기준 모델을 학습합니다.
2. 클래스별·그룹별 AP로 약한 곳을 확인합니다.
3. 희귀 약이 약하면 Copy-Paste 증강이나 Repeat Factor Sampling을 더합니다. 배경이 균일한 연회색이라 Copy-Paste가 자연스럽습니다.
4. 그래도 부족하면 클래스 가중치(제곱근 역수처럼 완만하게), Focal Loss, 2단계(찾기 → 분류) 방식을 검토합니다.

- 회전 증강을 우선합니다. 알약은 방향과 관계없이 같은 약이고 70° 사진은 뒤집혀 있습니다. 좌우·상하 뒤집기는 각인이 거울상이 되므로 후순위입니다.
- 색은 약을 구별하는 단서라 색조(hue) 변화는 약하게 줍니다. 배경·조명이 한 가지뿐이므로 test 사진의 배경·조명이 다르면 밝기·대비 증강이 필요합니다(7.2).
- 각도별 bbox 크기 차이가 1–2%라 원근 변환 증강은 우선순위가 낮습니다(7.2). 작은 객체 대응보다 불균형 대응이 먼저입니다(6.2).

**라벨과 추론**

- 원본 라벨 오류를 그대로 학습하면 18357 캡슐 같은 알약이 배경으로 학습됩니다. 학습에는 보정 기록을 적용한 라벨을 쓰고,
  어떤 기록을 적용했는지 실험마다 남깁니다(5.8).
- 맞닿은 알약이 있으므로 추론 NMS가 클래스별로 적용되는지, IoU 기준이 너무 낮지 않은지 확인합니다(5.6).
- test 사진에는 JSON이 없고 파일 이름도 번호뿐입니다. 촬영 조건과 파일 이름의 조합 정보는 추론에 쓸 수 없습니다(7장, kwon 셀 39 E).
- 외부 데이터(`img_key`의 AI Hub 계열 낱알 이미지)는 과제 규칙에서 허용되는지 먼저 확인합니다(kwon 셀 39 G).